# creacion carpetas

In [2]:
from pathlib import Path
import shutil

# ============================================================
# CONFIGURACIÓN
# ============================================================
DRY_RUN = False  # primero True; luego cámbialo a False para ejecutar de verdad

# Detectar raíz del proyecto
current_dir = Path.cwd().resolve()

if current_dir.name.upper() == "CODIGO":
    project_root = current_dir.parent
else:
    project_root = current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"

if not salidas_sens.exists():
    raise FileNotFoundError(f"No existe la carpeta: {salidas_sens}")

print(f"Raíz del proyecto: {project_root}")
print(f"Carpeta objetivo : {salidas_sens}")
print(f"Modo DRY_RUN     : {DRY_RUN}")

# ============================================================
# CARPETAS VIEJAS A BORRAR DENTRO DE SALIDAS_SENSIBILIDAD
# ============================================================
folders_to_delete = [
    salidas_sens / "00_logs_validacion",
    salidas_sens / "01_corrida_base",
    salidas_sens / "01_insumos_consolidados",
    salidas_sens / "01_insumos_finales_hbm",
    salidas_sens / "02_corrida_base",
    salidas_sens / "02_escenarios",
    salidas_sens / "03_sensibilidad",
    salidas_sens / "04_figuras",
    salidas_sens / "04_resultados_finales",
    salidas_sens / "05_tablas",
    salidas_sens / "06_logs",
]

# ============================================================
# NUEVA ESTRUCTURA
# ============================================================
folders_to_create = [
    salidas_sens / "00_logs_validacion",
    salidas_sens / "01_insumos",
    salidas_sens / "01_insumos" / "hbm",
    salidas_sens / "01_insumos" / "benmap",
    salidas_sens / "01_insumos" / "poblacion",
    salidas_sens / "01_insumos" / "incidencia",
    salidas_sens / "01_insumos" / "grilla",
    salidas_sens / "02_base_analitica",
    salidas_sens / "03_corrida_principal",
    salidas_sens / "04_sensibilidad",
    salidas_sens / "04_sensibilidad" / "beta",
    salidas_sens / "04_sensibilidad" / "percentiles_hbm",
    salidas_sens / "04_sensibilidad" / "incidencia",
    salidas_sens / "04_sensibilidad" / "valoracion",
    salidas_sens / "04_sensibilidad" / "escenarios",
    salidas_sens / "05_complementarios",
    salidas_sens / "05_complementarios" / "no2",
    salidas_sens / "05_complementarios" / "o3",
    salidas_sens / "06_figuras",
    salidas_sens / "07_tablas",
    salidas_sens / "08_resumenes_finales",
]

# ============================================================
# BORRADO
# ============================================================
print("\n" + "=" * 70)
print("CARPETAS A BORRAR")
print("=" * 70)

for folder in folders_to_delete:
    if folder.exists():
        print(f"[BORRAR] {folder}")
        if not DRY_RUN:
            shutil.rmtree(folder)
    else:
        print(f"[NO EXISTE] {folder}")

# ============================================================
# CREACIÓN
# ============================================================
print("\n" + "=" * 70)
print("CARPETAS A CREAR")
print("=" * 70)

for folder in folders_to_create:
    print(f"[CREAR] {folder}")
    if not DRY_RUN:
        folder.mkdir(parents=True, exist_ok=True)

print("\nProceso terminado.")
if DRY_RUN:
    print("No se borró ni creó nada porque DRY_RUN=True.")
else:
    print("Se borraron las carpetas viejas y se creó la nueva estructura.")

Raíz del proyecto: D:\TRABAJO DE GRADO BEN-MAP
Carpeta objetivo : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD
Modo DRY_RUN     : False

CARPETAS A BORRAR
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_corrida_base
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos_consolidados
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos_finales_hbm
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_corrida_base
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_escenarios
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_sensibilidad
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_figuras
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_resultados_finales
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\05_tablas
[BORRAR] D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\06_logs

CARPETAS A CREAR
[CREAR

## Paso 1. Consolidación de insumos base para el nuevo pipeline de sensibilidad

En esta etapa se reorganizan los insumos mínimos necesarios para reconstruir el flujo sanitario, económico y de sensibilidad dentro de una estructura única y trazable. La idea no es volver a generar todavía superficies, impactos o resultados finales, sino dejar una base ordenada de trabajo a partir de los archivos ya construidos en etapas previas del proyecto.

Para ello, esta celda identifica y copia a la carpeta `SALIDAS_SENSIBILIDAD/01_insumos` los archivos fundamentales del análisis: la grilla espacial, la población consolidada, la incidencia base, las superficies de exposición para PM2.5, NO2 y O3, y los archivos de funciones sanitarias disponibles. La copia se hace sin alterar los archivos originales, con el fin de preservar la reproducibilidad del proyecto y evitar pérdidas accidentales de información.

El resultado esperado es una primera capa organizada de insumos, separada por tipo de fuente (`grilla`, `poblacion`, `incidencia`, `benmap`, `hbm`), junto con un manifiesto en formato JSON que deja registro exacto de qué archivos fueron localizados y copiados para alimentar el nuevo pipeline.

In [3]:
# ============================================================
# PASO 1: consolidación de insumos base
# Copia insumos clave a SALIDAS_SENSIBILIDAD/01_insumos
# sin modificar los archivos originales
# ============================================================

from pathlib import Path
import shutil
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Detectar raíz del proyecto
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_benmap = project_root / "SALIDAS_BENMAP"
resultados_hbm = project_root / "RESULTADOS_HBM"
salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"

insumos_root = salidas_sens / "01_insumos"
dest_grilla = insumos_root / "grilla"
dest_poblacion = insumos_root / "poblacion"
dest_incidencia = insumos_root / "incidencia"
dest_benmap = insumos_root / "benmap"
dest_hbm = insumos_root / "hbm"
logs_dir = salidas_sens / "00_logs_validacion"

for folder in [dest_grilla, dest_poblacion, dest_incidencia, dest_benmap, dest_hbm, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto :", project_root)
print("SALIDAS_BENMAP    :", salidas_benmap)
print("RESULTADOS_HBM    :", resultados_hbm)
print("SALIDAS_SENSIB... :", salidas_sens)

# ------------------------------------------------------------
# 2) Definir archivos esperados
# ------------------------------------------------------------
expected_files = {
    "grilla": [
        salidas_benmap / "benmap_grid_definition_final_ok_debug.csv",
        salidas_benmap / "benmap_grid_definition_summary.csv",
    ],
    "poblacion": [
        salidas_benmap / "benmap_population_2026_long_ready.csv",
        salidas_benmap / "benmap_population_2026.csv",
        salidas_benmap / "population_bogota_rangos_2026.csv",
    ],
    "incidencia": [
        salidas_benmap / "benmap_incidence_ready_only.csv",
        salidas_benmap / "benmap_incidence_2026.csv",
    ],
    "benmap_superficies": [
        salidas_benmap / "benmap_PM25_baseline.csv",
        salidas_benmap / "benmap_PM25_control.csv",
        salidas_benmap / "benmap_NO2_baseline.csv",
        salidas_benmap / "benmap_NO2_control.csv",
        salidas_benmap / "benmap_O3_baseline.csv",
        salidas_benmap / "benmap_O3_control.csv",
    ],
    "funciones_salud": [
        salidas_benmap / "benmap_health_impact_functions_import_full_v4.csv",
        salidas_benmap / "benmap_health_impact_functions_import_full_v3.csv",
        salidas_benmap / "benmap_health_impact_functions_import_full_v2.csv",
        salidas_benmap / "benmap_health_impact_functions_import_full.csv",
        salidas_benmap / "benmap_health_impact_functions_ready_only.csv",
        salidas_benmap / "benmap_health_impact_functions_ready_only_working.csv",
        salidas_benmap / "benmap_health_impact_functions_working.csv",
    ],
}

# ------------------------------------------------------------
# 3) Buscar algunos insumos HBM dentro de RESULTADOS_HBM
#    (copia todo archivo csv/xlsx/json/txt que tenga nombre útil)
# ------------------------------------------------------------
hbm_patterns = [
    "*pm25*.csv", "*PM25*.csv",
    "*no2*.csv", "*NO2*.csv",
    "*o3*.csv",  "*O3*.csv",
    "*.json", "*.txt", "*.xlsx"
]

hbm_candidates = []
if resultados_hbm.exists():
    seen = set()
    for pattern in hbm_patterns:
        for p in resultados_hbm.rglob(pattern):
            if p.is_file():
                key = str(p.resolve()).lower()
                if key not in seen:
                    seen.add(key)
                    hbm_candidates.append(p.resolve())

# ------------------------------------------------------------
# 4) Función segura de copia
# ------------------------------------------------------------
copied = []
missing = []

def copy_if_exists(src: Path, dst_folder: Path, tag: str):
    if src.exists() and src.is_file():
        dst = dst_folder / src.name
        shutil.copy2(src, dst)
        copied.append({
            "tag": tag,
            "source": str(src),
            "destination": str(dst)
        })
        print(f"[COPIADO] {src.name}  -->  {dst_folder}")
    else:
        missing.append({
            "tag": tag,
            "source": str(src)
        })
        print(f"[FALTA]   {src}")

# ------------------------------------------------------------
# 5) Copiar archivos BENMAP / base
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("COPIA DE INSUMOS BASE")
print("=" * 70)

for src in expected_files["grilla"]:
    copy_if_exists(src, dest_grilla, "grilla")

for src in expected_files["poblacion"]:
    copy_if_exists(src, dest_poblacion, "poblacion")

for src in expected_files["incidencia"]:
    copy_if_exists(src, dest_incidencia, "incidencia")

for src in expected_files["benmap_superficies"]:
    copy_if_exists(src, dest_benmap, "superficie_benmap")

for src in expected_files["funciones_salud"]:
    copy_if_exists(src, dest_benmap, "funcion_salud")

# ------------------------------------------------------------
# 6) Copiar candidatos HBM
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("COPIA DE INSUMOS HBM")
print("=" * 70)

if hbm_candidates:
    for src in hbm_candidates:
        try:
            dst = dest_hbm / src.name
            shutil.copy2(src, dst)
            copied.append({
                "tag": "hbm",
                "source": str(src),
                "destination": str(dst)
            })
            print(f"[COPIADO] {src.name}  -->  {dest_hbm}")
        except Exception as e:
            print(f"[ERROR]   {src.name}: {e}")
else:
    print("No se detectaron archivos candidatos dentro de RESULTADOS_HBM.")

# ------------------------------------------------------------
# 7) Guardar manifiesto
# ------------------------------------------------------------
manifest = {
    "timestamp": datetime.now().isoformat(),
    "project_root": str(project_root),
    "copied_count": len(copied),
    "missing_count": len(missing),
    "copied_files": copied,
    "missing_files": missing,
}

manifest_path = logs_dir / "01_manifiesto_consolidacion_insumos.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print("RESUMEN")
print("=" * 70)
print("Archivos copiados :", len(copied))
print("Archivos faltantes:", len(missing))
print("Manifiesto        :", manifest_path)

print("\nSubcarpetas destino:")
print("-", dest_grilla)
print("-", dest_poblacion)
print("-", dest_incidencia)
print("-", dest_benmap)
print("-", dest_hbm)

Raíz del proyecto : D:\TRABAJO DE GRADO BEN-MAP
SALIDAS_BENMAP    : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP
RESULTADOS_HBM    : D:\TRABAJO DE GRADO BEN-MAP\RESULTADOS_HBM
SALIDAS_SENSIB... : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD

COPIA DE INSUMOS BASE
[COPIADO] benmap_grid_definition_final_ok_debug.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\grilla
[COPIADO] benmap_grid_definition_summary.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\grilla
[COPIADO] benmap_population_2026_long_ready.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\poblacion
[COPIADO] benmap_population_2026.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\poblacion
[COPIADO] population_bogota_rangos_2026.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\poblacion
[COPIADO] benmap_incidence_ready_only.csv  -->  D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\incidencia
[COPIADO] benma

## Paso 2. Validación estructural e inventario de los insumos consolidados

En esta etapa se revisa la consistencia básica de los archivos que fueron copiados a la carpeta `SALIDAS_SENSIBILIDAD/01_insumos`. El objetivo no es todavía construir la base analítica ni calcular impactos, sino verificar que los insumos mínimos del pipeline estén presentes, que sus columnas principales sean reconocibles y que exista compatibilidad preliminar entre la grilla, la población, la incidencia y las superficies de exposición.

La validación se centra en cuatro componentes esenciales. Primero, la grilla espacial, que debe contener identificadores de celda y coordenadas discretas de fila y columna. Segundo, la población, que debe incluir año, población, sexo y grupo etario. Tercero, la incidencia, que debe contener endpoint, grupo de endpoint, edades y valor de la tasa. Cuarto, las superficies de exposición por contaminante, que deben conservar al menos el identificador espacial y la variable de concentración correspondiente al escenario baseline o control.

Además de imprimir un resumen en consola, esta celda genera un inventario de validación en formato JSON y CSV. Ese inventario servirá como punto de auditoría para saber exactamente con qué estructura arrancó el nuevo pipeline antes de pasar a la construcción de la base analítica.

In [4]:
# ============================================================
# PASO 2: validación estructural e inventario de insumos
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Detectar raíz del proyecto
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
logs_dir = salidas_sens / "00_logs_validacion"

paths = {
    "grilla": insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv",
    "poblacion": insumos_root / "poblacion" / "benmap_population_2026_long_ready.csv",
    "incidencia": insumos_root / "incidencia" / "benmap_incidence_ready_only.csv",
    "pm25_baseline": insumos_root / "benmap" / "benmap_PM25_baseline.csv",
    "pm25_control": insumos_root / "benmap" / "benmap_PM25_control.csv",
    "no2_baseline": insumos_root / "benmap" / "benmap_NO2_baseline.csv",
    "no2_control": insumos_root / "benmap" / "benmap_NO2_control.csv",
    "o3_baseline": insumos_root / "benmap" / "benmap_O3_baseline.csv",
    "o3_control": insumos_root / "benmap" / "benmap_O3_control.csv",
}

print("Raíz del proyecto :", project_root)
print("Insumos           :", insumos_root)
print("Logs              :", logs_dir)

# ------------------------------------------------------------
# 2) Funciones auxiliares
# ------------------------------------------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

def read_csv_safe(path: Path, nrows=None) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", nrows=nrows)
    except Exception:
        df = pd.read_csv(path, encoding="latin1", nrows=nrows)
    return clean_columns(df)

def summarize_file(name: str, path: Path, key_columns=None):
    result = {
        "nombre": name,
        "ruta": str(path),
        "existe": path.exists(),
        "shape": None,
        "columnas": [],
        "nulos_columnas_clave": {},
        "unicos_columnas_clave": {},
    }

    if not path.exists():
        return result

    df = read_csv_safe(path)
    result["shape"] = list(df.shape)
    result["columnas"] = df.columns.tolist()

    if key_columns:
        for col in key_columns:
            if col in df.columns:
                result["nulos_columnas_clave"][col] = int(df[col].isna().sum())
                try:
                    result["unicos_columnas_clave"][col] = int(df[col].nunique())
                except Exception:
                    result["unicos_columnas_clave"][col] = None
            else:
                result["nulos_columnas_clave"][col] = None
                result["unicos_columnas_clave"][col] = None

    return result

# ------------------------------------------------------------
# 3) Verificar existencia de archivos críticos
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("CHEQUEO DE EXISTENCIA")
print("=" * 70)

missing_critical = []
for k, v in paths.items():
    status = "OK" if v.exists() else "FALTA"
    print(f"[{status}] {k}: {v}")
    if not v.exists():
        missing_critical.append(k)

# ------------------------------------------------------------
# 4) Resumen estructural de archivos principales
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("RESUMEN ESTRUCTURAL")
print("=" * 70)

summaries = []

summaries.append(
    summarize_file(
        "grilla",
        paths["grilla"],
        key_columns=["cell_id", "ROW", "COL", "Row", "Column"]
    )
)

summaries.append(
    summarize_file(
        "poblacion",
        paths["poblacion"],
        key_columns=["Row", "Column", "Year", "Population", "Gender", "AgeRange"]
    )
)

summaries.append(
    summarize_file(
        "incidencia",
        paths["incidencia"],
        key_columns=["Endpoint Group", "Endpoint", "Start Age", "End Age", "Row", "Column", "Value"]
    )
)

for pollutant_key in [
    "pm25_baseline", "pm25_control",
    "no2_baseline", "no2_control",
    "o3_baseline", "o3_control"
]:
    expected_value_col = None
    if "pm25" in pollutant_key:
        expected_value_col = "pm25_baseline" if "baseline" in pollutant_key else "pm25_control"
    elif "no2" in pollutant_key:
        expected_value_col = "no2_baseline" if "baseline" in pollutant_key else "no2_control"
    elif "o3" in pollutant_key:
        expected_value_col = "o3_baseline" if "baseline" in pollutant_key else "o3_control"

    summaries.append(
        summarize_file(
            pollutant_key,
            paths[pollutant_key],
            key_columns=["cell_id", "fecha", "year", "month", expected_value_col]
        )
    )

for s in summaries:
    print(f"\n--- {s['nombre']} ---")
    print("Existe :", s["existe"])
    print("Shape  :", s["shape"])
    print("Columnas:")
    print(s["columnas"])
    print("Nulos columnas clave:")
    print(s["nulos_columnas_clave"])
    print("Únicos columnas clave:")
    print(s["unicos_columnas_clave"])

# ------------------------------------------------------------
# 5) Validaciones específicas mínimas
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("VALIDACIONES ESPECÍFICAS")
print("=" * 70)

checks = {}

# Grilla
if paths["grilla"].exists():
    df_grid = read_csv_safe(paths["grilla"])
    grid_cols = set(df_grid.columns)
    checks["grilla_tiene_cell_id"] = "cell_id" in grid_cols
    checks["grilla_tiene_row"] = ("ROW" in grid_cols) or ("Row" in grid_cols)
    checks["grilla_tiene_col"] = ("COL" in grid_cols) or ("Column" in grid_cols)
    checks["grilla_n_filas"] = int(len(df_grid))
else:
    checks["grilla_tiene_cell_id"] = False
    checks["grilla_tiene_row"] = False
    checks["grilla_tiene_col"] = False
    checks["grilla_n_filas"] = None

# Población
if paths["poblacion"].exists():
    df_pop = read_csv_safe(paths["poblacion"])
    pop_cols = set(df_pop.columns)
    checks["poblacion_tiene_campos_minimos"] = all(
        c in pop_cols for c in ["Row", "Column", "Year", "Population", "Gender", "AgeRange"]
    )
    checks["poblacion_anios"] = sorted(pd.to_numeric(df_pop["Year"], errors="coerce").dropna().unique().tolist()) if "Year" in df_pop.columns else []
else:
    checks["poblacion_tiene_campos_minimos"] = False
    checks["poblacion_anios"] = []

# Incidencia
if paths["incidencia"].exists():
    df_inc = read_csv_safe(paths["incidencia"])
    inc_cols = set(df_inc.columns)
    checks["incidencia_tiene_campos_minimos"] = all(
        c in inc_cols for c in ["Endpoint Group", "Endpoint", "Start Age", "End Age", "Row", "Column", "Value"]
    )
    if checks["incidencia_tiene_campos_minimos"]:
        endpoints = (
            df_inc[["Endpoint Group", "Endpoint", "Start Age", "End Age"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        checks["incidencia_n_endpoints_unicos"] = int(len(endpoints))
    else:
        checks["incidencia_n_endpoints_unicos"] = None
else:
    checks["incidencia_tiene_campos_minimos"] = False
    checks["incidencia_n_endpoints_unicos"] = None

# Superficies
for key in ["pm25_baseline", "pm25_control", "no2_baseline", "no2_control", "o3_baseline", "o3_control"]:
    if paths[key].exists():
        df_tmp = read_csv_safe(paths[key], nrows=10)
        cols = set(df_tmp.columns)
        checks[f"{key}_tiene_cell_id"] = "cell_id" in cols
        checks[f"{key}_tiene_fecha"] = "fecha" in cols
        checks[f"{key}_tiene_year"] = "year" in cols
    else:
        checks[f"{key}_tiene_cell_id"] = False
        checks[f"{key}_tiene_fecha"] = False
        checks[f"{key}_tiene_year"] = False

for k, v in checks.items():
    print(f"{k}: {v}")

# ------------------------------------------------------------
# 6) Inventario de archivos por carpeta
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("INVENTARIO DE ARCHIVOS")
print("=" * 70)

inventory_rows = []
for subfolder in ["grilla", "poblacion", "incidencia", "benmap", "hbm"]:
    folder = insumos_root / subfolder
    if folder.exists():
        for f in sorted(folder.iterdir()):
            if f.is_file():
                inventory_rows.append({
                    "subcarpeta": subfolder,
                    "archivo": f.name,
                    "extension": f.suffix.lower(),
                    "ruta": str(f),
                    "tamano_bytes": f.stat().st_size
                })

inventory_df = pd.DataFrame(inventory_rows)

print("Número de archivos inventariados:", len(inventory_df))
display(inventory_df.head(20))

# ------------------------------------------------------------
# 7) Exportar resultados de validación
# ------------------------------------------------------------
logs_dir.mkdir(parents=True, exist_ok=True)

inventory_csv = logs_dir / "02_inventario_insumos.csv"
summary_json = logs_dir / "02_validacion_estructural_insumos.json"

inventory_df.to_csv(inventory_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "project_root": str(project_root),
    "missing_critical": missing_critical,
    "checks": checks,
    "summaries": summaries,
    "inventory_csv": str(inventory_csv),
}

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print("Inventario CSV :", inventory_csv)
print("Resumen JSON  :", summary_json)
print("\nValidación estructural terminada.")

Raíz del proyecto : D:\TRABAJO DE GRADO BEN-MAP
Insumos           : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos
Logs              : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion

CHEQUEO DE EXISTENCIA
[OK] grilla: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\grilla\benmap_grid_definition_final_ok_debug.csv
[OK] poblacion: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\poblacion\benmap_population_2026_long_ready.csv
[OK] incidencia: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\incidencia\benmap_incidence_ready_only.csv
[OK] pm25_baseline: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\benmap\benmap_PM25_baseline.csv
[OK] pm25_control: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\benmap\benmap_PM25_control.csv
[OK] no2_baseline: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\01_insumos\benmap\benmap_NO2_baseline.csv
[OK] no2_control: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILI

,subcarpeta,archivo,extension,ruta,tamano_bytes
0,grilla,benmap_grid_definition_final_ok_debug.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2668
1,grilla,benmap_grid_definition_summary.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,14184
2,poblacion,benmap_population_2026.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,98454
3,poblacion,benmap_population_2026_long_ready.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,90166
4,poblacion,population_bogota_rangos_2026.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,326
5,incidencia,benmap_incidence_2026.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,66661
6,incidencia,benmap_incidence_ready_only.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,34122
7,benmap,benmap_health_impact_functions_import_full.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,845
8,benmap,benmap_health_impact_functions_import_full_v2.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,845
9,benmap,benmap_health_impact_functions_import_full_v3.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,815



ARCHIVOS GENERADOS
Inventario CSV : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\02_inventario_insumos.csv
Resumen JSON  : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\02_validacion_estructural_insumos.json

Validación estructural terminada.


## Paso 3. Definición de las corridas principales por contaminante

En esta etapa se fija la estructura epidemiológica del análisis para los tres contaminantes de interés, evitando imponer un mismo endpoint sanitario a todos ellos. La lógica del pipeline será construir una corrida principal por contaminante, pero usando en cada caso el endpoint que resulta más coherente con el comportamiento epidemiológico y con la evidencia disponible en el proyecto.

De esta manera, se define a PM2.5 con mortalidad por todas las causas en población de 65 a 99 años, a NO2 con asma en población de 0 a 64 años y a O3 con mortalidad respiratoria en población de 0 a 99 años. Esta decisión permite analizar cómo cambian los resultados sanitarios y económicos ante modificaciones en la calidad del aire, sin forzar combinaciones menos defendibles entre contaminante y endpoint.

La finalidad de esta celda es dejar una tabla maestra de especificaciones para las corridas, incluyendo contaminante, endpoint, grupo de endpoint, edades objetivo, aproximación poblacional disponible y criterio de valoración económica. Esa tabla servirá como referencia central para las siguientes etapas del pipeline.

In [5]:
# ============================================================
# PASO 3: definir corridas principales por contaminante
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

inc_path = insumos_root / "incidencia" / "benmap_incidence_ready_only.csv"

# ------------------------------------------------------------
# 2) Cargar incidencia para contrastar endpoints disponibles
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

df_inc = read_csv_safe(inc_path)

for c in ["Start Age", "End Age", "Row", "Column", "Value"]:
    if c in df_inc.columns:
        df_inc[c] = pd.to_numeric(df_inc[c], errors="coerce")

# ------------------------------------------------------------
# 3) Definir corridas principales
# ------------------------------------------------------------
corridas = pd.DataFrame([
    {
        "run_id": "PM25_MAIN",
        "pollutant": "PM25",
        "endpoint_group": "Mortality",
        "endpoint": "All-cause mortality",
        "start_age": 65,
        "end_age": 99,
        "population_age_proxy": "65+",
        "target_year": 2026,
        "economic_method": "VSL",
        "beta_manual": 0.008066,
        "beta_status": "fijado_manual",
        "priority": 1
    },
    {
        "run_id": "NO2_MAIN",
        "pollutant": "NO2",
        "endpoint_group": "Morbidity",
        "endpoint": "Asthma",
        "start_age": 0,
        "end_age": 64,
        "population_age_proxy": "0-4|5-14|15-44|45-64",
        "target_year": 2026,
        "economic_method": "COI_or_unit_value",
        "beta_manual": 0.003324,
        "beta_status": "fijado_manual",
        "priority": 2
    },
    {
        "run_id": "O3_MAIN",
        "pollutant": "O3",
        "endpoint_group": "Mortality",
        "endpoint": "Respiratory mortality",
        "start_age": 0,
        "end_age": 99,
        "population_age_proxy": "TOTAL",
        "target_year": 2026,
        "economic_method": "VSL",
        "beta_manual": None,
        "beta_status": "pendiente_validacion",
        "priority": 3
    }
])

# ------------------------------------------------------------
# 4) Revisar disponibilidad en incidencia
# ------------------------------------------------------------
available_endpoints = (
    df_inc[["Endpoint Group", "Endpoint", "Start Age", "End Age"]]
    .drop_duplicates()
    .sort_values(["Endpoint Group", "Endpoint", "Start Age", "End Age"])
    .reset_index(drop=True)
)

def check_incidence_match(row):
    hit = df_inc[
        (df_inc["Endpoint Group"].astype(str).str.strip() == row["endpoint_group"]) &
        (df_inc["Endpoint"].astype(str).str.strip() == row["endpoint"]) &
        (df_inc["Start Age"] == row["start_age"]) &
        (df_inc["End Age"] == row["end_age"])
    ]
    return len(hit) > 0

corridas["incidence_available"] = corridas.apply(check_incidence_match, axis=1)

# ------------------------------------------------------------
# 5) Resumen
# ------------------------------------------------------------
print("=" * 70)
print("CORRIDAS PRINCIPALES DEFINIDAS")
print("=" * 70)
display(corridas)

print("\n" + "=" * 70)
print("ENDPOINTS DISPONIBLES EN INCIDENCIA")
print("=" * 70)
display(available_endpoints)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
corridas_csv = base_root / "corridas_principales_especificacion.csv"
corridas_json = base_root / "corridas_principales_especificacion.json"
log_json = logs_dir / "03_corridas_principales_definidas.json"

corridas.to_csv(corridas_csv, index=False, encoding="utf-8-sig")
with open(corridas_json, "w", encoding="utf-8") as f:
    json.dump(corridas.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

payload = {
    "timestamp": datetime.now().isoformat(),
    "corridas_definidas": corridas.to_dict(orient="records"),
    "available_endpoints_n": int(len(available_endpoints)),
    "outputs": {
        "corridas_csv": str(corridas_csv),
        "corridas_json": str(corridas_json)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", corridas_csv)
print("-", corridas_json)
print("-", log_json)

CORRIDAS PRINCIPALES DEFINIDAS


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,population_age_proxy,target_year,economic_method,beta_manual,beta_status,priority,incidence_available
0,PM25_MAIN,PM25,Mortality,All-cause mortality,65,99,65+,2026,VSL,0.008066,fijado_manual,1,True
1,NO2_MAIN,NO2,Morbidity,Asthma,0,64,0-4|5-14|15-44|45-64,2026,COI_or_unit_value,0.003324,fijado_manual,2,False
2,O3_MAIN,O3,Mortality,Respiratory mortality,0,99,TOTAL,2026,VSL,NaN,pendiente_validacion,3,True



ENDPOINTS DISPONIBLES EN INCIDENCIA


,Endpoint Group,Endpoint,Start Age,End Age
0,Mortality,All-cause mortality,65,99
1,Mortality,Respiratory mortality,0,99



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_principales_especificacion.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_principales_especificacion.json
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\03_corridas_principales_definidas.json


## Paso 3. Definición final de las corridas base por contaminante y endpoint

En esta etapa se fija de manera definitiva la estructura de corridas que alimentará el pipeline sanitario y económico del estudio. La lógica ya no será imponer un mismo endpoint a todos los contaminantes, sino respetar la combinación que resultó más consistente en el flujo metodológico previo y en la estructuración epidemiológica ya trabajada.

De esta manera, PM2.5 se mantiene con mortalidad por todas las causas en población de 65 a 99 años, O3 se conserva con mortalidad respiratoria en población de 0 a 99 años, y NO2 se incorpora mediante dos endpoints de morbilidad respiratoria: asma en población de 0 a 64 años y enfermedad pulmonar crónica en población de 65 a 99 años. Esta decisión evita dejar a NO2 como caso incompleto y permite cerrar desde ahora una estructura coherente para los tres contaminantes.

Además, esta celda deja explícito el modo en que se obtendrá la incidencia para cada corrida. En PM2.5 y O3 se usará el archivo consolidado de incidencia ya disponible, mientras que en NO2 se utilizarán incidencias fijas heredadas del pipeline epidemiológico previamente construido para sus endpoints respiratorios. El resultado será una tabla maestra de corridas cerradas, lista para alimentar los pasos de preparación poblacional y construcción de bases analíticas.

In [6]:
# ============================================================
# PASO 3: definición final de corridas base
# PM2.5, O3 y NO2 sin dejar huecos
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

inc_path = insumos_root / "incidencia" / "benmap_incidence_ready_only.csv"

# ------------------------------------------------------------
# 2) Cargar incidencia disponible solo para referencia
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

df_inc = read_csv_safe(inc_path)

for c in ["Start Age", "End Age", "Row", "Column", "Value"]:
    if c in df_inc.columns:
        df_inc[c] = pd.to_numeric(df_inc[c], errors="coerce")

available_endpoints = (
    df_inc[["Endpoint Group", "Endpoint", "Start Age", "End Age"]]
    .drop_duplicates()
    .sort_values(["Endpoint Group", "Endpoint", "Start Age", "End Age"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3) Corridas finales cerradas
#    Nota:
#    - PM2.5 y O3 toman incidencia del archivo consolidado
#    - NO2 usa incidencias fijas del pipeline previo
# ------------------------------------------------------------
corridas = pd.DataFrame([
    {
        "run_id": "PM25_MAIN",
        "pollutant": "PM25",
        "endpoint_group": "Mortality",
        "endpoint": "All-cause mortality",
        "start_age": 65,
        "end_age": 99,
        "population_age_proxy": "65+",
        "population_mode": "age_proxy",
        "target_year": 2026,
        "economic_method": "VSL",
        "beta": 0.001094,
        "beta_se": 0.000050,
        "beta_status": "cerrado_pipeline_previo",
        "incidence_mode": "from_file",
        "incidence_value_fixed": None,
        "priority": 1,
        "notes": "Corrida central de PM2.5"
    },
    {
        "run_id": "O3_MAIN",
        "pollutant": "O3",
        "endpoint_group": "Mortality",
        "endpoint": "Respiratory mortality",
        "start_age": 0,
        "end_age": 99,
        "population_age_proxy": "TOTAL",
        "population_mode": "all_ages",
        "target_year": 2026,
        "economic_method": "VSL",
        "beta": 0.000867,
        "beta_se": 0.000304,
        "beta_status": "cerrado_pipeline_previo",
        "incidence_mode": "from_file",
        "incidence_value_fixed": None,
        "priority": 2,
        "notes": "Corrida respiratoria de O3"
    },
    {
        "run_id": "NO2_ASTHMA",
        "pollutant": "NO2",
        "endpoint_group": "Morbidity",
        "endpoint": "Asthma",
        "start_age": 0,
        "end_age": 64,
        "population_age_proxy": "0-4|5-14|15-44|45-64",
        "population_mode": "age_band_union",
        "target_year": 2026,
        "economic_method": "COI_or_unit_value",
        "beta": 0.003324,
        "beta_se": None,
        "beta_status": "cerrado_pipeline_previo",
        "incidence_mode": "fixed_constant",
        "incidence_value_fixed": 0.000167,
        "priority": 3,
        "notes": "Morbidez respiratoria NO2 - asma"
    },
    {
        "run_id": "NO2_CLD",
        "pollutant": "NO2",
        "endpoint_group": "Morbidity",
        "endpoint": "Chronic Lung Disease",
        "start_age": 65,
        "end_age": 99,
        "population_age_proxy": "65+",
        "population_mode": "age_proxy",
        "target_year": 2026,
        "economic_method": "COI_or_unit_value",
        "beta": 0.001850,
        "beta_se": None,
        "beta_status": "cerrado_pipeline_previo",
        "incidence_mode": "fixed_constant",
        "incidence_value_fixed": 0.002097,
        "priority": 4,
        "notes": "Morbidez respiratoria NO2 - enfermedad pulmonar crónica"
    }
])

# ------------------------------------------------------------
# 4) Verificación solo para corridas que usan incidencia desde archivo
# ------------------------------------------------------------
def check_incidence_from_file(row):
    if row["incidence_mode"] != "from_file":
        return "not_required"

    hit = df_inc[
        (df_inc["Endpoint Group"].astype(str).str.strip() == row["endpoint_group"]) &
        (df_inc["Endpoint"].astype(str).str.strip() == row["endpoint"]) &
        (df_inc["Start Age"] == row["start_age"]) &
        (df_inc["End Age"] == row["end_age"])
    ]
    return "available" if len(hit) > 0 else "missing"

corridas["incidence_check"] = corridas.apply(check_incidence_from_file, axis=1)

# ------------------------------------------------------------
# 5) Resumen
# ------------------------------------------------------------
print("=" * 80)
print("CORRIDAS FINALES DEFINIDAS")
print("=" * 80)
display(corridas)

print("\n" + "=" * 80)
print("ENDPOINTS DISPONIBLES EN EL ARCHIVO DE INCIDENCIA")
print("=" * 80)
display(available_endpoints)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
corridas_csv = base_root / "corridas_base_definitivas.csv"
corridas_json = base_root / "corridas_base_definitivas.json"
log_json = logs_dir / "03_corridas_base_definitivas.json"

corridas.to_csv(corridas_csv, index=False, encoding="utf-8-sig")

with open(corridas_json, "w", encoding="utf-8") as f:
    json.dump(corridas.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

payload = {
    "timestamp": datetime.now().isoformat(),
    "corridas_definitivas": corridas.to_dict(orient="records"),
    "endpoints_disponibles_en_archivo_incidencia": available_endpoints.to_dict(orient="records"),
    "outputs": {
        "corridas_csv": str(corridas_csv),
        "corridas_json": str(corridas_json)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", corridas_csv)
print("-", corridas_json)
print("-", log_json)

CORRIDAS FINALES DEFINIDAS


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,population_age_proxy,population_mode,target_year,economic_method,beta,beta_se,beta_status,incidence_mode,incidence_value_fixed,priority,notes,incidence_check
0,PM25_MAIN,PM25,Mortality,All-cause mortality,65,99,65+,age_proxy,2026,VSL,0.001094,0.000050,cerrado_pipeline_previo,from_file,NaN,1,Corrida central de PM2.5,available
1,O3_MAIN,O3,Mortality,Respiratory mortality,0,99,TOTAL,all_ages,2026,VSL,0.000867,0.000304,cerrado_pipeline_previo,from_file,NaN,2,Corrida respiratoria de O3,available
2,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,0-4|5-14|15-44|45-64,age_band_union,2026,COI_or_unit_value,0.003324,NaN,cerrado_pipeline_previo,fixed_constant,0.000167,3,Morbidez respiratoria NO2 - asma,not_required
3,NO2_CLD,NO2,Morbidity,Chronic Lung Disease,65,99,65+,age_proxy,2026,COI_or_unit_value,0.001850,NaN,cerrado_pipeline_previo,fixed_constant,0.002097,4,Morbidez respiratoria NO2 - enfermedad pulmona...,not_required



ENDPOINTS DISPONIBLES EN EL ARCHIVO DE INCIDENCIA


,Endpoint Group,Endpoint,Start Age,End Age
0,Mortality,All-cause mortality,65,99
1,Mortality,Respiratory mortality,0,99



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.json
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\03_corridas_base_definitivas.json


## Paso 3b. Corrección del beta principal de PM2.5

En esta etapa se corrige la especificación de la corrida principal de PM2.5, fijando como coeficiente epidemiológico base el valor \\(\\beta = 0.008066\\). El valor anterior de 0.001094 no se elimina del flujo metodológico, pero deja de ser el parámetro central y pasa a considerarse como candidato para análisis de sensibilidad.

La finalidad de esta celda es ajustar únicamente la parametrización de la corrida principal, sin modificar las demás corridas ya definidas para O3 y NO2. El resultado esperado es una versión corregida de la tabla maestra de corridas base, consistente con la decisión metodológica del estudio.

In [7]:
# ============================================================
# PASO 3b: corregir beta principal de PM2.5
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_csv = base_root / "corridas_base_definitivas.csv"
corridas_json = base_root / "corridas_base_definitivas.json"
log_json = logs_dir / "03b_correccion_beta_pm25.json"

if not corridas_csv.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{corridas_csv}")

# ------------------------------------------------------------
# 2) Cargar corridas
# ------------------------------------------------------------
corridas = pd.read_csv(corridas_csv, encoding="utf-8-sig")
corridas.columns = [str(c).replace("\ufeff", "").strip() for c in corridas.columns]

if "run_id" not in corridas.columns or "beta" not in corridas.columns:
    raise ValueError("El archivo de corridas no tiene las columnas esperadas ('run_id', 'beta').")

# ------------------------------------------------------------
# 3) Corregir PM25_MAIN
# ------------------------------------------------------------
mask = corridas["run_id"].astype(str).str.strip() == "PM25_MAIN"

if mask.sum() != 1:
    raise ValueError("No se encontró una única fila para PM25_MAIN.")

beta_anterior = float(corridas.loc[mask, "beta"].iloc[0])

corridas.loc[mask, "beta"] = 0.008066
corridas.loc[mask, "beta_status"] = "corregido_manual_base"
corridas.loc[mask, "notes"] = (
    "Corrida central de PM2.5. Beta base fijado en 0.008066. "
    "El valor 0.001094 se reserva para sensibilidad."
)

# opcional: guardar beta alternativo para trazabilidad si existe la columna
if "beta_sensitivity_low" not in corridas.columns:
    corridas["beta_sensitivity_low"] = pd.NA

corridas.loc[mask, "beta_sensitivity_low"] = beta_anterior

# ------------------------------------------------------------
# 4) Mostrar resultado
# ------------------------------------------------------------
print("=" * 80)
print("CORRECCIÓN BETA PM2.5")
print("=" * 80)
print(f"Beta anterior PM25_MAIN : {beta_anterior}")
print("Beta nuevo PM25_MAIN    : 0.008066")

display(corridas)

# ------------------------------------------------------------
# 5) Guardar de nuevo CSV y JSON
# ------------------------------------------------------------
corridas.to_csv(corridas_csv, index=False, encoding="utf-8-sig")

with open(corridas_json, "w", encoding="utf-8") as f:
    json.dump(corridas.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

payload = {
    "timestamp": datetime.now().isoformat(),
    "accion": "correccion_beta_pm25_main",
    "beta_anterior": beta_anterior,
    "beta_nuevo": 0.008066,
    "archivo_csv": str(corridas_csv),
    "archivo_json": str(corridas_json),
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", corridas_csv)
print("-", corridas_json)
print("-", log_json)

CORRECCIÓN BETA PM2.5
Beta anterior PM25_MAIN : 0.001094
Beta nuevo PM25_MAIN    : 0.008066


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,population_age_proxy,population_mode,target_year,economic_method,beta,beta_se,beta_status,incidence_mode,incidence_value_fixed,priority,notes,incidence_check,beta_sensitivity_low
0,PM25_MAIN,PM25,Mortality,All-cause mortality,65,99,65+,age_proxy,2026,VSL,0.008066,0.000050,corregido_manual_base,from_file,NaN,1,Corrida central de PM2.5. Beta base fijado en ...,available,0.001094
1,O3_MAIN,O3,Mortality,Respiratory mortality,0,99,TOTAL,all_ages,2026,VSL,0.000867,0.000304,cerrado_pipeline_previo,from_file,NaN,2,Corrida respiratoria de O3,available,<NA>
2,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,0-4|5-14|15-44|45-64,age_band_union,2026,COI_or_unit_value,0.003324,NaN,cerrado_pipeline_previo,fixed_constant,0.000167,3,Morbidez respiratoria NO2 - asma,not_required,<NA>
3,NO2_CLD,NO2,Morbidity,Chronic Lung Disease,65,99,65+,age_proxy,2026,COI_or_unit_value,0.001850,NaN,cerrado_pipeline_previo,fixed_constant,0.002097,4,Morbidez respiratoria NO2 - enfermedad pulmona...,not_required,<NA>



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.json
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\03b_correccion_beta_pm25.json


## Paso 3c. Ajuste fino del método de valoración económica por corrida

En esta etapa se mejora la especificación económica de las corridas base, reemplazando etiquetas genéricas por métodos de valoración más explícitos. La finalidad es evitar ambigüedad desde el inicio del pipeline y dejar claro qué tipo de monetización se aplicará a cada endpoint sanitario.

De esta forma, las corridas de mortalidad asociadas a PM2.5 y O3 conservarán el enfoque de Valor Estadístico de la Vida (VSL), mientras que las corridas de NO2 quedarán parametrizadas con valores unitarios específicos por caso de asma y por caso de enfermedad pulmonar crónica. Esta modificación no cambia todavía los montos monetarios, pero deja lista la estructura para que la etapa de valoración económica se construya sin vacíos conceptuales.

In [8]:
# ============================================================
# PASO 3c: ajustar economic_method de forma explícita
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_csv = base_root / "corridas_base_definitivas.csv"
corridas_json = base_root / "corridas_base_definitivas.json"
log_json = logs_dir / "03c_ajuste_metodo_valoracion.json"

if not corridas_csv.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{corridas_csv}")

# ------------------------------------------------------------
# 2) Cargar corridas
# ------------------------------------------------------------
corridas = pd.read_csv(corridas_csv, encoding="utf-8-sig")
corridas.columns = [str(c).replace("\ufeff", "").strip() for c in corridas.columns]

required_cols = {"run_id", "economic_method"}
missing = required_cols - set(corridas.columns)
if missing:
    raise ValueError(f"Faltan columnas en corridas: {sorted(missing)}")

# ------------------------------------------------------------
# 3) Crear columnas nuevas más explícitas
# ------------------------------------------------------------
if "valuation_parameter_name" not in corridas.columns:
    corridas["valuation_parameter_name"] = pd.NA

if "valuation_value_fixed" not in corridas.columns:
    corridas["valuation_value_fixed"] = pd.NA

if "valuation_status" not in corridas.columns:
    corridas["valuation_status"] = pd.NA

# ------------------------------------------------------------
# 4) Ajustar corrida por corrida
# ------------------------------------------------------------
# PM2.5
mask = corridas["run_id"].astype(str).str.strip() == "PM25_MAIN"
corridas.loc[mask, "economic_method"] = "VSL"
corridas.loc[mask, "valuation_parameter_name"] = "VSL_2015_USD"
corridas.loc[mask, "valuation_status"] = "definido_metodo"

# O3
mask = corridas["run_id"].astype(str).str.strip() == "O3_MAIN"
corridas.loc[mask, "economic_method"] = "VSL"
corridas.loc[mask, "valuation_parameter_name"] = "VSL_2015_USD"
corridas.loc[mask, "valuation_status"] = "definido_metodo"

# NO2 Asthma
mask = corridas["run_id"].astype(str).str.strip() == "NO2_ASTHMA"
corridas.loc[mask, "economic_method"] = "UNIT_VALUE_ASTHMA"
corridas.loc[mask, "valuation_parameter_name"] = "UNIT_VALUE_ASTHMA_2015_USD"
corridas.loc[mask, "valuation_status"] = "definido_metodo"

# NO2 CLD
mask = corridas["run_id"].astype(str).str.strip() == "NO2_CLD"
corridas.loc[mask, "economic_method"] = "UNIT_VALUE_CHRONIC_LUNG_DISEASE"
corridas.loc[mask, "valuation_parameter_name"] = "UNIT_VALUE_CLD_2015_USD"
corridas.loc[mask, "valuation_status"] = "definido_metodo"

# ------------------------------------------------------------
# 5) Mostrar resultado
# ------------------------------------------------------------
print("=" * 80)
print("AJUSTE DEL MÉTODO DE VALORACIÓN")
print("=" * 80)

display(corridas[[
    "run_id", "pollutant", "endpoint", "economic_method",
    "valuation_parameter_name", "valuation_value_fixed", "valuation_status"
]])

# ------------------------------------------------------------
# 6) Guardar
# ------------------------------------------------------------
corridas.to_csv(corridas_csv, index=False, encoding="utf-8-sig")

with open(corridas_json, "w", encoding="utf-8") as f:
    json.dump(corridas.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

payload = {
    "timestamp": datetime.now().isoformat(),
    "accion": "ajuste_metodo_valoracion",
    "archivo_csv": str(corridas_csv),
    "archivo_json": str(corridas_json)
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", corridas_csv)
print("-", corridas_json)
print("-", log_json)

AJUSTE DEL MÉTODO DE VALORACIÓN


,run_id,pollutant,endpoint,economic_method,valuation_parameter_name,valuation_value_fixed,valuation_status
0,PM25_MAIN,PM25,All-cause mortality,VSL,VSL_2015_USD,<NA>,definido_metodo
1,O3_MAIN,O3,Respiratory mortality,VSL,VSL_2015_USD,<NA>,definido_metodo
2,NO2_ASTHMA,NO2,Asthma,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,<NA>,definido_metodo
3,NO2_CLD,NO2,Chronic Lung Disease,UNIT_VALUE_CHRONIC_LUNG_DISEASE,UNIT_VALUE_CLD_2015_USD,<NA>,definido_metodo



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.json
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\03c_ajuste_metodo_valoracion.json


## Paso 3d. Corrección estructural de NO2 al esquema de hospital admissions

En esta etapa se corrige la especificación de las corridas de NO2 para que queden alineadas con la estructura epidemiológica trabajada previamente en el pipeline. En lugar de tratar estos endpoints como morbilidad genérica, se redefinen como hospital admissions, lo cual es más consistente con la lógica de BenMAP y con los parámetros sanitarios y económicos ya utilizados en el flujo anterior.

De esta forma, el endpoint de asma para NO2 pasa a representarse como hospital admissions en población de 0 a 99 años con población total, mientras que el endpoint de enfermedad pulmonar crónica se mantiene en 65 a 99 años. También se fijan explícitamente las incidencias constantes y los valores unitarios por caso que luego serán utilizados en la valoración económica.

La finalidad de esta celda es corregir únicamente la tabla maestra de corridas base, sin recalcular todavía población, incidencia, base analítica ni impactos sanitarios. Esos pasos se reconstruirán después sobre esta versión ya corregida.

In [13]:
# ============================================================
# PASO 3d: corregir NO2 al esquema Hospital Admissions
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_csv = base_root / "corridas_base_definitivas.csv"
corridas_json = base_root / "corridas_base_definitivas.json"
log_json = logs_dir / "03d_correccion_no2_hospital_admissions.json"

if not corridas_csv.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{corridas_csv}")

# ------------------------------------------------------------
# 2) Cargar corridas
# ------------------------------------------------------------
corridas = pd.read_csv(corridas_csv, encoding="utf-8-sig")
corridas.columns = [str(c).replace("\ufeff", "").strip() for c in corridas.columns]

required_cols = {"run_id", "endpoint_group", "endpoint", "start_age", "end_age",
                 "population_age_proxy", "population_mode", "economic_method",
                 "incidence_mode", "incidence_value_fixed",
                 "valuation_parameter_name", "valuation_value_fixed"}
missing = required_cols - set(corridas.columns)
if missing:
    raise ValueError(f"Faltan columnas en corridas: {sorted(missing)}")

# ------------------------------------------------------------
# 3) Corregir NO2_ASTHMA
# ------------------------------------------------------------
mask_asthma = corridas["run_id"].astype(str).str.strip() == "NO2_ASTHMA"
if mask_asthma.sum() != 1:
    raise ValueError("No se encontró una única fila para NO2_ASTHMA.")

corridas.loc[mask_asthma, "endpoint_group"] = "Hospital Admissions"
corridas.loc[mask_asthma, "endpoint"] = "Asthma"
corridas.loc[mask_asthma, "start_age"] = 0
corridas.loc[mask_asthma, "end_age"] = 99
corridas.loc[mask_asthma, "population_age_proxy"] = "TOTAL"
corridas.loc[mask_asthma, "population_mode"] = "all_ages"
corridas.loc[mask_asthma, "economic_method"] = "UNIT_VALUE_ASTHMA_HA"
corridas.loc[mask_asthma, "incidence_mode"] = "fixed_constant"
corridas.loc[mask_asthma, "incidence_value_fixed"] = 0.0001667347
corridas.loc[mask_asthma, "valuation_parameter_name"] = "UNIT_VALUE_ASTHMA_HA_2015_USD"
corridas.loc[mask_asthma, "valuation_value_fixed"] = 11232.0
corridas.loc[mask_asthma, "notes"] = (
    "NO2 hospital admissions - Asthma (0-99), all ages, "
    "alineado con pipeline previo."
)
corridas.loc[mask_asthma, "incidence_check"] = "not_required"

# ------------------------------------------------------------
# 4) Corregir NO2_CLD
# ------------------------------------------------------------
mask_cld = corridas["run_id"].astype(str).str.strip() == "NO2_CLD"
if mask_cld.sum() != 1:
    raise ValueError("No se encontró una única fila para NO2_CLD.")

corridas.loc[mask_cld, "endpoint_group"] = "Hospital Admissions"
corridas.loc[mask_cld, "endpoint"] = "Chronic Lung Disease"
corridas.loc[mask_cld, "start_age"] = 65
corridas.loc[mask_cld, "end_age"] = 99
corridas.loc[mask_cld, "population_age_proxy"] = "65+"
corridas.loc[mask_cld, "population_mode"] = "age_proxy"
corridas.loc[mask_cld, "economic_method"] = "UNIT_VALUE_CLD_HA"
corridas.loc[mask_cld, "incidence_mode"] = "fixed_constant"
corridas.loc[mask_cld, "incidence_value_fixed"] = 0.0020967572
corridas.loc[mask_cld, "valuation_parameter_name"] = "UNIT_VALUE_CLD_HA_2015_USD"
corridas.loc[mask_cld, "valuation_value_fixed"] = 15375.0
corridas.loc[mask_cld, "notes"] = (
    "NO2 hospital admissions - Chronic Lung Disease (65-99), "
    "alineado con pipeline previo."
)
corridas.loc[mask_cld, "incidence_check"] = "not_required"

# ------------------------------------------------------------
# 5) Mostrar resultado
# ------------------------------------------------------------
print("=" * 90)
print("CORRECCIÓN NO2 -> HOSPITAL ADMISSIONS")
print("=" * 90)

display(
    corridas[[
        "run_id", "pollutant", "endpoint_group", "endpoint",
        "start_age", "end_age", "population_age_proxy", "population_mode",
        "incidence_mode", "incidence_value_fixed",
        "economic_method", "valuation_parameter_name", "valuation_value_fixed"
    ]]
)

# ------------------------------------------------------------
# 6) Guardar
# ------------------------------------------------------------
corridas.to_csv(corridas_csv, index=False, encoding="utf-8-sig")

with open(corridas_json, "w", encoding="utf-8") as f:
    json.dump(corridas.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

payload = {
    "timestamp": datetime.now().isoformat(),
    "accion": "correccion_no2_hospital_admissions",
    "archivo_csv": str(corridas_csv),
    "archivo_json": str(corridas_json),
    "no2_asthma": {
        "endpoint_group": "Hospital Admissions",
        "endpoint": "Asthma",
        "start_age": 0,
        "end_age": 99,
        "population_mode": "all_ages",
        "incidence_value_fixed": 0.0001667347,
        "valuation_value_fixed": 11232.0
    },
    "no2_cld": {
        "endpoint_group": "Hospital Admissions",
        "endpoint": "Chronic Lung Disease",
        "start_age": 65,
        "end_age": 99,
        "population_mode": "age_proxy",
        "incidence_value_fixed": 0.0020967572,
        "valuation_value_fixed": 15375.0
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", corridas_csv)
print("-", corridas_json)
print("-", log_json)

CORRECCIÓN NO2 -> HOSPITAL ADMISSIONS


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,population_age_proxy,population_mode,incidence_mode,incidence_value_fixed,economic_method,valuation_parameter_name,valuation_value_fixed
0,PM25_MAIN,PM25,Mortality,All-cause mortality,65,99,65+,age_proxy,from_file,NaN,VSL,VSL_2015_USD,NaN
1,O3_MAIN,O3,Mortality,Respiratory mortality,0,99,TOTAL,all_ages,from_file,NaN,VSL,VSL_2015_USD,NaN
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,TOTAL,all_ages,fixed_constant,0.000167,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
3,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,65,99,65+,age_proxy,fixed_constant,0.002097,UNIT_VALUE_CLD_HA,UNIT_VALUE_CLD_HA_2015_USD,15375.0



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\corridas_base_definitivas.json
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\03d_correccion_no2_hospital_admissions.json


## Paso 4. Preparación automática de la población objetivo para cada corrida

En esta etapa se construye la población expuesta correspondiente a cada una de las corridas base definidas en el pipeline. La finalidad es traducir las reglas epidemiológicas de edad a una estructura operativa compatible con la grilla del estudio, de manera que cada contaminante y endpoint quede asociado a la población realmente relevante para su cálculo sanitario.

La construcción se hace de forma automática a partir del campo `population_mode` definido en la tabla maestra de corridas. Cuando la corrida utiliza `age_proxy`, se toma un único grupo etario disponible en la base poblacional, como ocurre con el grupo `65+`. Cuando utiliza `all_ages`, se agrega toda la población de la celda. Cuando utiliza `age_band_union`, se suman varios rangos etarios para representar una población compuesta, como en el caso de NO2 y asma.

El resultado de esta celda será una tabla de población objetivo por corrida y por celda, ya armonizada con la grilla espacial del estudio. Esta tabla será uno de los insumos directos para construir, en el siguiente paso, la base analítica final de cada contaminante y endpoint.

In [9]:
# ============================================================
# PASO 4: preparar población objetivo por corrida
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
pop_path = insumos_root / "poblacion" / "benmap_population_2026_long_ready.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
df_pop = read_csv_safe(pop_path)
df_grid = read_csv_safe(grid_path)

# ------------------------------------------------------------
# 3) Normalizar tipos
# ------------------------------------------------------------
for c in ["Row", "Column", "Year", "Population"]:
    if c in df_pop.columns:
        df_pop[c] = pd.to_numeric(df_pop[c], errors="coerce")

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_grid = df_grid.rename(columns=rename_grid)

for c in ["cell_id", "Row", "Column"]:
    if c in df_grid.columns:
        df_grid[c] = pd.to_numeric(df_grid[c], errors="coerce")

grid_master = (
    df_grid[["cell_id", "Row", "Column"]]
    .dropna()
    .drop_duplicates(subset=["cell_id"])
    .copy()
)

grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 4) Función para construir población por corrida
# ------------------------------------------------------------
def build_population_for_run(run_row, pop_df):
    run_id = run_row["run_id"]
    target_year = int(run_row["target_year"])
    population_mode = str(run_row["population_mode"]).strip()
    proxy = str(run_row["population_age_proxy"]).strip()

    tmp = pop_df[pop_df["Year"] == target_year].copy()

    if population_mode == "age_proxy":
        tmp = tmp[tmp["AgeRange"].astype(str).str.strip() == proxy].copy()

    elif population_mode == "all_ages":
        # no filtra AgeRange; suma todos los rangos de edad
        tmp = tmp.copy()

    elif population_mode == "age_band_union":
        selected_bands = [x.strip() for x in proxy.split("|") if x.strip()]
        tmp = tmp[tmp["AgeRange"].astype(str).str.strip().isin(selected_bands)].copy()

    else:
        raise ValueError(f"population_mode no reconocido para {run_id}: {population_mode}")

    out = (
        tmp.groupby(["Row", "Column"], as_index=False)["Population"]
        .sum()
        .rename(columns={"Population": "population_target"})
    )

    out["run_id"] = run_id
    out["target_year"] = target_year
    out["population_mode"] = population_mode
    out["population_age_proxy"] = proxy

    return out

# ------------------------------------------------------------
# 5) Construir para todas las corridas
# ------------------------------------------------------------
population_runs = []

for _, run_row in corridas.iterrows():
    run_pop = build_population_for_run(run_row, df_pop)
    run_pop = run_pop.merge(grid_master, on=["Row", "Column"], how="left")
    population_runs.append(run_pop)

population_target_all = pd.concat(population_runs, ignore_index=True)

# Orden
population_target_all = population_target_all[
    ["run_id", "cell_id", "Row", "Column", "target_year",
     "population_mode", "population_age_proxy", "population_target"]
].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

# ------------------------------------------------------------
# 6) Validaciones
# ------------------------------------------------------------
summary_rows = []

for run_id, g in population_target_all.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "population_total": float(g["population_target"].sum()),
        "population_min": float(g["population_target"].min()),
        "population_max": float(g["population_target"].max()),
        "missing_cell_id": int(g["cell_id"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("POBLACIÓN OBJETIVO POR CORRIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(population_target_all.head(20))

# ------------------------------------------------------------
# 7) Exportar
# ------------------------------------------------------------
pop_targets_csv = base_root / "population_targets_by_run.csv"
pop_summary_csv = base_root / "population_targets_summary.csv"
log_json = logs_dir / "04_population_targets_by_run.json"

population_target_all.to_csv(pop_targets_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(pop_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "population_targets_shape": list(population_target_all.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "population_targets_csv": str(pop_targets_csv),
        "population_summary_csv": str(pop_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", pop_targets_csv)
print("-", pop_summary_csv)
print("-", log_json)

POBLACIÓN OBJETIVO POR CORRIDA


,run_id,n_rows,n_unique_cells,population_total,population_min,population_max,missing_cell_id
0,NO2_ASTHMA,254,254,14351383.0,27.0,78998.0,0
1,NO2_CLD,254,254,1704060.0,3.0,9380.0,0
2,O3_MAIN,254,254,16055443.0,30.0,88378.0,0
3,PM25_MAIN,254,254,1704060.0,3.0,9380.0,0



Vista previa:


,run_id,cell_id,Row,Column,target_year,population_mode,population_age_proxy,population_target
0,NO2_ASTHMA,0,1,1,2026,age_band_union,0-4|5-14|15-44|45-64,59001
1,NO2_ASTHMA,41,1,2,2026,age_band_union,0-4|5-14|15-44|45-64,47672
2,NO2_ASTHMA,82,1,3,2026,age_band_union,0-4|5-14|15-44|45-64,5524
3,NO2_ASTHMA,1,2,1,2026,age_band_union,0-4|5-14|15-44|45-64,32043
4,NO2_ASTHMA,42,2,2,2026,age_band_union,0-4|5-14|15-44|45-64,78995
5,NO2_ASTHMA,83,2,3,2026,age_band_union,0-4|5-14|15-44|45-64,34522
6,NO2_ASTHMA,2,3,1,2026,age_band_union,0-4|5-14|15-44|45-64,609
7,NO2_ASTHMA,43,3,2,2026,age_band_union,0-4|5-14|15-44|45-64,57508
8,NO2_ASTHMA,84,3,3,2026,age_band_union,0-4|5-14|15-44|45-64,77229
9,NO2_ASTHMA,125,3,4,2026,age_band_union,0-4|5-14|15-44|45-64,31767



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\population_targets_by_run.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\population_targets_summary.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\04_population_targets_by_run.json


## Paso 4b. Reconstrucción de la población objetivo después de la corrección de NO2

En esta etapa se reconstruye la tabla de población objetivo por corrida para que quede consistente con la nueva definición epidemiológica de NO2. La modificación más importante es que el endpoint de asma para NO2 ya no trabajará con la unión de grupos etarios de 0 a 64 años, sino con población total, siguiendo el esquema de hospital admissions previamente definido.

La lógica de construcción se mantiene igual: las corridas con `age_proxy` toman un grupo etario específico disponible en la base poblacional, las corridas con `all_ages` agregan toda la población de la celda, y las corridas con `age_band_union` sumarían varios grupos si alguna corrida futura lo requiriera. En esta versión corregida, NO2 y asma pasa al modo `all_ages`, mientras que PM2.5 y NO2 con enfermedad pulmonar crónica continúan usando el grupo `65+`.

El resultado esperado es una nueva tabla de población objetivo por corrida y por celda, coherente con la especificación final de las corridas base y lista para alimentar de nuevo la incidencia, la base analítica y los impactos sanitarios.

In [14]:
# ============================================================
# PASO 4b: reconstruir población objetivo por corrida
# después de corregir NO2 a Hospital Admissions
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
pop_path = insumos_root / "poblacion" / "benmap_population_2026_long_ready.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
df_pop = read_csv_safe(pop_path)
df_grid = read_csv_safe(grid_path)

# ------------------------------------------------------------
# 3) Normalizar tipos
# ------------------------------------------------------------
for c in ["Row", "Column", "Year", "Population"]:
    if c in df_pop.columns:
        df_pop[c] = pd.to_numeric(df_pop[c], errors="coerce")

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_grid = df_grid.rename(columns=rename_grid)

for c in ["cell_id", "Row", "Column"]:
    if c in df_grid.columns:
        df_grid[c] = pd.to_numeric(df_grid[c], errors="coerce")

grid_master = (
    df_grid[["cell_id", "Row", "Column"]]
    .dropna()
    .drop_duplicates(subset=["cell_id"])
    .copy()
)

grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 4) Función por corrida
# ------------------------------------------------------------
def build_population_for_run(run_row, pop_df):
    run_id = run_row["run_id"]
    target_year = int(run_row["target_year"])
    population_mode = str(run_row["population_mode"]).strip()
    proxy = str(run_row["population_age_proxy"]).strip()

    tmp = pop_df[pop_df["Year"] == target_year].copy()

    if population_mode == "age_proxy":
        tmp = tmp[tmp["AgeRange"].astype(str).str.strip() == proxy].copy()

    elif population_mode == "all_ages":
        tmp = tmp.copy()

    elif population_mode == "age_band_union":
        selected_bands = [x.strip() for x in proxy.split("|") if x.strip()]
        tmp = tmp[tmp["AgeRange"].astype(str).str.strip().isin(selected_bands)].copy()

    else:
        raise ValueError(f"population_mode no reconocido para {run_id}: {population_mode}")

    out = (
        tmp.groupby(["Row", "Column"], as_index=False)["Population"]
        .sum()
        .rename(columns={"Population": "population_target"})
    )

    out["run_id"] = run_id
    out["target_year"] = target_year
    out["population_mode"] = population_mode
    out["population_age_proxy"] = proxy

    return out

# ------------------------------------------------------------
# 5) Reconstruir tabla completa
# ------------------------------------------------------------
population_runs = []

for _, run_row in corridas.iterrows():
    run_pop = build_population_for_run(run_row, df_pop)
    run_pop = run_pop.merge(grid_master, on=["Row", "Column"], how="left")
    population_runs.append(run_pop)

population_target_all = pd.concat(population_runs, ignore_index=True)

population_target_all = population_target_all[
    ["run_id", "cell_id", "Row", "Column", "target_year",
     "population_mode", "population_age_proxy", "population_target"]
].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

# ------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------
summary_rows = []

for run_id, g in population_target_all.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "population_total": float(g["population_target"].sum()),
        "population_min": float(g["population_target"].min()),
        "population_max": float(g["population_target"].max()),
        "missing_cell_id": int(g["cell_id"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("POBLACIÓN OBJETIVO RECONSTRUIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(population_target_all.head(20))

# ------------------------------------------------------------
# 7) Guardar
# ------------------------------------------------------------
pop_targets_csv = base_root / "population_targets_by_run.csv"
pop_summary_csv = base_root / "population_targets_summary.csv"
log_json = logs_dir / "04b_population_targets_rebuilt.json"

population_target_all.to_csv(pop_targets_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(pop_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "population_targets_shape": list(population_target_all.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "population_targets_csv": str(pop_targets_csv),
        "population_summary_csv": str(pop_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", pop_targets_csv)
print("-", pop_summary_csv)
print("-", log_json)

POBLACIÓN OBJETIVO RECONSTRUIDA


,run_id,n_rows,n_unique_cells,population_total,population_min,population_max,missing_cell_id
0,NO2_ASTHMA,254,254,16055443.0,30.0,88378.0,0
1,NO2_CLD,254,254,1704060.0,3.0,9380.0,0
2,O3_MAIN,254,254,16055443.0,30.0,88378.0,0
3,PM25_MAIN,254,254,1704060.0,3.0,9380.0,0



Vista previa:


,run_id,cell_id,Row,Column,target_year,population_mode,population_age_proxy,population_target
0,NO2_ASTHMA,0,1,1,2026,all_ages,TOTAL,66007
1,NO2_ASTHMA,41,1,2,2026,all_ages,TOTAL,53333
2,NO2_ASTHMA,82,1,3,2026,all_ages,TOTAL,6180
3,NO2_ASTHMA,1,2,1,2026,all_ages,TOTAL,35848
4,NO2_ASTHMA,42,2,2,2026,all_ages,TOTAL,88375
5,NO2_ASTHMA,83,2,3,2026,all_ages,TOTAL,38621
6,NO2_ASTHMA,2,3,1,2026,all_ages,TOTAL,682
7,NO2_ASTHMA,43,3,2,2026,all_ages,TOTAL,64336
8,NO2_ASTHMA,84,3,3,2026,all_ages,TOTAL,86400
9,NO2_ASTHMA,125,3,4,2026,all_ages,TOTAL,35539



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\population_targets_by_run.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\population_targets_summary.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\04b_population_targets_rebuilt.json


## Paso 5. Preparación de la incidencia objetivo para cada corrida

En esta etapa se construye la incidencia base que se utilizará en cada una de las corridas del estudio. La finalidad es armonizar en una sola estructura tanto las incidencias extraídas del archivo epidemiológico consolidado como las incidencias fijas definidas para los endpoints respiratorios de NO2.

Cuando la corrida utiliza el modo `from_file`, la incidencia se extrae directamente del archivo consolidado a partir del endpoint, grupo de endpoint y rango etario correspondiente. Cuando la corrida utiliza el modo `fixed_constant`, se asigna un valor constante a todas las celdas de la grilla, manteniendo así la lógica epidemiológica previamente definida para NO2 sin dejar vacíos en el pipeline.

El resultado será una tabla de incidencia objetivo por corrida y por celda, totalmente alineada con la grilla del estudio. Esta tabla será insumo directo de la base analítica final, junto con la población objetivo y las superficies de exposición.

In [10]:
# ============================================================
# PASO 5: preparar incidencia objetivo por corrida
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
inc_path = insumos_root / "incidencia" / "benmap_incidence_ready_only.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
df_inc = read_csv_safe(inc_path)
df_grid = read_csv_safe(grid_path)

# ------------------------------------------------------------
# 3) Normalizar tipos
# ------------------------------------------------------------
for c in ["Start Age", "End Age", "Row", "Column", "Value"]:
    if c in df_inc.columns:
        df_inc[c] = pd.to_numeric(df_inc[c], errors="coerce")

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_grid = df_grid.rename(columns=rename_grid)

for c in ["cell_id", "Row", "Column"]:
    if c in df_grid.columns:
        df_grid[c] = pd.to_numeric(df_grid[c], errors="coerce")

grid_master = (
    df_grid[["cell_id", "Row", "Column"]]
    .dropna()
    .drop_duplicates(subset=["cell_id"])
    .copy()
)

grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 4) Construir incidencia por corrida
# ------------------------------------------------------------
incidence_runs = []

for _, run in corridas.iterrows():
    run_id = run["run_id"]
    mode = str(run["incidence_mode"]).strip()

    if mode == "from_file":
        tmp = df_inc[
            (df_inc["Endpoint Group"].astype(str).str.strip() == str(run["endpoint_group"]).strip()) &
            (df_inc["Endpoint"].astype(str).str.strip() == str(run["endpoint"]).strip()) &
            (df_inc["Start Age"] == pd.to_numeric(run["start_age"], errors="coerce")) &
            (df_inc["End Age"] == pd.to_numeric(run["end_age"], errors="coerce"))
        ][["Row", "Column", "Value"]].copy()

        tmp = (
            tmp.dropna()
            .drop_duplicates()
            .rename(columns={"Value": "incidence_target"})
        )

        tmp["run_id"] = run_id

    elif mode == "fixed_constant":
        fixed_val = pd.to_numeric(run["incidence_value_fixed"], errors="coerce")
        if pd.isna(fixed_val):
            raise ValueError(f"Incidencia fija faltante para {run_id}")

        tmp = grid_master[["cell_id", "Row", "Column"]].copy()
        tmp["incidence_target"] = float(fixed_val)
        tmp["run_id"] = run_id

    else:
        raise ValueError(f"Modo de incidencia no reconocido para {run_id}: {mode}")

    # Si viene desde archivo, adjuntar cell_id
    if "cell_id" not in tmp.columns:
        tmp = tmp.merge(grid_master, on=["Row", "Column"], how="left")

    incidence_runs.append(tmp)

incidence_target_all = pd.concat(incidence_runs, ignore_index=True)

# ------------------------------------------------------------
# 5) Ordenar y validar
# ------------------------------------------------------------
incidence_target_all = incidence_target_all[
    ["run_id", "cell_id", "Row", "Column", "incidence_target"]
].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

summary_rows = []
for run_id, g in incidence_target_all.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "incidence_min": float(g["incidence_target"].min()),
        "incidence_max": float(g["incidence_target"].max()),
        "incidence_mean": float(g["incidence_target"].mean()),
        "missing_cell_id": int(g["cell_id"].isna().sum()),
        "missing_incidence": int(g["incidence_target"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("INCIDENCIA OBJETIVO POR CORRIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(incidence_target_all.head(20))

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
inc_targets_csv = base_root / "incidence_targets_by_run.csv"
inc_summary_csv = base_root / "incidence_targets_summary.csv"
log_json = logs_dir / "05_incidence_targets_by_run.json"

incidence_target_all.to_csv(inc_targets_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(inc_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "incidence_targets_shape": list(incidence_target_all.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "incidence_targets_csv": str(inc_targets_csv),
        "incidence_summary_csv": str(inc_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", inc_targets_csv)
print("-", inc_summary_csv)
print("-", log_json)

INCIDENCIA OBJETIVO POR CORRIDA


,run_id,n_rows,n_unique_cells,incidence_min,incidence_max,incidence_mean,missing_cell_id,missing_incidence
0,NO2_ASTHMA,254,254,0.000167,0.000167,0.000167,0,0
1,NO2_CLD,254,254,0.002097,0.002097,0.002097,0,0
2,O3_MAIN,254,254,0.000132,0.000132,0.000132,0,0
3,PM25_MAIN,254,254,0.000064,0.000064,0.000064,0,0



Vista previa:


,run_id,cell_id,Row,Column,incidence_target
0,NO2_ASTHMA,0,1,1,0.000167
1,NO2_ASTHMA,41,1,2,0.000167
2,NO2_ASTHMA,82,1,3,0.000167
3,NO2_ASTHMA,1,2,1,0.000167
4,NO2_ASTHMA,42,2,2,0.000167
5,NO2_ASTHMA,83,2,3,0.000167
6,NO2_ASTHMA,2,3,1,0.000167
7,NO2_ASTHMA,43,3,2,0.000167
8,NO2_ASTHMA,84,3,3,0.000167
9,NO2_ASTHMA,125,3,4,0.000167



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\incidence_targets_by_run.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\incidence_targets_summary.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\05_incidence_targets_by_run.json


## Paso 5b. Reconstrucción de la incidencia objetivo después de la corrección de NO2

En esta etapa se reconstruye la tabla de incidencia objetivo por corrida para que quede plenamente consistente con la versión corregida de las corridas base. Aunque los valores fijos de incidencia definidos para NO2 no cambian, esta reconstrucción permite actualizar la trazabilidad del pipeline bajo la nueva estructura de hospital admissions y asegurar que todas las salidas intermedias queden alineadas con la tabla maestra de corridas.

La lógica de construcción se mantiene: las corridas con `from_file` extraen la incidencia desde el archivo epidemiológico consolidado según endpoint, grupo y rango etario, mientras que las corridas con `fixed_constant` asignan el mismo valor a todas las celdas de la grilla. En esta versión, PM2.5 y O3 continúan usando incidencias provenientes del archivo, y NO2 conserva sus incidencias fijas por endpoint.

El resultado será una nueva tabla de incidencia objetivo por corrida y por celda, actualizada y lista para alimentar de nuevo la base analítica final del estudio.

In [15]:
# ============================================================
# PASO 5b: reconstruir incidencia objetivo por corrida
# después de corregir NO2 a Hospital Admissions
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
inc_path = insumos_root / "incidencia" / "benmap_incidence_ready_only.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
df_inc = read_csv_safe(inc_path)
df_grid = read_csv_safe(grid_path)

# ------------------------------------------------------------
# 3) Normalizar tipos
# ------------------------------------------------------------
for c in ["Start Age", "End Age", "Row", "Column", "Value"]:
    if c in df_inc.columns:
        df_inc[c] = pd.to_numeric(df_inc[c], errors="coerce")

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_grid = df_grid.rename(columns=rename_grid)

for c in ["cell_id", "Row", "Column"]:
    if c in df_grid.columns:
        df_grid[c] = pd.to_numeric(df_grid[c], errors="coerce")

grid_master = (
    df_grid[["cell_id", "Row", "Column"]]
    .dropna()
    .drop_duplicates(subset=["cell_id"])
    .copy()
)

grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 4) Reconstruir incidencia por corrida
# ------------------------------------------------------------
incidence_runs = []

for _, run in corridas.iterrows():
    run_id = run["run_id"]
    mode = str(run["incidence_mode"]).strip()

    if mode == "from_file":
        tmp = df_inc[
            (df_inc["Endpoint Group"].astype(str).str.strip() == str(run["endpoint_group"]).strip()) &
            (df_inc["Endpoint"].astype(str).str.strip() == str(run["endpoint"]).strip()) &
            (df_inc["Start Age"] == pd.to_numeric(run["start_age"], errors="coerce")) &
            (df_inc["End Age"] == pd.to_numeric(run["end_age"], errors="coerce"))
        ][["Row", "Column", "Value"]].copy()

        tmp = (
            tmp.dropna()
            .drop_duplicates()
            .rename(columns={"Value": "incidence_target"})
        )
        tmp["run_id"] = run_id

    elif mode == "fixed_constant":
        fixed_val = pd.to_numeric(run["incidence_value_fixed"], errors="coerce")
        if pd.isna(fixed_val):
            raise ValueError(f"Incidencia fija faltante para {run_id}")

        tmp = grid_master[["cell_id", "Row", "Column"]].copy()
        tmp["incidence_target"] = float(fixed_val)
        tmp["run_id"] = run_id

    else:
        raise ValueError(f"Modo de incidencia no reconocido para {run_id}: {mode}")

    if "cell_id" not in tmp.columns:
        tmp = tmp.merge(grid_master, on=["Row", "Column"], how="left")

    incidence_runs.append(tmp)

incidence_target_all = pd.concat(incidence_runs, ignore_index=True)

# ------------------------------------------------------------
# 5) Ordenar y resumir
# ------------------------------------------------------------
incidence_target_all = incidence_target_all[
    ["run_id", "cell_id", "Row", "Column", "incidence_target"]
].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

summary_rows = []
for run_id, g in incidence_target_all.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "incidence_min": float(g["incidence_target"].min()),
        "incidence_max": float(g["incidence_target"].max()),
        "incidence_mean": float(g["incidence_target"].mean()),
        "missing_cell_id": int(g["cell_id"].isna().sum()),
        "missing_incidence": int(g["incidence_target"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("INCIDENCIA OBJETIVO RECONSTRUIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(incidence_target_all.head(20))

# ------------------------------------------------------------
# 6) Guardar
# ------------------------------------------------------------
inc_targets_csv = base_root / "incidence_targets_by_run.csv"
inc_summary_csv = base_root / "incidence_targets_summary.csv"
log_json = logs_dir / "05b_incidence_targets_rebuilt.json"

incidence_target_all.to_csv(inc_targets_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(inc_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "incidence_targets_shape": list(incidence_target_all.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "incidence_targets_csv": str(inc_targets_csv),
        "incidence_summary_csv": str(inc_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", inc_targets_csv)
print("-", inc_summary_csv)
print("-", log_json)

INCIDENCIA OBJETIVO RECONSTRUIDA


,run_id,n_rows,n_unique_cells,incidence_min,incidence_max,incidence_mean,missing_cell_id,missing_incidence
0,NO2_ASTHMA,254,254,0.000167,0.000167,0.000167,0,0
1,NO2_CLD,254,254,0.002097,0.002097,0.002097,0,0
2,O3_MAIN,254,254,0.000132,0.000132,0.000132,0,0
3,PM25_MAIN,254,254,0.000064,0.000064,0.000064,0,0



Vista previa:


,run_id,cell_id,Row,Column,incidence_target
0,NO2_ASTHMA,0,1,1,0.000167
1,NO2_ASTHMA,41,1,2,0.000167
2,NO2_ASTHMA,82,1,3,0.000167
3,NO2_ASTHMA,1,2,1,0.000167
4,NO2_ASTHMA,42,2,2,0.000167
5,NO2_ASTHMA,83,2,3,0.000167
6,NO2_ASTHMA,2,3,1,0.000167
7,NO2_ASTHMA,43,3,2,0.000167
8,NO2_ASTHMA,84,3,3,0.000167
9,NO2_ASTHMA,125,3,4,0.000167



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\incidence_targets_by_run.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\incidence_targets_summary.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\05b_incidence_targets_rebuilt.json


## Paso 6. Construcción de la base analítica final por corrida

En esta etapa se integran en una sola estructura los tres componentes centrales del análisis: exposición, población e incidencia. La finalidad es producir una base analítica final por corrida que contenga, para cada celda de la grilla, la concentración baseline, la concentración control, el cambio de concentración, la población objetivo y la incidencia correspondiente al endpoint sanitario.

Para ello, las superficies mensuales de exposición se agregan a nivel anual por celda, siguiendo la lógica ya utilizada en la corrida de referencia. Después, dichas superficies anuales se combinan con la tabla de población objetivo y con la tabla de incidencia objetivo, ambas ya preparadas específicamente para cada corrida. El resultado es una estructura totalmente armonizada, lista para aplicar las funciones concentración–respuesta.

Esta base analítica final constituye el insumo directo de la etapa de cálculo sanitario. A partir de ella será posible estimar casos evitados o atribuibles y, posteriormente, realizar la valoración económica y los ejercicios de sensibilidad del estudio.

In [11]:
# ============================================================
# PASO 6: construir base analítica final por corrida
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
pop_targets_path = base_root / "population_targets_by_run.csv"
inc_targets_path = base_root / "incidence_targets_by_run.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

surface_paths = {
    "PM25": {
        "baseline": insumos_root / "benmap" / "benmap_PM25_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_PM25_control.csv",
    },
    "NO2": {
        "baseline": insumos_root / "benmap" / "benmap_NO2_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_NO2_control.csv",
    },
    "O3": {
        "baseline": insumos_root / "benmap" / "benmap_O3_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_O3_control.csv",
    },
}

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
pop_targets = read_csv_safe(pop_targets_path)
inc_targets = read_csv_safe(inc_targets_path)
df_grid = read_csv_safe(grid_path)

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"
df_grid = df_grid.rename(columns=rename_grid)

grid_master = df_grid[["cell_id", "Row", "Column"]].copy()
for c in ["cell_id", "Row", "Column"]:
    grid_master[c] = pd.to_numeric(grid_master[c], errors="coerce")
grid_master = grid_master.dropna().drop_duplicates(subset=["cell_id"])
grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 3) Función para construir superficies anuales
# ------------------------------------------------------------
def build_annual_surface(pollutant: str):
    b_path = surface_paths[pollutant]["baseline"]
    c_path = surface_paths[pollutant]["control"]

    df_b = read_csv_safe(b_path)
    df_c = read_csv_safe(c_path)

    value_col_b = f"{pollutant.lower()}_baseline"
    value_col_c = f"{pollutant.lower()}_control"

    if value_col_b not in df_b.columns:
        raise ValueError(f"No se encontró {value_col_b} en {b_path.name}")
    if value_col_c not in df_c.columns:
        raise ValueError(f"No se encontró {value_col_c} en {c_path.name}")

    df_b["cell_id"] = pd.to_numeric(df_b["cell_id"], errors="coerce")
    df_c["cell_id"] = pd.to_numeric(df_c["cell_id"], errors="coerce")
    df_b[value_col_b] = pd.to_numeric(df_b[value_col_b], errors="coerce")
    df_c[value_col_c] = pd.to_numeric(df_c[value_col_c], errors="coerce")

    annual_b = (
        df_b.groupby("cell_id", as_index=False)[value_col_b]
        .mean()
        .rename(columns={value_col_b: "concentration_baseline"})
    )

    annual_c = (
        df_c.groupby("cell_id", as_index=False)[value_col_c]
        .mean()
        .rename(columns={value_col_c: "concentration_control"})
    )

    annual = (
        annual_b.merge(annual_c, on="cell_id", how="inner")
        .merge(grid_master, on="cell_id", how="left")
    )

    annual["pollutant"] = pollutant
    annual["delta_concentration"] = (
        annual["concentration_baseline"] - annual["concentration_control"]
    )

    return annual[[
        "pollutant", "cell_id", "Row", "Column",
        "concentration_baseline", "concentration_control", "delta_concentration"
    ]]

surface_pm25 = build_annual_surface("PM25")
surface_no2 = build_annual_surface("NO2")
surface_o3 = build_annual_surface("O3")

surface_all = pd.concat([surface_pm25, surface_no2, surface_o3], ignore_index=True)

# ------------------------------------------------------------
# 4) Unir por corrida
# ------------------------------------------------------------
analytic_runs = []

for _, run in corridas.iterrows():
    run_id = run["run_id"]
    pollutant = run["pollutant"]

    surf = surface_all[surface_all["pollutant"] == pollutant].copy()
    pop = pop_targets[pop_targets["run_id"] == run_id].copy()
    inc = inc_targets[inc_targets["run_id"] == run_id].copy()

    merged = (
        surf.merge(pop[["run_id", "cell_id", "population_target"]], on="cell_id", how="left")
            .merge(inc[["run_id", "cell_id", "incidence_target"]], on="cell_id", how="left")
    )

    merged["run_id"] = run_id
    merged["endpoint_group"] = run["endpoint_group"]
    merged["endpoint"] = run["endpoint"]
    merged["start_age"] = run["start_age"]
    merged["end_age"] = run["end_age"]
    merged["economic_method"] = run["economic_method"]
    merged["beta"] = run["beta"]
    merged["beta_se"] = run["beta_se"]
    merged["valuation_parameter_name"] = run.get("valuation_parameter_name", pd.NA)
    merged["valuation_value_fixed"] = run.get("valuation_value_fixed", pd.NA)

    analytic_runs.append(merged)

base_analitica_final = pd.concat(analytic_runs, ignore_index=True)

# ------------------------------------------------------------
# 5) Ordenar y validar
# ------------------------------------------------------------
base_analitica_final = base_analitica_final[[
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "start_age", "end_age", "cell_id", "Row", "Column",
    "concentration_baseline", "concentration_control", "delta_concentration",
    "population_target", "incidence_target",
    "beta", "beta_se", "economic_method",
    "valuation_parameter_name", "valuation_value_fixed"
]].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

summary_rows = []
for run_id, g in base_analitica_final.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "delta_min": float(g["delta_concentration"].min()),
        "delta_max": float(g["delta_concentration"].max()),
        "delta_mean": float(g["delta_concentration"].mean()),
        "missing_population": int(g["population_target"].isna().sum()),
        "missing_incidence": int(g["incidence_target"].isna().sum()),
        "missing_delta": int(g["delta_concentration"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("BASE ANALÍTICA FINAL POR CORRIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(base_analitica_final.head(20))

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
base_csv = base_root / "base_analitica_final_por_corrida.csv"
summary_csv = base_root / "base_analitica_final_resumen.csv"
log_json = logs_dir / "06_base_analitica_final_por_corrida.json"

base_analitica_final.to_csv(base_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "base_analitica_shape": list(base_analitica_final.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "base_csv": str(base_csv),
        "summary_csv": str(summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", base_csv)
print("-", summary_csv)
print("-", log_json)

BASE ANALÍTICA FINAL POR CORRIDA


,run_id,n_rows,n_unique_cells,delta_min,delta_max,delta_mean,missing_population,missing_incidence,missing_delta
0,NO2_ASTHMA,254,254,0.952636,1.957764,1.433793,0,0,0
1,NO2_CLD,254,254,0.952636,1.957764,1.433793,0,0,0
2,O3_MAIN,254,254,0.638176,1.805216,1.195564,0,0,0
3,PM25_MAIN,254,254,1.053457,2.194635,1.478043,0,0,0



Vista previa:


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,cell_id,Row,Column,concentration_baseline,concentration_control,delta_concentration,population_target,incidence_target,beta,beta_se,economic_method,valuation_parameter_name,valuation_value_fixed
0,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,0,1,1,14.231315,12.808184,1.423132,59001,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
1,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,41,1,2,14.436909,12.993218,1.443691,47672,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
2,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,82,1,3,14.046768,12.642091,1.404677,5524,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
3,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,1,2,1,14.547155,13.092439,1.454715,32043,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
4,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,42,2,2,13.634199,12.270779,1.363420,78995,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
5,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,83,2,3,14.249579,12.824621,1.424958,34522,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
6,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,2,3,1,13.999116,12.599205,1.399912,609,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
7,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,43,3,2,14.465433,13.018890,1.446543,57508,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
8,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,84,3,3,14.293179,12.863861,1.429318,77229,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN
9,NO2_ASTHMA,NO2,Morbidity,Asthma,0,64,125,3,4,14.597198,13.137478,1.459720,31767,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA,UNIT_VALUE_ASTHMA_2015_USD,NaN



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\base_analitica_final_por_corrida.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\base_analitica_final_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\06_base_analitica_final_por_corrida.json


## Paso 6b. Reconstrucción de la base analítica final después de la corrección de NO2

En esta etapa se reconstruye la base analítica final por corrida para que quede totalmente consistente con la versión corregida de las corridas base. Dado que la definición de NO2 fue ajustada hacia hospital admissions y que la población objetivo de asma pasó a utilizar población total, es necesario rehacer la integración entre exposición, población e incidencia antes de recalcular los impactos sanitarios.

La lógica del procedimiento se mantiene igual a la versión anterior. Las superficies mensuales de exposición se agregan a nivel anual por celda, generando una concentración baseline, una concentración control y un cambio de concentración. Luego, esas superficies anuales se combinan con la población objetivo reconstruida y con la incidencia objetivo actualizada para cada corrida.

El resultado será una nueva base analítica final por corrida y por celda, coherente con la definición epidemiológica definitiva del estudio y lista para alimentar nuevamente el cálculo sanitario.

In [16]:
    # ============================================================
# PASO 6b: reconstruir base analítica final por corrida
# después de corregir NO2
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
base_root = salidas_sens / "02_base_analitica"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
pop_targets_path = base_root / "population_targets_by_run.csv"
inc_targets_path = base_root / "incidence_targets_by_run.csv"
grid_path = insumos_root / "grilla" / "benmap_grid_definition_final_ok_debug.csv"

surface_paths = {
    "PM25": {
        "baseline": insumos_root / "benmap" / "benmap_PM25_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_PM25_control.csv",
    },
    "NO2": {
        "baseline": insumos_root / "benmap" / "benmap_NO2_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_NO2_control.csv",
    },
    "O3": {
        "baseline": insumos_root / "benmap" / "benmap_O3_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_O3_control.csv",
    },
}

base_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
pop_targets = read_csv_safe(pop_targets_path)
inc_targets = read_csv_safe(inc_targets_path)
df_grid = read_csv_safe(grid_path)

rename_grid = {}
for c in df_grid.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_grid = df_grid.rename(columns=rename_grid)

grid_master = df_grid[["cell_id", "Row", "Column"]].copy()
for c in ["cell_id", "Row", "Column"]:
    grid_master[c] = pd.to_numeric(grid_master[c], errors="coerce")
grid_master = grid_master.dropna().drop_duplicates(subset=["cell_id"])
grid_master["cell_id"] = grid_master["cell_id"].astype(int)
grid_master["Row"] = grid_master["Row"].astype(int)
grid_master["Column"] = grid_master["Column"].astype(int)

# ------------------------------------------------------------
# 3) Superficies anuales por contaminante
# ------------------------------------------------------------
def build_annual_surface(pollutant: str):
    b_path = surface_paths[pollutant]["baseline"]
    c_path = surface_paths[pollutant]["control"]

    df_b = read_csv_safe(b_path)
    df_c = read_csv_safe(c_path)

    value_col_b = f"{pollutant.lower()}_baseline"
    value_col_c = f"{pollutant.lower()}_control"

    if value_col_b not in df_b.columns:
        raise ValueError(f"No se encontró {value_col_b} en {b_path.name}")
    if value_col_c not in df_c.columns:
        raise ValueError(f"No se encontró {value_col_c} en {c_path.name}")

    df_b["cell_id"] = pd.to_numeric(df_b["cell_id"], errors="coerce")
    df_c["cell_id"] = pd.to_numeric(df_c["cell_id"], errors="coerce")
    df_b[value_col_b] = pd.to_numeric(df_b[value_col_b], errors="coerce")
    df_c[value_col_c] = pd.to_numeric(df_c[value_col_c], errors="coerce")

    annual_b = (
        df_b.groupby("cell_id", as_index=False)[value_col_b]
        .mean()
        .rename(columns={value_col_b: "concentration_baseline"})
    )

    annual_c = (
        df_c.groupby("cell_id", as_index=False)[value_col_c]
        .mean()
        .rename(columns={value_col_c: "concentration_control"})
    )

    annual = (
        annual_b.merge(annual_c, on="cell_id", how="inner")
        .merge(grid_master, on="cell_id", how="left")
    )

    annual["pollutant"] = pollutant
    annual["delta_concentration"] = (
        annual["concentration_baseline"] - annual["concentration_control"]
    )

    return annual[[
        "pollutant", "cell_id", "Row", "Column",
        "concentration_baseline", "concentration_control", "delta_concentration"
    ]]

surface_all = pd.concat(
    [
        build_annual_surface("PM25"),
        build_annual_surface("NO2"),
        build_annual_surface("O3"),
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 4) Unir por corrida
# ------------------------------------------------------------
analytic_runs = []

for _, run in corridas.iterrows():
    run_id = run["run_id"]
    pollutant = run["pollutant"]

    surf = surface_all[surface_all["pollutant"] == pollutant].copy()
    pop = pop_targets[pop_targets["run_id"] == run_id].copy()
    inc = inc_targets[inc_targets["run_id"] == run_id].copy()

    merged = (
        surf.merge(pop[["run_id", "cell_id", "population_target"]], on="cell_id", how="left")
            .merge(inc[["run_id", "cell_id", "incidence_target"]], on="cell_id", how="left")
    )

    merged["run_id"] = run_id
    merged["endpoint_group"] = run["endpoint_group"]
    merged["endpoint"] = run["endpoint"]
    merged["start_age"] = run["start_age"]
    merged["end_age"] = run["end_age"]
    merged["economic_method"] = run["economic_method"]
    merged["beta"] = run["beta"]
    merged["beta_se"] = run["beta_se"]
    merged["valuation_parameter_name"] = run.get("valuation_parameter_name", pd.NA)
    merged["valuation_value_fixed"] = run.get("valuation_value_fixed", pd.NA)

    analytic_runs.append(merged)

base_analitica_final = pd.concat(analytic_runs, ignore_index=True)

# ------------------------------------------------------------
# 5) Ordenar y resumir
# ------------------------------------------------------------
base_analitica_final = base_analitica_final[[
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "start_age", "end_age", "cell_id", "Row", "Column",
    "concentration_baseline", "concentration_control", "delta_concentration",
    "population_target", "incidence_target",
    "beta", "beta_se", "economic_method",
    "valuation_parameter_name", "valuation_value_fixed"
]].sort_values(["run_id", "Row", "Column"]).reset_index(drop=True)

summary_rows = []
for run_id, g in base_analitica_final.groupby("run_id"):
    summary_rows.append({
        "run_id": run_id,
        "n_rows": int(len(g)),
        "n_unique_cells": int(g["cell_id"].nunique()),
        "delta_min": float(g["delta_concentration"].min()),
        "delta_max": float(g["delta_concentration"].max()),
        "delta_mean": float(g["delta_concentration"].mean()),
        "population_total": float(g["population_target"].sum()),
        "incidence_mean": float(g["incidence_target"].mean()),
        "missing_population": int(g["population_target"].isna().sum()),
        "missing_incidence": int(g["incidence_target"].isna().sum()),
        "missing_delta": int(g["delta_concentration"].isna().sum())
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("BASE ANALÍTICA FINAL RECONSTRUIDA")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(base_analitica_final.head(20))

# ------------------------------------------------------------
# 6) Guardar
# ------------------------------------------------------------
base_csv = base_root / "base_analitica_final_por_corrida.csv"
summary_csv = base_root / "base_analitica_final_resumen.csv"
log_json = logs_dir / "06b_base_analitica_final_rebuilt.json"

base_analitica_final.to_csv(base_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "base_analitica_shape": list(base_analitica_final.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "base_csv": str(base_csv),
        "summary_csv": str(summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", base_csv)
print("-", summary_csv)
print("-", log_json)

BASE ANALÍTICA FINAL RECONSTRUIDA


,run_id,n_rows,n_unique_cells,delta_min,delta_max,delta_mean,population_total,incidence_mean,missing_population,missing_incidence,missing_delta
0,NO2_ASTHMA,254,254,0.952636,1.957764,1.433793,16055443.0,0.000167,0,0,0
1,NO2_CLD,254,254,0.952636,1.957764,1.433793,1704060.0,0.002097,0,0,0
2,O3_MAIN,254,254,0.638176,1.805216,1.195564,16055443.0,0.000132,0,0,0
3,PM25_MAIN,254,254,1.053457,2.194635,1.478043,1704060.0,0.000064,0,0,0



Vista previa:


,run_id,pollutant,endpoint_group,endpoint,start_age,end_age,cell_id,Row,Column,concentration_baseline,concentration_control,delta_concentration,population_target,incidence_target,beta,beta_se,economic_method,valuation_parameter_name,valuation_value_fixed
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,0,1,1,14.231315,12.808184,1.423132,66007,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,41,1,2,14.436909,12.993218,1.443691,53333,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,82,1,3,14.046768,12.642091,1.404677,6180,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
3,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,1,2,1,14.547155,13.092439,1.454715,35848,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
4,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,42,2,2,13.634199,12.270779,1.363420,88375,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
5,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,83,2,3,14.249579,12.824621,1.424958,38621,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
6,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,2,3,1,13.999116,12.599205,1.399912,682,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
7,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,43,3,2,14.465433,13.018890,1.446543,64336,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
8,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,84,3,3,14.293179,12.863861,1.429318,86400,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0
9,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,99,125,3,4,14.597198,13.137478,1.459720,35539,0.000167,0.003324,NaN,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\base_analitica_final_por_corrida.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\base_analitica_final_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\06b_base_analitica_final_rebuilt.json


## Paso 7. Cálculo sanitario por corrida mediante función concentración–respuesta

En esta etapa se aplica la función concentración–respuesta a la base analítica final para estimar el impacto sanitario asociado al cambio en la calidad del aire. La finalidad es traducir el cambio anual de concentración de cada contaminante en casos esperados y casos evitados, utilizando la población objetivo, la incidencia base y el coeficiente epidemiológico correspondiente a cada corrida.

Se utiliza una formulación log-lineal del tipo:

\\[
RR = e^{\\beta \\Delta C}
\\]

\\[
AF_{evitada} = 1 - e^{-\\beta \\Delta C}
\\]

\\[
Casos\\ base = Incidencia \\times Población
\\]

\\[
Casos\\ evitados = Casos\\ base \\times AF_{evitada}
\\]

donde \\(\\Delta C = C_{baseline} - C_{control}\\). Bajo esta convención, un valor positivo de \\(\\Delta C\\) representa una mejora en la calidad del aire y, por tanto, un número positivo de casos evitados. El resultado de esta celda será una tabla de impactos sanitarios por corrida y por celda, junto con un resumen agregado para cada combinación contaminante-endpoint.

In [12]:
# ============================================================
# PASO 7: cálculo sanitario por corrida
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
out_main = salidas_sens / "03_corrida_principal"
logs_dir = salidas_sens / "00_logs_validacion"

base_path = base_root / "base_analitica_final_por_corrida.csv"

out_main.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not base_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{base_path}")

# ------------------------------------------------------------
# 2) Cargar base analítica
# ------------------------------------------------------------
base = pd.read_csv(base_path, encoding="utf-8-sig")
base.columns = [str(c).replace("\ufeff", "").strip() for c in base.columns]

required_cols = [
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "cell_id", "Row", "Column",
    "delta_concentration", "population_target", "incidence_target",
    "beta", "beta_se", "economic_method",
    "valuation_parameter_name", "valuation_value_fixed"
]

missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Faltan columnas en la base analítica: {missing}")

for c in ["delta_concentration", "population_target", "incidence_target", "beta", "beta_se"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# ------------------------------------------------------------
# 3) Cálculo sanitario
# ------------------------------------------------------------
health = base.copy()

health["rr"] = np.exp(health["beta"] * health["delta_concentration"])
health["af_avoided"] = 1 - np.exp(-health["beta"] * health["delta_concentration"])
health["baseline_cases"] = health["incidence_target"] * health["population_target"]
health["avoided_cases"] = health["baseline_cases"] * health["af_avoided"]

# ------------------------------------------------------------
# 4) Resumen por corrida
# ------------------------------------------------------------
summary = (
    health.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        delta_mean=("delta_concentration", "mean"),
        population_total=("population_target", "sum"),
        incidence_mean=("incidence_target", "mean"),
        beta=("beta", "first"),
        beta_se=("beta_se", "first"),
        rr_mean=("rr", "mean"),
        af_avoided_mean=("af_avoided", "mean"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
    )
)

# ------------------------------------------------------------
# 5) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("RESULTADOS SANITARIOS POR CORRIDA")
print("=" * 80)
display(summary)

print("\nVista previa por celda:")
display(
    health[[
        "run_id", "pollutant", "endpoint",
        "cell_id", "Row", "Column",
        "delta_concentration", "population_target", "incidence_target",
        "beta", "rr", "af_avoided", "baseline_cases", "avoided_cases"
    ]].head(20)
)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
health_cells_csv = out_main / "impactos_sanitarios_por_celda.csv"
health_summary_csv = out_main / "impactos_sanitarios_resumen.csv"
log_json = logs_dir / "07_impactos_sanitarios_por_corrida.json"

health.to_csv(health_cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(health_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "health_shape": list(health.shape),
    "summary_shape": list(summary.shape),
    "outputs": {
        "health_cells_csv": str(health_cells_csv),
        "health_summary_csv": str(health_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", health_cells_csv)
print("-", health_summary_csv)
print("-", log_json)

RESULTADOS SANITARIOS POR CORRIDA


,run_id,pollutant,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,delta_mean,population_total,incidence_mean,beta,beta_se,rr_mean,af_avoided_mean,baseline_cases_total,avoided_cases_total
0,NO2_ASTHMA,NO2,Morbidity,Asthma,UNIT_VALUE_ASTHMA,254,254,1.433793,14351383,0.000167,0.003324,NaN,1.004777,0.004755,2396.680961,11.382246
1,NO2_CLD,NO2,Morbidity,Chronic Lung Disease,UNIT_VALUE_CHRONIC_LUNG_DISEASE,254,254,1.433793,1704060,0.002097,0.001850,NaN,1.002656,0.002649,3573.413820,9.455233
2,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,254,254,1.195564,16055443,0.000132,0.000867,0.000304,1.001037,0.001036,2120.000000,2.204553
3,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,254,254,1.478043,1704060,0.000064,0.008066,0.000050,1.011994,0.011851,109.744592,1.299799



Vista previa por celda:


,run_id,pollutant,endpoint,cell_id,Row,Column,delta_concentration,population_target,incidence_target,beta,rr,af_avoided,baseline_cases,avoided_cases
0,NO2_ASTHMA,NO2,Asthma,0,1,1,1.423132,59001,0.000167,0.003324,1.004742,0.004719,9.853167,0.046500
1,NO2_ASTHMA,NO2,Asthma,41,1,2,1.443691,47672,0.000167,0.003324,1.004810,0.004787,7.961224,0.038113
2,NO2_ASTHMA,NO2,Asthma,82,1,3,1.404677,5524,0.000167,0.003324,1.004680,0.004658,0.922508,0.004297
3,NO2_ASTHMA,NO2,Asthma,1,2,1,1.454715,32043,0.000167,0.003324,1.004847,0.004824,5.351181,0.025813
4,NO2_ASTHMA,NO2,Asthma,42,2,2,1.363420,78995,0.000167,0.003324,1.004542,0.004522,13.192165,0.059652
5,NO2_ASTHMA,NO2,Asthma,83,2,3,1.424958,34522,0.000167,0.003324,1.004748,0.004725,5.765174,0.027243
6,NO2_ASTHMA,NO2,Asthma,2,3,1,1.399912,609,0.000167,0.003324,1.004664,0.004642,0.101703,0.000472
7,NO2_ASTHMA,NO2,Asthma,43,3,2,1.446543,57508,0.000167,0.003324,1.004820,0.004797,9.603836,0.046067
8,NO2_ASTHMA,NO2,Asthma,84,3,3,1.429318,77229,0.000167,0.003324,1.004762,0.004740,12.897243,0.061130
9,NO2_ASTHMA,NO2,Asthma,125,3,4,1.459720,31767,0.000167,0.003324,1.004864,0.004840,5.305089,0.025679



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\impactos_sanitarios_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\impactos_sanitarios_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\07_impactos_sanitarios_por_corrida.json


## Paso 7b. Recalcular los impactos sanitarios después de la corrección de NO2

En esta etapa se recalculan los impactos sanitarios por corrida utilizando la base analítica final ya reconstruida. La finalidad es asegurar que los resultados de casos base y casos evitados sean plenamente consistentes con la versión definitiva de las corridas del estudio, en particular con la redefinición de NO2 bajo el esquema de hospital admissions y con la población total en el endpoint de asma.

La formulación sigue siendo log-lineal. A partir del cambio de concentración anual, la población objetivo, la incidencia base y el coeficiente epidemiológico de cada corrida, se calculan el riesgo relativo, la fracción evitada, los casos base y los casos evitados en cada celda de la grilla. Después, esos resultados se agregan para obtener un resumen por corrida.

El resultado esperado es una nueva tabla de impactos sanitarios por celda y un resumen agregado por contaminante y endpoint, ya alineados con la especificación epidemiológica final del pipeline.

In [17]:
# ============================================================
# PASO 7b: recalcular impactos sanitarios por corrida
# después de corregir NO2
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
out_main = salidas_sens / "03_corrida_principal"
logs_dir = salidas_sens / "00_logs_validacion"

base_path = base_root / "base_analitica_final_por_corrida.csv"

out_main.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not base_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{base_path}")

# ------------------------------------------------------------
# 2) Cargar base analítica reconstruida
# ------------------------------------------------------------
health = pd.read_csv(base_path, encoding="utf-8-sig")
health.columns = [str(c).replace("\ufeff", "").strip() for c in health.columns]

required_cols = [
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "cell_id", "Row", "Column",
    "delta_concentration", "population_target", "incidence_target",
    "beta", "beta_se", "economic_method",
    "valuation_parameter_name", "valuation_value_fixed"
]

missing = [c for c in required_cols if c not in health.columns]
if missing:
    raise ValueError(f"Faltan columnas en la base analítica: {missing}")

for c in ["delta_concentration", "population_target", "incidence_target", "beta", "beta_se"]:
    health[c] = pd.to_numeric(health[c], errors="coerce")

# ------------------------------------------------------------
# 3) Cálculo sanitario
# ------------------------------------------------------------
health["rr"] = np.exp(health["beta"] * health["delta_concentration"])
health["af_avoided"] = 1 - np.exp(-health["beta"] * health["delta_concentration"])
health["baseline_cases"] = health["incidence_target"] * health["population_target"]
health["avoided_cases"] = health["baseline_cases"] * health["af_avoided"]

# ------------------------------------------------------------
# 4) Resumen agregado
# ------------------------------------------------------------
summary = (
    health.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        delta_mean=("delta_concentration", "mean"),
        population_total=("population_target", "sum"),
        incidence_mean=("incidence_target", "mean"),
        beta=("beta", "first"),
        beta_se=("beta_se", "first"),
        rr_mean=("rr", "mean"),
        af_avoided_mean=("af_avoided", "mean"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
    )
)

# ------------------------------------------------------------
# 5) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("IMPACTOS SANITARIOS RECALCULADOS")
print("=" * 80)
display(summary)

print("\nVista previa por celda:")
display(
    health[[
        "run_id", "pollutant", "endpoint_group", "endpoint",
        "cell_id", "Row", "Column",
        "delta_concentration", "population_target", "incidence_target",
        "beta", "rr", "af_avoided", "baseline_cases", "avoided_cases"
    ]].head(20)
)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
health_cells_csv = out_main / "impactos_sanitarios_por_celda.csv"
health_summary_csv = out_main / "impactos_sanitarios_resumen.csv"
log_json = logs_dir / "07b_impactos_sanitarios_rebuilt.json"

health.to_csv(health_cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(health_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "health_shape": list(health.shape),
    "summary_shape": list(summary.shape),
    "outputs": {
        "health_cells_csv": str(health_cells_csv),
        "health_summary_csv": str(health_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", health_cells_csv)
print("-", health_summary_csv)
print("-", log_json)

IMPACTOS SANITARIOS RECALCULADOS


,run_id,pollutant,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,delta_mean,population_total,incidence_mean,beta,beta_se,rr_mean,af_avoided_mean,baseline_cases_total,avoided_cases_total
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,1.433793,16055443,0.000167,0.003324,NaN,1.004777,0.004755,2676.999472,12.713526
1,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,1.433793,1704060,0.002097,0.001850,NaN,1.002656,0.002649,3573.000074,9.454138
2,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,254,254,1.195564,16055443,0.000132,0.000867,0.000304,1.001037,0.001036,2120.000000,2.204553
3,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,254,254,1.478043,1704060,0.000064,0.008066,0.000050,1.011994,0.011851,109.744592,1.299799



Vista previa por celda:


,run_id,pollutant,endpoint_group,endpoint,cell_id,Row,Column,delta_concentration,population_target,incidence_target,beta,rr,af_avoided,baseline_cases,avoided_cases
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,0,1,1,1.423132,66007,0.000167,0.003324,1.004742,0.004719,11.005657,0.051939
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,41,1,2,1.443691,53333,0.000167,0.003324,1.004810,0.004787,8.892462,0.042571
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,82,1,3,1.404677,6180,0.000167,0.003324,1.004680,0.004658,1.030420,0.004800
3,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,1,2,1,1.454715,35848,0.000167,0.003324,1.004847,0.004824,5.977106,0.028832
4,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,42,2,2,1.363420,88375,0.000167,0.003324,1.004542,0.004522,14.735179,0.066629
5,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,83,2,3,1.424958,38621,0.000167,0.003324,1.004748,0.004725,6.439461,0.030429
6,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,2,3,1,1.399912,682,0.000167,0.003324,1.004664,0.004642,0.113713,0.000528
7,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,43,3,2,1.446543,64336,0.000167,0.003324,1.004820,0.004797,10.727044,0.051455
8,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,84,3,3,1.429318,86400,0.000167,0.003324,1.004762,0.004740,14.405878,0.068281
9,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,125,3,4,1.459720,35539,0.000167,0.003324,1.004864,0.004840,5.925585,0.028682



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\impactos_sanitarios_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\impactos_sanitarios_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\07b_impactos_sanitarios_rebuilt.json


## Paso 8. Valoración económica base por corrida en USD de 2015

En esta etapa se monetizan los impactos sanitarios estimados para cada corrida, utilizando el método de valoración definido previamente en la tabla maestra del pipeline. La finalidad es traducir los casos evitados en una magnitud económica comparable, manteniendo por ahora una unidad monetaria común en dólares de 2015.

Para las corridas de mortalidad asociadas a PM2.5 y O3 se utiliza el enfoque de Valor Estadístico de la Vida (VSL). Para las corridas de NO2, que corresponden a hospital admissions, se emplean valores unitarios por caso ya parametrizados para asma y enfermedad pulmonar crónica. De esta manera, cada endpoint se valora con el instrumento económico más coherente con su naturaleza sanitaria.

En esta etapa no se realiza todavía la conversión a pesos colombianos ni la actualización a un año monetario local. El objetivo inmediato es cerrar una primera valoración económica base en la lógica del pipeline, dejando para una etapa posterior la adaptación monetaria al contexto colombiano y los ejercicios de sensibilidad económica.

In [18]:
# ============================================================
# PASO 8: valoración económica base por corrida (USD 2015)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_main = salidas_sens / "03_corrida_principal"
logs_dir = salidas_sens / "00_logs_validacion"

health_path = out_main / "impactos_sanitarios_por_celda.csv"

out_main.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not health_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{health_path}")

# ------------------------------------------------------------
# 2) Cargar impactos sanitarios
# ------------------------------------------------------------
econ = pd.read_csv(health_path, encoding="utf-8-sig")
econ.columns = [str(c).replace("\ufeff", "").strip() for c in econ.columns]

required_cols = [
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "economic_method", "valuation_parameter_name", "valuation_value_fixed",
    "avoided_cases"
]
missing = [c for c in required_cols if c not in econ.columns]
if missing:
    raise ValueError(f"Faltan columnas en impactos sanitarios: {missing}")

econ["avoided_cases"] = pd.to_numeric(econ["avoided_cases"], errors="coerce")
econ["valuation_value_fixed"] = pd.to_numeric(econ["valuation_value_fixed"], errors="coerce")

# ------------------------------------------------------------
# 3) Parámetro base VSL en USD 2015
#    (se deja explícito para mortalidad)
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0

# ------------------------------------------------------------
# 4) Asignar valor unitario económico por corrida
# ------------------------------------------------------------
econ["valuation_value_used_usd_2015"] = np.nan

# PM2.5 y O3 -> VSL
mask_vsl = econ["economic_method"].astype(str).str.strip() == "VSL"
econ.loc[mask_vsl, "valuation_value_used_usd_2015"] = VSL_2015_USD

# NO2 -> valor fijo por caso ya parametrizado
mask_unit = econ["economic_method"].astype(str).str.strip().isin([
    "UNIT_VALUE_ASTHMA_HA",
    "UNIT_VALUE_CLD_HA"
])
econ.loc[mask_unit, "valuation_value_used_usd_2015"] = econ.loc[mask_unit, "valuation_value_fixed"]

# ------------------------------------------------------------
# 5) Cálculo del valor económico
# ------------------------------------------------------------
econ["economic_value_usd_2015"] = (
    econ["avoided_cases"] * econ["valuation_value_used_usd_2015"]
)

# ------------------------------------------------------------
# 6) Resumen por corrida
# ------------------------------------------------------------
summary = (
    econ.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        avoided_cases_total=("avoided_cases", "sum"),
        valuation_value_used_usd_2015=("valuation_value_used_usd_2015", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_mean_usd_2015=("economic_value_usd_2015", "mean"),
    )
)

# ------------------------------------------------------------
# 7) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("VALORACIÓN ECONÓMICA BASE (USD 2015)")
print("=" * 80)
display(summary)

print("\nVista previa por celda:")
display(
    econ[[
        "run_id", "pollutant", "endpoint",
        "cell_id", "Row", "Column",
        "avoided_cases", "economic_method",
        "valuation_parameter_name", "valuation_value_used_usd_2015",
        "economic_value_usd_2015"
    ]].head(20)
)

# ------------------------------------------------------------
# 8) Exportar
# ------------------------------------------------------------
econ_cells_csv = out_main / "valoracion_economica_por_celda_usd2015.csv"
econ_summary_csv = out_main / "valoracion_economica_resumen_usd2015.csv"
log_json = logs_dir / "08_valoracion_economica_base_usd2015.json"

econ.to_csv(econ_cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(econ_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "VSL_2015_USD": VSL_2015_USD,
    "econ_shape": list(econ.shape),
    "summary_shape": list(summary.shape),
    "outputs": {
        "econ_cells_csv": str(econ_cells_csv),
        "econ_summary_csv": str(econ_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", econ_cells_csv)
print("-", econ_summary_csv)
print("-", log_json)

VALORACIÓN ECONÓMICA BASE (USD 2015)


,run_id,pollutant,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,avoided_cases_total,valuation_value_used_usd_2015,economic_value_total_usd_2015,economic_value_mean_usd_2015
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,12.713526,11232.0,1.427983e+05,562.198130
1,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,9.454138,15375.0,1.453574e+05,572.273112
2,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,254,254,2.204553,8705114.0,1.919088e+07,75554.656875
3,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,254,254,1.299799,8705114.0,1.131490e+07,44546.846331



Vista previa por celda:


,run_id,pollutant,endpoint,cell_id,Row,Column,avoided_cases,economic_method,valuation_parameter_name,valuation_value_used_usd_2015,economic_value_usd_2015
0,NO2_ASTHMA,NO2,Asthma,0,1,1,0.051939,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,583.381073
1,NO2_ASTHMA,NO2,Asthma,41,1,2,0.042571,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,478.159413
2,NO2_ASTHMA,NO2,Asthma,82,1,3,0.004800,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,53.913247
3,NO2_ASTHMA,NO2,Asthma,1,2,1,0.028832,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,323.845226
4,NO2_ASTHMA,NO2,Asthma,42,2,2,0.066629,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,748.375246
5,NO2_ASTHMA,NO2,Asthma,83,2,3,0.030429,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,341.775979
6,NO2_ASTHMA,NO2,Asthma,2,3,1,0.000528,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,5.929513
7,NO2_ASTHMA,NO2,Asthma,43,3,2,0.051455,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,577.944194
8,NO2_ASTHMA,NO2,Asthma,84,3,3,0.068281,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,766.929427
9,NO2_ASTHMA,NO2,Asthma,125,3,4,0.028682,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,322.155542



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\valoracion_economica_por_celda_usd2015.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\valoracion_economica_resumen_usd2015.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\08_valoracion_economica_base_usd2015.json


## Paso 9. Conversión de la valoración económica a pesos colombianos y construcción de la tabla resumen para Bogotá

En esta etapa se transforma la valoración económica base desde dólares de 2015 a pesos colombianos, manteniendo una estructura flexible para que la conversión monetaria pueda ajustarse después con los parámetros oficiales que se adopten para el estudio. La finalidad es producir una primera salida económica en moneda local, coherente con el contexto de Bogotá y con la necesidad de reportar resultados interpretables para Colombia.

La conversión se hará en dos niveles. Primero, se calculará el valor económico en pesos colombianos de 2015 a partir de una tasa de cambio promedio anual definida como parámetro explícito. Segundo, se dejará habilitada la actualización a un año base local posterior mediante un factor de inflación o indexación, también parametrizado de forma explícita. De esta manera, la lógica del pipeline queda lista para incorporar más adelante la TRM y el ajuste inflacionario oficial que se seleccione.

El resultado esperado es una tabla por celda con valoración en COP, junto con un resumen agregado por corrida para Bogotá. Estas salidas convivirán con la versión en USD de 2015, de modo que el pipeline conserve trazabilidad metodológica y comparabilidad económica.

In [ ]:
# ============================================================
# PASO 9: conversión a COP y tabla resumen final para Bogotá
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) PARÁMETROS MONETARIOS
#    Reemplaza estos valores cuando definas la fuente oficial.
# ------------------------------------------------------------
FX_COP_PER_USD_2015 = 2773.43
IPC_FACTOR_2015_TO_LOCAL_BASE = 1.0
LOCAL_BASE_YEAR_LABEL = "COP_BASE_LOCAL"

if FX_COP_PER_USD_2015 is None:
    raise ValueError(
        "Debes definir FX_COP_PER_USD_2015 antes de correr esta celda. "
        "Ejemplo: FX_COP_PER_USD_2015 = 3000.0"
    )

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_main = salidas_sens / "03_corrida_principal"
out_tables = salidas_sens / "07_tablas"
out_final = salidas_sens / "08_resumenes_finales"
logs_dir = salidas_sens / "00_logs_validacion"

econ_path = out_main / "valoracion_economica_por_celda_usd2015.csv"

for folder in [out_main, out_tables, out_final, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not econ_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{econ_path}")

# ------------------------------------------------------------
# 3) Cargar valoración económica base
# ------------------------------------------------------------
econ = pd.read_csv(econ_path, encoding="utf-8-sig")
econ.columns = [str(c).replace("\ufeff", "").strip() for c in econ.columns]

required_cols = [
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "cell_id", "Row", "Column",
    "avoided_cases", "economic_method",
    "valuation_parameter_name", "valuation_value_used_usd_2015",
    "economic_value_usd_2015"
]
missing = [c for c in required_cols if c not in econ.columns]
if missing:
    raise ValueError(f"Faltan columnas en la valoración económica: {missing}")

for c in ["avoided_cases", "valuation_value_used_usd_2015", "economic_value_usd_2015"]:
    econ[c] = pd.to_numeric(econ[c], errors="coerce")

# ------------------------------------------------------------
# 4) Conversión a COP
# ------------------------------------------------------------
econ["fx_cop_per_usd_2015"] = float(FX_COP_PER_USD_2015)
econ["ipc_factor_2015_to_local_base"] = float(IPC_FACTOR_2015_TO_LOCAL_BASE)

econ["valuation_value_used_cop_2015"] = (
    econ["valuation_value_used_usd_2015"] * econ["fx_cop_per_usd_2015"]
)

econ["economic_value_cop_2015"] = (
    econ["economic_value_usd_2015"] * econ["fx_cop_per_usd_2015"]
)

econ["economic_value_cop_local_base"] = (
    econ["economic_value_cop_2015"] * econ["ipc_factor_2015_to_local_base"]
)

# ------------------------------------------------------------
# 5) Resumen por corrida
# ------------------------------------------------------------
summary = (
    econ.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        avoided_cases_total=("avoided_cases", "sum"),
        valuation_value_used_usd_2015=("valuation_value_used_usd_2015", "first"),
        valuation_value_used_cop_2015=("valuation_value_used_cop_2015", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
        economic_value_total_cop_local_base=("economic_value_cop_local_base", "sum"),
    )
)

summary["local_base_year_label"] = LOCAL_BASE_YEAR_LABEL

# ------------------------------------------------------------
# 6) Tabla ejecutiva para Bogotá
# ------------------------------------------------------------
tabla_bogota = summary[[
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "avoided_cases_total",
    "economic_value_total_usd_2015",
    "economic_value_total_cop_2015",
    "economic_value_total_cop_local_base",
    "local_base_year_label"
]].copy()

tabla_bogota = tabla_bogota.sort_values(["pollutant", "run_id"]).reset_index(drop=True)

# ------------------------------------------------------------
# 7) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("VALORACIÓN ECONÓMICA EN COP")
print("=" * 80)
print(f"FX_COP_PER_USD_2015         : {FX_COP_PER_USD_2015}")
print(f"IPC_FACTOR_2015_TO_LOCAL_BASE: {IPC_FACTOR_2015_TO_LOCAL_BASE}")
print(f"LOCAL_BASE_YEAR_LABEL        : {LOCAL_BASE_YEAR_LABEL}")

print("\nResumen por corrida:")
display(summary)

print("\nTabla resumen para Bogotá:")
display(tabla_bogota)

print("\nVista previa por celda:")
display(
    econ[[
        "run_id", "pollutant", "endpoint",
        "cell_id", "Row", "Column",
        "avoided_cases",
        "economic_value_usd_2015",
        "economic_value_cop_2015",
        "economic_value_cop_local_base"
    ]].head(20)
)

# ------------------------------------------------------------
# 8) Exportar
# ------------------------------------------------------------
econ_cells_csv = out_main / "valoracion_economica_por_celda_cop.csv"
summary_csv = out_tables / "valoracion_economica_resumen_cop.csv"
bogota_csv = out_final / "tabla_resumen_bogota_cop.csv"
log_json = logs_dir / "09_conversion_cop_y_resumen_bogota.json"

econ.to_csv(econ_cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
tabla_bogota.to_csv(bogota_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "fx_cop_per_usd_2015": FX_COP_PER_USD_2015,
    "ipc_factor_2015_to_local_base": IPC_FACTOR_2015_TO_LOCAL_BASE,
    "local_base_year_label": LOCAL_BASE_YEAR_LABEL,
    "econ_shape": list(econ.shape),
    "summary_shape": list(summary.shape),
    "bogota_shape": list(tabla_bogota.shape),
    "outputs": {
        "econ_cells_csv": str(econ_cells_csv),
        "summary_csv": str(summary_csv),
        "bogota_csv": str(bogota_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", econ_cells_csv)
print("-", summary_csv)
print("-", bogota_csv)
print("-", log_json)

VALORACIÓN ECONÓMICA EN COP
FX_COP_PER_USD_2015         : 2773.43
IPC_FACTOR_2015_TO_LOCAL_BASE: 1.0
LOCAL_BASE_YEAR_LABEL        : COP_BASE_LOCAL

Resumen por corrida:


,run_id,pollutant,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,avoided_cases_total,valuation_value_used_usd_2015,valuation_value_used_cop_2015,economic_value_total_usd_2015,economic_value_total_cop_2015,economic_value_total_cop_local_base,local_base_year_label
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,12.713526,11232.0,3.115117e+07,1.427983e+05,3.960412e+08,3.960412e+08,COP_BASE_LOCAL
1,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,9.454138,15375.0,4.264149e+07,1.453574e+05,4.031385e+08,4.031385e+08,COP_BASE_LOCAL
2,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,254,254,2.204553,8705114.0,2.414302e+10,1.919088e+07,5.322457e+10,5.322457e+10,COP_BASE_LOCAL
3,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,254,254,1.299799,8705114.0,2.414302e+10,1.131490e+07,3.138108e+10,3.138108e+10,COP_BASE_LOCAL



Tabla resumen para Bogotá:


,run_id,pollutant,endpoint_group,endpoint,avoided_cases_total,economic_value_total_usd_2015,economic_value_total_cop_2015,economic_value_total_cop_local_base,local_base_year_label
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,12.713526,1.427983e+05,3.960412e+08,3.960412e+08,COP_BASE_LOCAL
1,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,9.454138,1.453574e+05,4.031385e+08,4.031385e+08,COP_BASE_LOCAL
2,O3_MAIN,O3,Mortality,Respiratory mortality,2.204553,1.919088e+07,5.322457e+10,5.322457e+10,COP_BASE_LOCAL
3,PM25_MAIN,PM25,Mortality,All-cause mortality,1.299799,1.131490e+07,3.138108e+10,3.138108e+10,COP_BASE_LOCAL



Vista previa por celda:


,run_id,pollutant,endpoint,cell_id,Row,Column,avoided_cases,economic_value_usd_2015,economic_value_cop_2015,economic_value_cop_local_base
0,NO2_ASTHMA,NO2,Asthma,0,1,1,0.051939,583.381073,1.617967e+06,1.617967e+06
1,NO2_ASTHMA,NO2,Asthma,41,1,2,0.042571,478.159413,1.326142e+06,1.326142e+06
2,NO2_ASTHMA,NO2,Asthma,82,1,3,0.004800,53.913247,1.495246e+05,1.495246e+05
3,NO2_ASTHMA,NO2,Asthma,1,2,1,0.028832,323.845226,8.981621e+05,8.981621e+05
4,NO2_ASTHMA,NO2,Asthma,42,2,2,0.066629,748.375246,2.075566e+06,2.075566e+06
5,NO2_ASTHMA,NO2,Asthma,83,2,3,0.030429,341.775979,9.478918e+05,9.478918e+05
6,NO2_ASTHMA,NO2,Asthma,2,3,1,0.000528,5.929513,1.644509e+04,1.644509e+04
7,NO2_ASTHMA,NO2,Asthma,43,3,2,0.051455,577.944194,1.602888e+06,1.602888e+06
8,NO2_ASTHMA,NO2,Asthma,84,3,3,0.068281,766.929427,2.127025e+06,2.127025e+06
9,NO2_ASTHMA,NO2,Asthma,125,3,4,0.028682,322.155542,8.934758e+05,8.934758e+05



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\03_corrida_principal\valoracion_economica_por_celda_cop.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\valoracion_economica_resumen_cop.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\08_resumenes_finales\tabla_resumen_bogota_cop.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\09_conversion_cop_y_resumen_bogota.json


## Paso 10. Construcción de escenarios de sensibilidad HBM mediante percentiles posteriores

En esta etapa se construyen los escenarios de sensibilidad asociados a la incertidumbre del modelo jerárquico bayesiano. La finalidad es representar cómo cambia la exposición estimada cuando, en lugar de utilizar únicamente la superficie central de concentración, se emplean superficies alternativas derivadas de percentiles de la distribución posterior.

Para cada contaminante se definirán tres escenarios de exposición baseline: un escenario bajo asociado al percentil 5, un escenario central asociado a la superficie base y un escenario alto asociado al percentil 95. En esta lógica, el percentil 50 se aproxima mediante la superficie baseline ya utilizada en la corrida principal. El escenario control no cambia en esta etapa, de manera que la sensibilidad se concentra en la incertidumbre de la superficie estimada por el HBM.

El resultado será una tabla de superficies anuales por celda, contaminante y percentil, junto con el cambio de concentración frente al escenario control. Estas superficies servirán como insumo para recalcular impactos sanitarios y económicos bajo distintos niveles de incertidumbre del modelo bayesiano.

In [1]:
# ============================================================
# PASO 10: construir escenarios de sensibilidad HBM
# por percentiles p05 / p50 / p95
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
insumos_root = salidas_sens / "01_insumos"
sens_root = salidas_sens / "04_sensibilidad" / "percentiles_hbm"
logs_dir = salidas_sens / "00_logs_validacion"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

surface_paths = {
    "PM25": {
        "baseline": insumos_root / "benmap" / "benmap_PM25_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_PM25_control.csv",
    },
    "NO2": {
        "baseline": insumos_root / "benmap" / "benmap_NO2_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_NO2_control.csv",
    },
    "O3": {
        "baseline": insumos_root / "benmap" / "benmap_O3_baseline.csv",
        "control": insumos_root / "benmap" / "benmap_O3_control.csv",
    },
}

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

# ------------------------------------------------------------
# 3) Construir superficies anuales por percentil
# ------------------------------------------------------------
all_surfaces = []
summary_rows = []

for pollutant, paths in surface_paths.items():
    df_b = read_csv_safe(paths["baseline"])
    df_c = read_csv_safe(paths["control"])

    base_col = f"{pollutant.lower()}_baseline"
    p05_col = f"{pollutant.lower()}_p05"
    p95_col = f"{pollutant.lower()}_p95"
    control_col = f"{pollutant.lower()}_control"

    required_b = ["cell_id", base_col, p05_col, p95_col]
    required_c = ["cell_id", control_col]

    missing_b = [c for c in required_b if c not in df_b.columns]
    missing_c = [c for c in required_c if c not in df_c.columns]

    if missing_b:
        raise ValueError(f"Faltan columnas en baseline de {pollutant}: {missing_b}")
    if missing_c:
        raise ValueError(f"Faltan columnas en control de {pollutant}: {missing_c}")

    for col in ["cell_id", base_col, p05_col, p95_col]:
        df_b[col] = pd.to_numeric(df_b[col], errors="coerce")
    for col in ["cell_id", control_col]:
        df_c[col] = pd.to_numeric(df_c[col], errors="coerce")

    annual_control = (
        df_c.groupby("cell_id", as_index=False)[control_col]
        .mean()
        .rename(columns={control_col: "concentration_control"})
    )

    percentile_map = {
        "p05": p05_col,
        "p50": base_col,   # p50 aproximado por baseline central
        "p95": p95_col,
    }

    for percentile_label, src_col in percentile_map.items():
        annual_base = (
            df_b.groupby("cell_id", as_index=False)[src_col]
            .mean()
            .rename(columns={src_col: "concentration_baseline_sens"})
        )

        merged = annual_base.merge(annual_control, on="cell_id", how="inner")
        merged["pollutant"] = pollutant
        merged["percentile_hbm"] = percentile_label
        merged["delta_concentration_sens"] = (
            merged["concentration_baseline_sens"] - merged["concentration_control"]
        )

        all_surfaces.append(merged)

        summary_rows.append({
            "pollutant": pollutant,
            "percentile_hbm": percentile_label,
            "n_rows": int(len(merged)),
            "n_unique_cells": int(merged["cell_id"].nunique()),
            "baseline_min": float(merged["concentration_baseline_sens"].min()),
            "baseline_max": float(merged["concentration_baseline_sens"].max()),
            "control_min": float(merged["concentration_control"].min()),
            "control_max": float(merged["concentration_control"].max()),
            "delta_min": float(merged["delta_concentration_sens"].min()),
            "delta_max": float(merged["delta_concentration_sens"].max()),
            "delta_mean": float(merged["delta_concentration_sens"].mean()),
        })

surface_percentiles = pd.concat(all_surfaces, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 4) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("ESCENARIOS DE SENSIBILIDAD HBM POR PERCENTILES")
print("=" * 80)
display(summary_df)

print("\nVista previa:")
display(surface_percentiles.head(20))

# ------------------------------------------------------------
# 5) Exportar
# ------------------------------------------------------------
surfaces_csv = sens_root / "superficies_anuales_percentiles_hbm.csv"
summary_csv = sens_root / "superficies_anuales_percentiles_hbm_resumen.csv"
log_json = logs_dir / "10_superficies_percentiles_hbm.json"

surface_percentiles.to_csv(surfaces_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "surface_percentiles_shape": list(surface_percentiles.shape),
    "summary_shape": list(summary_df.shape),
    "outputs": {
        "surfaces_csv": str(surfaces_csv),
        "summary_csv": str(summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", surfaces_csv)
print("-", summary_csv)
print("-", log_json)

ESCENARIOS DE SENSIBILIDAD HBM POR PERCENTILES


,pollutant,percentile_hbm,n_rows,n_unique_cells,baseline_min,baseline_max,control_min,control_max,delta_min,delta_max,delta_mean
0,PM25,p05,254,254,2.425184,19.911712,9.481110,19.751716,-11.828708,0.196395,-9.974680
1,PM25,p50,254,254,10.534567,21.946352,9.481110,19.751716,1.053457,2.194635,1.478043
2,PM25,p95,254,254,11.686223,91.694546,9.481110,19.751716,2.205113,77.381628,63.552332
3,NO2,p05,254,254,2.209559,17.783035,8.573728,17.619880,-11.356495,0.163155,-9.736438
4,NO2,p50,254,254,9.526364,19.577644,8.573728,17.619880,0.952636,1.957764,1.433793
5,NO2,p95,254,254,10.546412,85.992611,8.573728,17.619880,1.972684,73.054952,62.386051
6,O3,p05,254,254,1.875557,16.563321,5.743586,16.246941,-9.962069,0.316380,-8.103976
7,O3,p50,254,254,6.381762,18.052157,5.743586,16.246941,0.638176,1.805216,1.195564
8,O3,p95,254,254,6.985413,77.359948,5.743586,16.246941,1.241827,65.309995,51.932743



Vista previa:


,cell_id,concentration_baseline_sens,concentration_control,pollutant,percentile_hbm,delta_concentration_sens
0,0,2.741336,13.546608,PM25,p05,-10.805271
1,1,2.625258,13.339014,PM25,p05,-10.713757
2,2,2.612143,13.305940,PM25,p05,-10.693797
3,41,2.497865,13.257479,PM25,p05,-10.759614
4,42,2.816960,12.650579,PM25,p05,-9.833619
5,43,2.620981,12.901293,PM25,p05,-10.280312
6,44,2.643437,12.920829,PM25,p05,-10.277392
7,82,2.629834,13.366787,PM25,p05,-10.736953
8,83,2.602873,13.814585,PM25,p05,-11.211712
9,84,2.709903,13.238059,PM25,p05,-10.528156



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\percentiles_hbm\superficies_anuales_percentiles_hbm.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\percentiles_hbm\superficies_anuales_percentiles_hbm_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\10_superficies_percentiles_hbm.json


## Paso 11. Recalcular impactos sanitarios y económicos bajo sensibilidad HBM

En esta etapa se cierran los escenarios de sensibilidad asociados a la incertidumbre del modelo jerárquico bayesiano. La finalidad es evaluar cómo cambian los resultados sanitarios y económicos cuando la superficie de exposición baseline se reemplaza por escenarios alternativos derivados de los percentiles posteriores del HBM, manteniendo fijo el mismo escenario control.

Para cada contaminante y para cada corrida epidemiológica se utilizarán tres escenarios: p05, p50 y p95. El escenario p50 corresponde a la superficie central previamente empleada en la corrida base, mientras que p05 y p95 representan realizaciones bajas y altas de la concentración estimada. A partir de esos escenarios se recalculan el cambio de concentración, los casos base, los casos evitados y la valoración económica.

El resultado esperado es una tabla detallada por celda y una tabla resumen por corrida y percentil. Esto permitirá identificar cuánto se desplazan la carga sanitaria y la valoración económica cuando la incertidumbre del HBM empuja las concentraciones hacia valores bajos, centrales o altos.

In [2]:
# ============================================================
# PASO 11: sensibilidad HBM completa
# recalcular impactos sanitarios y económicos con p05 / p50 / p95
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetros económicos
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0
FX_COP_PER_USD_2015 = 2773.43

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
sens_root = salidas_sens / "04_sensibilidad" / "percentiles_hbm"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"
pop_targets_path = base_root / "population_targets_by_run.csv"
inc_targets_path = base_root / "incidence_targets_by_run.csv"
surfaces_path = sens_root / "superficies_anuales_percentiles_hbm.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

corridas = read_csv_safe(corridas_path)
pop_targets = read_csv_safe(pop_targets_path)
inc_targets = read_csv_safe(inc_targets_path)
surfaces = read_csv_safe(surfaces_path)

# ------------------------------------------------------------
# 4) Normalizar tipos
# ------------------------------------------------------------
for df, cols in [
    (pop_targets, ["cell_id", "population_target"]),
    (inc_targets, ["cell_id", "incidence_target"]),
    (surfaces, ["cell_id", "concentration_baseline_sens", "concentration_control", "delta_concentration_sens"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["beta", "beta_se", "valuation_value_fixed"]:
    if c in corridas.columns:
        corridas[c] = pd.to_numeric(corridas[c], errors="coerce")

# ------------------------------------------------------------
# 5) Reconstruir base de sensibilidad por corrida y percentil
# ------------------------------------------------------------
sens_runs = []

for _, run in corridas.iterrows():
    run_id = run["run_id"]
    pollutant = str(run["pollutant"]).strip()

    surf = surfaces[surfaces["pollutant"].astype(str).str.strip() == pollutant].copy()
    pop = pop_targets[pop_targets["run_id"].astype(str).str.strip() == run_id].copy()
    inc = inc_targets[inc_targets["run_id"].astype(str).str.strip() == run_id].copy()

    merged = (
        surf.merge(pop[["run_id", "cell_id", "population_target"]], on="cell_id", how="left")
            .merge(inc[["run_id", "cell_id", "incidence_target"]], on="cell_id", how="left")
    )

    merged["run_id"] = run_id
    merged["endpoint_group"] = run["endpoint_group"]
    merged["endpoint"] = run["endpoint"]
    merged["start_age"] = run["start_age"]
    merged["end_age"] = run["end_age"]
    merged["beta"] = run["beta"]
    merged["beta_se"] = run["beta_se"]
    merged["economic_method"] = run["economic_method"]
    merged["valuation_parameter_name"] = run.get("valuation_parameter_name", pd.NA)
    merged["valuation_value_fixed"] = run.get("valuation_value_fixed", pd.NA)

    sens_runs.append(merged)

sens_base = pd.concat(sens_runs, ignore_index=True)

# ------------------------------------------------------------
# 6) Cálculo sanitario
# ------------------------------------------------------------
sens_base["rr"] = np.exp(sens_base["beta"] * sens_base["delta_concentration_sens"])
sens_base["af_avoided"] = 1 - np.exp(-sens_base["beta"] * sens_base["delta_concentration_sens"])
sens_base["baseline_cases"] = sens_base["incidence_target"] * sens_base["population_target"]
sens_base["avoided_cases"] = sens_base["baseline_cases"] * sens_base["af_avoided"]

# ------------------------------------------------------------
# 7) Valoración económica
# ------------------------------------------------------------
sens_base["valuation_value_used_usd_2015"] = np.nan

mask_vsl = sens_base["economic_method"].astype(str).str.strip() == "VSL"
sens_base.loc[mask_vsl, "valuation_value_used_usd_2015"] = VSL_2015_USD

mask_unit = sens_base["economic_method"].astype(str).str.strip().isin([
    "UNIT_VALUE_ASTHMA_HA",
    "UNIT_VALUE_CLD_HA"
])
sens_base.loc[mask_unit, "valuation_value_used_usd_2015"] = pd.to_numeric(
    sens_base.loc[mask_unit, "valuation_value_fixed"], errors="coerce"
)

sens_base["economic_value_usd_2015"] = (
    sens_base["avoided_cases"] * sens_base["valuation_value_used_usd_2015"]
)

sens_base["economic_value_cop_2015"] = (
    sens_base["economic_value_usd_2015"] * FX_COP_PER_USD_2015
)

# ------------------------------------------------------------
# 8) Ordenar columnas
# ------------------------------------------------------------
sens_base = sens_base[[
    "run_id", "pollutant", "percentile_hbm", "endpoint_group", "endpoint",
    "cell_id",
    "concentration_baseline_sens", "concentration_control", "delta_concentration_sens",
    "population_target", "incidence_target",
    "beta", "beta_se",
    "rr", "af_avoided", "baseline_cases", "avoided_cases",
    "economic_method", "valuation_parameter_name", "valuation_value_used_usd_2015",
    "economic_value_usd_2015", "economic_value_cop_2015"
]].sort_values(["run_id", "percentile_hbm", "cell_id"]).reset_index(drop=True)

# ------------------------------------------------------------
# 9) Resumen por corrida y percentil
# ------------------------------------------------------------
summary = (
    sens_base.groupby(
        ["run_id", "pollutant", "percentile_hbm", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        delta_mean=("delta_concentration_sens", "mean"),
        delta_min=("delta_concentration_sens", "min"),
        delta_max=("delta_concentration_sens", "max"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
        valuation_value_used_usd_2015=("valuation_value_used_usd_2015", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 10) Comparación contra p50
# ------------------------------------------------------------
p50_ref = (
    summary[summary["percentile_hbm"] == "p50"][
        ["run_id", "avoided_cases_total", "economic_value_total_usd_2015", "economic_value_total_cop_2015"]
    ]
    .rename(columns={
        "avoided_cases_total": "avoided_cases_p50",
        "economic_value_total_usd_2015": "economic_usd_p50",
        "economic_value_total_cop_2015": "economic_cop_p50"
    })
)

summary_compare = summary.merge(p50_ref, on="run_id", how="left")

summary_compare["delta_vs_p50_cases"] = (
    summary_compare["avoided_cases_total"] - summary_compare["avoided_cases_p50"]
)

summary_compare["ratio_vs_p50_cases"] = np.where(
    summary_compare["avoided_cases_p50"].abs() > 0,
    summary_compare["avoided_cases_total"] / summary_compare["avoided_cases_p50"],
    np.nan
)

summary_compare["delta_vs_p50_usd"] = (
    summary_compare["economic_value_total_usd_2015"] - summary_compare["economic_usd_p50"]
)

summary_compare["ratio_vs_p50_usd"] = np.where(
    summary_compare["economic_usd_p50"].abs() > 0,
    summary_compare["economic_value_total_usd_2015"] / summary_compare["economic_usd_p50"],
    np.nan
)

# ------------------------------------------------------------
# 11) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("SENSIBILIDAD HBM: IMPACTOS SANITARIOS Y ECONÓMICOS")
print("=" * 80)
display(summary_compare)

print("\nVista previa por celda:")
display(sens_base.head(20))

# ------------------------------------------------------------
# 12) Exportar
# ------------------------------------------------------------
cells_csv = sens_root / "sensibilidad_hbm_por_celda.csv"
summary_csv = sens_root / "sensibilidad_hbm_resumen.csv"
compare_csv = sens_root / "sensibilidad_hbm_resumen_comparado_p50.csv"
log_json = logs_dir / "11_sensibilidad_hbm_completa.json"

sens_base.to_csv(cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "VSL_2015_USD": VSL_2015_USD,
    "FX_COP_PER_USD_2015": FX_COP_PER_USD_2015,
    "sens_base_shape": list(sens_base.shape),
    "summary_shape": list(summary.shape),
    "summary_compare_shape": list(summary_compare.shape),
    "outputs": {
        "cells_csv": str(cells_csv),
        "summary_csv": str(summary_csv),
        "compare_csv": str(compare_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", cells_csv)
print("-", summary_csv)
print("-", compare_csv)
print("-", log_json)

SENSIBILIDAD HBM: IMPACTOS SANITARIOS Y ECONÓMICOS


,run_id,pollutant,percentile_hbm,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,delta_mean,delta_min,...,valuation_value_used_usd_2015,economic_value_total_usd_2015,economic_value_total_cop_2015,avoided_cases_p50,economic_usd_p50,economic_cop_p50,delta_vs_p50_cases,ratio_vs_p50_cases,delta_vs_p50_usd,ratio_vs_p50_usd
0,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,-9.736438,-11.356495,...,11232.0,-9.738288e+05,-2.700846e+09,12.713526,1.427983e+05,3.960412e+08,-99.414812,-6.819610,-1.116627e+06,-6.819610
1,NO2_ASTHMA,NO2,p50,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,1.433793,0.952636,...,11232.0,1.427983e+05,3.960412e+08,12.713526,1.427983e+05,3.960412e+08,0.000000,1.000000,0.000000e+00,1.000000
2,NO2_ASTHMA,NO2,p95,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,62.386051,1.972684,...,11232.0,5.515408e+06,1.529660e+10,12.713526,1.427983e+05,3.960412e+08,478.330640,38.623759,5.372610e+06,38.623759
3,NO2_CLD,NO2,p05,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,-9.736438,-11.356495,...,15375.0,-9.827251e+05,-2.725519e+09,9.454138,1.453574e+05,4.031385e+08,-73.371218,-6.760752,-1.128082e+06,-6.760752
4,NO2_CLD,NO2,p50,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,1.433793,0.952636,...,15375.0,1.453574e+05,4.031385e+08,9.454138,1.453574e+05,4.031385e+08,0.000000,1.000000,0.000000e+00,1.000000
5,NO2_CLD,NO2,p95,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,62.386051,1.972684,...,15375.0,5.877445e+06,1.630068e+10,9.454138,1.453574e+05,4.031385e+08,372.818692,40.434446,5.732087e+06,40.434446
6,O3_MAIN,O3,p05,Mortality,Respiratory mortality,VSL,254,254,-8.103976,-9.962069,...,8705114.0,-1.281242e+08,-3.553434e+11,2.204553,1.919088e+07,5.322457e+10,-16.922818,-6.676304,-1.473151e+08,-6.676304
7,O3_MAIN,O3,p50,Mortality,Respiratory mortality,VSL,254,254,1.195564,0.638176,...,8705114.0,1.919088e+07,5.322457e+10,2.204553,1.919088e+07,5.322457e+10,0.000000,1.000000,0.000000e+00,1.000000
8,O3_MAIN,O3,p95,Mortality,Respiratory mortality,VSL,254,254,51.932743,1.241827,...,8705114.0,7.995430e+08,2.217476e+12,2.204553,1.919088e+07,5.322457e+10,89.642948,41.662646,7.803521e+08,41.662646
9,PM25_MAIN,PM25,p05,Mortality,All-cause mortality,VSL,254,254,-9.974680,-11.828708,...,8705114.0,-7.864739e+07,-2.181230e+11,1.299799,1.131490e+07,3.138108e+10,-10.334418,-6.950781,-8.996228e+07,-6.950781



Vista previa por celda:


,run_id,pollutant,percentile_hbm,endpoint_group,endpoint,cell_id,concentration_baseline_sens,concentration_control,delta_concentration_sens,population_target,...,beta_se,rr,af_avoided,baseline_cases,avoided_cases,economic_method,valuation_parameter_name,valuation_value_used_usd_2015,economic_value_usd_2015,economic_value_cop_2015
0,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,0,2.642624,12.808184,-10.165560,66007,...,NaN,0.966774,-0.034368,11.005657,-0.378239,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-4248.381838,-1.178259e+07
1,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,1,2.562797,13.092439,-10.529642,35848,...,NaN,0.965605,-0.035620,5.977106,-0.212906,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-2391.360557,-6.632271e+06
2,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,2,2.553587,12.599205,-10.045618,682,...,NaN,0.967160,-0.033955,0.113713,-0.003861,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-43.368681,-1.202800e+05
3,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,41,2.737744,12.993218,-10.255474,53333,...,NaN,0.966485,-0.034677,8.892462,-0.308363,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-3463.532389,-9.605865e+06
4,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,42,2.687013,12.270779,-9.583766,88375,...,NaN,0.968646,-0.032369,14.735179,-0.476967,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-5357.295893,-1.485809e+07
5,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,43,2.640156,13.018890,-10.378734,64336,...,NaN,0.966089,-0.035101,10.727044,-0.376529,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-4229.172618,-1.172931e+07
6,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,44,2.581686,12.808265,-10.226579,8855,...,NaN,0.966578,-0.034578,1.476436,-0.051051,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-573.410269,-1.590313e+06
7,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,82,2.681063,12.642091,-9.961028,6180,...,NaN,0.967432,-0.033665,1.030420,-0.034689,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-389.624652,-1.080597e+06
8,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,83,2.718463,12.824621,-10.106158,38621,...,NaN,0.966965,-0.034163,6.439461,-0.219994,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-2470.977151,-6.853082e+06
9,NO2_ASTHMA,NO2,p05,Hospital Admissions,Asthma,84,2.590983,12.863861,-10.272878,86400,...,NaN,0.966429,-0.034737,14.405878,-0.500413,UNIT_VALUE_ASTHMA_HA,UNIT_VALUE_ASTHMA_HA_2015_USD,11232.0,-5620.642866,-1.558846e+07



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\percentiles_hbm\sensibilidad_hbm_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\percentiles_hbm\sensibilidad_hbm_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\percentiles_hbm\sensibilidad_hbm_resumen_comparado_p50.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\11_sensibilidad_hbm_completa.json


## Paso 12. Tabla resumen clara de sensibilidad HBM

En esta etapa se construye una tabla resumen de sensibilidad con nombres más simples y de lectura más directa. La finalidad es sintetizar, para cada corrida, cómo cambian los resultados sanitarios y económicos cuando se utilizan escenarios bajos, centrales y altos de la superficie estimada por el modelo jerárquico bayesiano.

La tabla se organiza por escenario bajo (p05), escenario central (p50) y escenario alto (p95), e incluye el cambio medio de concentración, los casos evitados, el valor económico total en pesos colombianos de 2015, así como la diferencia y la razón respecto al escenario central. Esto permite interpretar con mayor claridad el efecto de la incertidumbre del HBM sobre la carga sanitaria y económica del estudio.

In [3]:
# ============================================================
# PASO 12: tabla resumen clara de sensibilidad HBM
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
sens_root = salidas_sens / "04_sensibilidad" / "percentiles_hbm"
out_tables = salidas_sens / "07_tablas"
out_final = salidas_sens / "08_resumenes_finales"
logs_dir = salidas_sens / "00_logs_validacion"

summary_compare_path = sens_root / "sensibilidad_hbm_resumen_comparado_p50.csv"

for folder in [sens_root, out_tables, out_final, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not summary_compare_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{summary_compare_path}")

# ------------------------------------------------------------
# 2) Cargar resumen comparado
# ------------------------------------------------------------
df = pd.read_csv(summary_compare_path, encoding="utf-8-sig")
df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

# ------------------------------------------------------------
# 3) Etiquetas simples
# ------------------------------------------------------------
map_percentil = {
    "p05": "Escenario bajo (p05)",
    "p50": "Escenario central (p50)",
    "p95": "Escenario alto (p95)"
}

map_corrida = {
    "PM25_MAIN": "PM2.5 - Mortalidad por todas las causas",
    "O3_MAIN": "O3 - Mortalidad respiratoria",
    "NO2_ASTHMA": "NO2 - Hospitalizaciones por asma",
    "NO2_CLD": "NO2 - Hospitalizaciones por enfermedad pulmonar crónica"
}

df["Escenario HBM"] = df["percentile_hbm"].map(map_percentil).fillna(df["percentile_hbm"])
df["Corrida"] = df["run_id"].map(map_corrida).fillna(df["run_id"])

# ------------------------------------------------------------
# 4) Seleccionar y renombrar columnas
# ------------------------------------------------------------
tabla_resumen = df[[
    "Corrida",
    "pollutant",
    "Escenario HBM",
    "delta_mean",
    "avoided_cases_total",
    "economic_value_total_cop_2015",
    "delta_vs_p50_cases",
    "ratio_vs_p50_cases",
    "delta_vs_p50_usd",
    "ratio_vs_p50_usd"
]].copy()

tabla_resumen = tabla_resumen.rename(columns={
    "pollutant": "Contaminante",
    "delta_mean": "Cambio medio en concentración",
    "avoided_cases_total": "Casos evitados",
    "economic_value_total_cop_2015": "Valor económico total (COP 2015)",
    "delta_vs_p50_cases": "Diferencia de casos vs. central",
    "ratio_vs_p50_cases": "Razón de casos vs. central",
    "delta_vs_p50_usd": "Diferencia económica vs. central (USD 2015)",
    "ratio_vs_p50_usd": "Razón económica vs. central"
})

# ------------------------------------------------------------
# 5) Redondear para lectura
# ------------------------------------------------------------
cols_round_3 = [
    "Cambio medio en concentración",
    "Casos evitados",
    "Diferencia de casos vs. central",
    "Razón de casos vs. central",
    "Razón económica vs. central"
]

for col in cols_round_3:
    if col in tabla_resumen.columns:
        tabla_resumen[col] = pd.to_numeric(tabla_resumen[col], errors="coerce").round(3)

for col in ["Valor económico total (COP 2015)", "Diferencia económica vs. central (USD 2015)"]:
    if col in tabla_resumen.columns:
        tabla_resumen[col] = pd.to_numeric(tabla_resumen[col], errors="coerce").round(0)

# ------------------------------------------------------------
# 6) Ordenar
# ------------------------------------------------------------
orden_corridas = [
    "PM2.5 - Mortalidad por todas las causas",
    "O3 - Mortalidad respiratoria",
    "NO2 - Hospitalizaciones por asma",
    "NO2 - Hospitalizaciones por enfermedad pulmonar crónica"
]

orden_escenarios = [
    "Escenario bajo (p05)",
    "Escenario central (p50)",
    "Escenario alto (p95)"
]

tabla_resumen["orden_corrida"] = tabla_resumen["Corrida"].apply(
    lambda x: orden_corridas.index(x) if x in orden_corridas else 999
)
tabla_resumen["orden_escenario"] = tabla_resumen["Escenario HBM"].apply(
    lambda x: orden_escenarios.index(x) if x in orden_escenarios else 999
)

tabla_resumen = (
    tabla_resumen
    .sort_values(["orden_corrida", "orden_escenario"])
    .drop(columns=["orden_corrida", "orden_escenario"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7) Mostrar
# ------------------------------------------------------------
print("=" * 90)
print("TABLA RESUMEN CLARA DE SENSIBILIDAD HBM")
print("=" * 90)
display(tabla_resumen)

# ------------------------------------------------------------
# 8) Exportar
# ------------------------------------------------------------
tabla_csv = out_tables / "tabla_resumen_sensibilidad_hbm_clara.csv"
tabla_xlsx = out_final / "tabla_resumen_sensibilidad_hbm_clara.xlsx"
log_json = logs_dir / "12_tabla_resumen_clara_sensibilidad_hbm.json"

tabla_resumen.to_csv(tabla_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(tabla_xlsx, engine="openpyxl") as writer:
    tabla_resumen.to_excel(writer, index=False, sheet_name="Sensibilidad_HBM")

payload = {
    "timestamp": datetime.now().isoformat(),
    "tabla_shape": list(tabla_resumen.shape),
    "outputs": {
        "tabla_csv": str(tabla_csv),
        "tabla_xlsx": str(tabla_xlsx)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", tabla_csv)
print("-", tabla_xlsx)
print("-", log_json)

TABLA RESUMEN CLARA DE SENSIBILIDAD HBM


,Corrida,Contaminante,Escenario HBM,Cambio medio en concentración,Casos evitados,Valor económico total (COP 2015),Diferencia de casos vs. central,Razón de casos vs. central,Diferencia económica vs. central (USD 2015),Razón económica vs. central
0,PM2.5 - Mortalidad por todas las causas,PM25,Escenario bajo (p05),-9.975,-9.035,-2.181230e+11,-10.334,-6.951,-89962285.0,-6.951
1,PM2.5 - Mortalidad por todas las causas,PM25,Escenario central (p50),1.478,1.300,3.138108e+10,0.000,1.000,0.0,1.000
2,PM2.5 - Mortalidad por todas las causas,PM25,Escenario alto (p95),63.552,42.675,1.030300e+12,41.375,32.832,360174664.0,32.832
3,O3 - Mortalidad respiratoria,O3,Escenario bajo (p05),-8.104,-14.718,-3.553434e+11,-16.923,-6.676,-147315058.0,-6.676
4,O3 - Mortalidad respiratoria,O3,Escenario central (p50),1.196,2.205,5.322457e+10,0.000,1.000,0.0,1.000
5,O3 - Mortalidad respiratoria,O3,Escenario alto (p95),51.933,91.848,2.217476e+12,89.643,41.663,780352083.0,41.663
6,NO2 - Hospitalizaciones por asma,NO2,Escenario bajo (p05),-9.736,-86.701,-2.700846e+09,-99.415,-6.820,-1116627.0,-6.820
7,NO2 - Hospitalizaciones por asma,NO2,Escenario central (p50),1.434,12.714,3.960412e+08,0.000,1.000,0.0,1.000
8,NO2 - Hospitalizaciones por asma,NO2,Escenario alto (p95),62.386,491.044,1.529660e+10,478.331,38.624,5372610.0,38.624
9,NO2 - Hospitalizaciones por enfermedad pulmona...,NO2,Escenario bajo (p05),-9.736,-63.917,-2.725519e+09,-73.371,-6.761,-1128082.0,-6.761



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\tabla_resumen_sensibilidad_hbm_clara.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\08_resumenes_finales\tabla_resumen_sensibilidad_hbm_clara.xlsx
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\12_tabla_resumen_clara_sensibilidad_hbm.json


## Paso 13. Definición de escenarios de sensibilidad por beta

En esta etapa se definen los escenarios de sensibilidad asociados a la función concentración–respuesta, con el fin de evaluar cuánto cambian los resultados sanitarios y económicos cuando se modifica el coeficiente epidemiológico beta. La finalidad no es alterar la estructura de exposición, población o incidencia, sino aislar el efecto que tiene la elección del parámetro epidemiológico sobre la magnitud del impacto estimado.

Para PM2.5 se utilizarán dos escenarios: un beta bajo correspondiente al valor alternativo previamente identificado en el pipeline y un beta central correspondiente al valor base adoptado en la corrida principal. Para O3 se construirán tres escenarios a partir del beta central y su error estándar, generando una aproximación baja y alta basada en un intervalo normal. En el caso de NO2, mientras no se definan alternativas mejor sustentadas, se mantendrá únicamente el beta central ya adoptado.

El resultado será una tabla maestra de escenarios beta por corrida, lista para ser utilizada en el recálculo de impactos sanitarios y económicos bajo sensibilidad epidemiológica.

In [4]:
# ============================================================
# PASO 13: definir escenarios de sensibilidad por beta
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
sens_root = salidas_sens / "04_sensibilidad" / "beta"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not corridas_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{corridas_path}")

# ------------------------------------------------------------
# 2) Cargar corridas base
# ------------------------------------------------------------
corridas = pd.read_csv(corridas_path, encoding="utf-8-sig")
corridas.columns = [str(c).replace("\ufeff", "").strip() for c in corridas.columns]

for c in ["beta", "beta_se", "beta_sensitivity_low"]:
    if c in corridas.columns:
        corridas[c] = pd.to_numeric(corridas[c], errors="coerce")

# ------------------------------------------------------------
# 3) Construir escenarios beta
# ------------------------------------------------------------
rows = []

for _, row in corridas.iterrows():
    run_id = row["run_id"]
    pollutant = row["pollutant"]
    endpoint_group = row["endpoint_group"]
    endpoint = row["endpoint"]
    beta_base = pd.to_numeric(row["beta"], errors="coerce")
    beta_se = pd.to_numeric(row.get("beta_se", np.nan), errors="coerce")

    # PM2.5: beta bajo + beta central
    if run_id == "PM25_MAIN":
        beta_low = pd.to_numeric(row.get("beta_sensitivity_low", np.nan), errors="coerce")

        if pd.notna(beta_low):
            rows.append({
                "run_id": run_id,
                "pollutant": pollutant,
                "endpoint_group": endpoint_group,
                "endpoint": endpoint,
                "beta_scenario": "Escenario beta bajo",
                "beta_value": float(beta_low),
                "beta_reference": "beta_alternativo_pipeline_previo"
            })

        rows.append({
            "run_id": run_id,
            "pollutant": pollutant,
            "endpoint_group": endpoint_group,
            "endpoint": endpoint,
            "beta_scenario": "Escenario beta central",
            "beta_value": float(beta_base),
            "beta_reference": "beta_base_corrida_principal"
        })

    # O3: bajo / central / alto con beta ± 1.96 * se
    elif run_id == "O3_MAIN":
        if pd.isna(beta_base):
            raise ValueError("O3_MAIN no tiene beta base.")
        if pd.isna(beta_se):
            raise ValueError("O3_MAIN no tiene beta_se para sensibilidad.")

        beta_low = max(beta_base - 1.96 * beta_se, 0.0)
        beta_high = beta_base + 1.96 * beta_se

        rows.extend([
            {
                "run_id": run_id,
                "pollutant": pollutant,
                "endpoint_group": endpoint_group,
                "endpoint": endpoint,
                "beta_scenario": "Escenario beta bajo",
                "beta_value": float(beta_low),
                "beta_reference": "beta_minus_1.96_se"
            },
            {
                "run_id": run_id,
                "pollutant": pollutant,
                "endpoint_group": endpoint_group,
                "endpoint": endpoint,
                "beta_scenario": "Escenario beta central",
                "beta_value": float(beta_base),
                "beta_reference": "beta_base_corrida_principal"
            },
            {
                "run_id": run_id,
                "pollutant": pollutant,
                "endpoint_group": endpoint_group,
                "endpoint": endpoint,
                "beta_scenario": "Escenario beta alto",
                "beta_value": float(beta_high),
                "beta_reference": "beta_plus_1.96_se"
            }
        ])

    # NO2: por ahora solo central
    else:
        rows.append({
            "run_id": run_id,
            "pollutant": pollutant,
            "endpoint_group": endpoint_group,
            "endpoint": endpoint,
            "beta_scenario": "Escenario beta central",
            "beta_value": float(beta_base),
            "beta_reference": "beta_base_corrida_principal"
        })

beta_scenarios = pd.DataFrame(rows)

# ------------------------------------------------------------
# 4) Resumen corto
# ------------------------------------------------------------
summary = (
    beta_scenarios.groupby(["run_id", "pollutant", "endpoint", "beta_scenario"], as_index=False)
    .agg(
        beta_value=("beta_value", "first"),
        beta_reference=("beta_reference", "first")
    )
)

print("=" * 80)
print("ESCENARIOS DE SENSIBILIDAD POR BETA")
print("=" * 80)
display(summary)

# ------------------------------------------------------------
# 5) Exportar
# ------------------------------------------------------------
beta_csv = sens_root / "escenarios_beta.csv"
beta_summary_csv = sens_root / "escenarios_beta_resumen.csv"
log_json = logs_dir / "13_escenarios_beta.json"

beta_scenarios.to_csv(beta_csv, index=False, encoding="utf-8-sig")
summary.to_csv(beta_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "beta_scenarios_shape": list(beta_scenarios.shape),
    "summary_shape": list(summary.shape),
    "outputs": {
        "beta_csv": str(beta_csv),
        "beta_summary_csv": str(beta_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", beta_csv)
print("-", beta_summary_csv)
print("-", log_json)

ESCENARIOS DE SENSIBILIDAD POR BETA


,run_id,pollutant,endpoint,beta_scenario,beta_value,beta_reference
0,NO2_ASTHMA,NO2,Asthma,Escenario beta central,0.003324,beta_base_corrida_principal
1,NO2_CLD,NO2,Chronic Lung Disease,Escenario beta central,0.001850,beta_base_corrida_principal
2,O3_MAIN,O3,Respiratory mortality,Escenario beta alto,0.001463,beta_plus_1.96_se
3,O3_MAIN,O3,Respiratory mortality,Escenario beta bajo,0.000271,beta_minus_1.96_se
4,O3_MAIN,O3,Respiratory mortality,Escenario beta central,0.000867,beta_base_corrida_principal
5,PM25_MAIN,PM25,All-cause mortality,Escenario beta bajo,0.001094,beta_alternativo_pipeline_previo
6,PM25_MAIN,PM25,All-cause mortality,Escenario beta central,0.008066,beta_base_corrida_principal



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\beta\escenarios_beta.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\beta\escenarios_beta_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\13_escenarios_beta.json


## Paso 14. Recalcular impactos bajo sensibilidad por beta

En esta etapa se recalculan los impactos sanitarios y económicos variando únicamente el coeficiente epidemiológico beta. La finalidad es identificar cuánto cambian los resultados cuando la función concentración–respuesta se modifica, manteniendo fijos los demás componentes del análisis, es decir, la exposición, la población, la incidencia y el método de valoración económica.

La lógica consiste en combinar la base analítica final de la corrida principal con los escenarios beta definidos previamente. De este modo, cada corrida puede evaluarse bajo un beta central y, cuando exista sustento metodológico, también bajo escenarios bajos o altos. Posteriormente se recalculan el riesgo relativo, la fracción evitada, los casos evitados y el valor económico correspondiente.

El resultado será una tabla detallada por celda y una tabla resumen por corrida y escenario beta, lo cual permitirá medir la sensibilidad de la carga sanitaria y económica frente a cambios en el parámetro epidemiológico.

In [5]:
# ============================================================
# PASO 14: recalcular impactos con sensibilidad por beta
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetros económicos
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0
FX_COP_PER_USD_2015 = 2773.43

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
sens_root = salidas_sens / "04_sensibilidad" / "beta"
logs_dir = salidas_sens / "00_logs_validacion"

base_path = base_root / "base_analitica_final_por_corrida.csv"
beta_path = sens_root / "escenarios_beta.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not base_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{base_path}")
if not beta_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{beta_path}")

# ------------------------------------------------------------
# 3) Cargar archivos
# ------------------------------------------------------------
base = pd.read_csv(base_path, encoding="utf-8-sig")
beta_df = pd.read_csv(beta_path, encoding="utf-8-sig")

base.columns = [str(c).replace("\ufeff", "").strip() for c in base.columns]
beta_df.columns = [str(c).replace("\ufeff", "").strip() for c in beta_df.columns]

# ------------------------------------------------------------
# 4) Normalizar tipos
# ------------------------------------------------------------
for c in [
    "delta_concentration", "population_target", "incidence_target",
    "valuation_value_fixed"
]:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

if "beta_value" in beta_df.columns:
    beta_df["beta_value"] = pd.to_numeric(beta_df["beta_value"], errors="coerce")

# ------------------------------------------------------------
# 5) Cruzar base analítica con escenarios beta
# ------------------------------------------------------------
sens_beta = base.merge(
    beta_df[["run_id", "beta_scenario", "beta_value", "beta_reference"]],
    on="run_id",
    how="inner"
)

# ------------------------------------------------------------
# 6) Cálculo sanitario
# ------------------------------------------------------------
sens_beta["rr"] = np.exp(sens_beta["beta_value"] * sens_beta["delta_concentration"])
sens_beta["af_avoided"] = 1 - np.exp(-sens_beta["beta_value"] * sens_beta["delta_concentration"])
sens_beta["baseline_cases"] = sens_beta["incidence_target"] * sens_beta["population_target"]
sens_beta["avoided_cases"] = sens_beta["baseline_cases"] * sens_beta["af_avoided"]

# ------------------------------------------------------------
# 7) Valoración económica
# ------------------------------------------------------------
sens_beta["valuation_value_used_usd_2015"] = np.nan

mask_vsl = sens_beta["economic_method"].astype(str).str.strip() == "VSL"
sens_beta.loc[mask_vsl, "valuation_value_used_usd_2015"] = VSL_2015_USD

mask_unit = sens_beta["economic_method"].astype(str).str.strip().isin([
    "UNIT_VALUE_ASTHMA_HA",
    "UNIT_VALUE_CLD_HA"
])
sens_beta.loc[mask_unit, "valuation_value_used_usd_2015"] = pd.to_numeric(
    sens_beta.loc[mask_unit, "valuation_value_fixed"], errors="coerce"
)

sens_beta["economic_value_usd_2015"] = (
    sens_beta["avoided_cases"] * sens_beta["valuation_value_used_usd_2015"]
)
sens_beta["economic_value_cop_2015"] = (
    sens_beta["economic_value_usd_2015"] * FX_COP_PER_USD_2015
)

# ------------------------------------------------------------
# 8) Ordenar columnas
# ------------------------------------------------------------
sens_beta = sens_beta[[
    "run_id", "pollutant", "beta_scenario", "beta_reference",
    "endpoint_group", "endpoint", "cell_id",
    "delta_concentration", "population_target", "incidence_target",
    "beta_value", "rr", "af_avoided", "baseline_cases", "avoided_cases",
    "economic_method", "valuation_value_used_usd_2015",
    "economic_value_usd_2015", "economic_value_cop_2015"
]].sort_values(["run_id", "beta_scenario", "cell_id"]).reset_index(drop=True)

# ------------------------------------------------------------
# 9) Resumen por corrida y escenario beta
# ------------------------------------------------------------
summary = (
    sens_beta.groupby(
        ["run_id", "pollutant", "beta_scenario", "endpoint_group", "endpoint", "economic_method"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        delta_mean=("delta_concentration", "mean"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
        beta_value=("beta_value", "first"),
        valuation_value_used_usd_2015=("valuation_value_used_usd_2015", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 10) Comparación frente al escenario beta central
# ------------------------------------------------------------
central_ref = (
    summary[summary["beta_scenario"] == "Escenario beta central"][
        ["run_id", "avoided_cases_total", "economic_value_total_usd_2015", "economic_value_total_cop_2015"]
    ]
    .rename(columns={
        "avoided_cases_total": "avoided_cases_central",
        "economic_value_total_usd_2015": "economic_usd_central",
        "economic_value_total_cop_2015": "economic_cop_central"
    })
)

summary_compare = summary.merge(central_ref, on="run_id", how="left")

summary_compare["diferencia_casos_vs_central"] = (
    summary_compare["avoided_cases_total"] - summary_compare["avoided_cases_central"]
)
summary_compare["razon_casos_vs_central"] = np.where(
    summary_compare["avoided_cases_central"].abs() > 0,
    summary_compare["avoided_cases_total"] / summary_compare["avoided_cases_central"],
    np.nan
)

summary_compare["diferencia_usd_vs_central"] = (
    summary_compare["economic_value_total_usd_2015"] - summary_compare["economic_usd_central"]
)
summary_compare["razon_usd_vs_central"] = np.where(
    summary_compare["economic_usd_central"].abs() > 0,
    summary_compare["economic_value_total_usd_2015"] / summary_compare["economic_usd_central"],
    np.nan
)

# ------------------------------------------------------------
# 11) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("SENSIBILIDAD POR BETA: IMPACTOS SANITARIOS Y ECONÓMICOS")
print("=" * 80)
display(summary_compare)

print("\nVista previa por celda:")
display(sens_beta.head(20))

# ------------------------------------------------------------
# 12) Exportar
# ------------------------------------------------------------
cells_csv = sens_root / "sensibilidad_beta_por_celda.csv"
summary_csv = sens_root / "sensibilidad_beta_resumen.csv"
compare_csv = sens_root / "sensibilidad_beta_resumen_comparado_central.csv"
log_json = logs_dir / "14_sensibilidad_beta_completa.json"

sens_beta.to_csv(cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "VSL_2015_USD": VSL_2015_USD,
    "FX_COP_PER_USD_2015": FX_COP_PER_USD_2015,
    "sens_beta_shape": list(sens_beta.shape),
    "summary_shape": list(summary.shape),
    "summary_compare_shape": list(summary_compare.shape),
    "outputs": {
        "cells_csv": str(cells_csv),
        "summary_csv": str(summary_csv),
        "compare_csv": str(compare_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", cells_csv)
print("-", summary_csv)
print("-", compare_csv)
print("-", log_json)

SENSIBILIDAD POR BETA: IMPACTOS SANITARIOS Y ECONÓMICOS


,run_id,pollutant,beta_scenario,endpoint_group,endpoint,economic_method,n_rows,n_unique_cells,delta_mean,baseline_cases_total,...,valuation_value_used_usd_2015,economic_value_total_usd_2015,economic_value_total_cop_2015,avoided_cases_central,economic_usd_central,economic_cop_central,diferencia_casos_vs_central,razon_casos_vs_central,diferencia_usd_vs_central,razon_usd_vs_central
0,NO2_ASTHMA,NO2,Escenario beta central,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,254,254,1.433793,2676.999472,...,11232.0,1.427983e+05,3.960412e+08,12.713526,1.427983e+05,3.960412e+08,0.000000,1.000000,0.000000e+00,1.000000
1,NO2_CLD,NO2,Escenario beta central,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,254,254,1.433793,3573.000074,...,15375.0,1.453574e+05,4.031385e+08,9.454138,1.453574e+05,4.031385e+08,0.000000,1.000000,0.000000e+00,1.000000
2,O3_MAIN,O3,Escenario beta alto,Mortality,Respiratory mortality,VSL,254,254,1.195564,2120.000000,...,8705114.0,3.236806e+07,8.977055e+10,2.204553,1.919088e+07,5.322457e+10,1.513728,1.686637,1.317718e+07,1.686637
3,O3_MAIN,O3,Escenario beta bajo,Mortality,Respiratory mortality,VSL,254,254,1.195564,2120.000000,...,8705114.0,6.004232e+06,1.665232e+10,2.204553,1.919088e+07,5.322457e+10,-1.514817,0.312869,-1.318665e+07,0.312869
4,O3_MAIN,O3,Escenario beta central,Mortality,Respiratory mortality,VSL,254,254,1.195564,2120.000000,...,8705114.0,1.919088e+07,5.322457e+10,2.204553,1.919088e+07,5.322457e+10,0.000000,1.000000,0.000000e+00,1.000000
5,PM25_MAIN,PM25,Escenario beta bajo,Mortality,All-cause mortality,VSL,254,254,1.478043,109.744592,...,8705114.0,1.542609e+06,4.278319e+09,1.299799,1.131490e+07,3.138108e+10,-1.122592,0.136334,-9.772290e+06,0.136334
6,PM25_MAIN,PM25,Escenario beta central,Mortality,All-cause mortality,VSL,254,254,1.478043,109.744592,...,8705114.0,1.131490e+07,3.138108e+10,1.299799,1.131490e+07,3.138108e+10,0.000000,1.000000,0.000000e+00,1.000000



Vista previa por celda:


,run_id,pollutant,beta_scenario,beta_reference,endpoint_group,endpoint,cell_id,delta_concentration,population_target,incidence_target,beta_value,rr,af_avoided,baseline_cases,avoided_cases,economic_method,valuation_value_used_usd_2015,economic_value_usd_2015,economic_value_cop_2015
0,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,0,1.423132,66007,0.000167,0.003324,1.004742,0.004719,11.005657,0.051939,UNIT_VALUE_ASTHMA_HA,11232.0,583.381073,1.617967e+06
1,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,1,1.454715,35848,0.000167,0.003324,1.004847,0.004824,5.977106,0.028832,UNIT_VALUE_ASTHMA_HA,11232.0,323.845226,8.981621e+05
2,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,2,1.399912,682,0.000167,0.003324,1.004664,0.004642,0.113713,0.000528,UNIT_VALUE_ASTHMA_HA,11232.0,5.929513,1.644509e+04
3,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,41,1.443691,53333,0.000167,0.003324,1.004810,0.004787,8.892462,0.042571,UNIT_VALUE_ASTHMA_HA,11232.0,478.159413,1.326142e+06
4,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,42,1.363420,88375,0.000167,0.003324,1.004542,0.004522,14.735179,0.066629,UNIT_VALUE_ASTHMA_HA,11232.0,748.375246,2.075566e+06
5,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,43,1.446543,64336,0.000167,0.003324,1.004820,0.004797,10.727044,0.051455,UNIT_VALUE_ASTHMA_HA,11232.0,577.944194,1.602888e+06
6,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,44,1.423141,8855,0.000167,0.003324,1.004742,0.004719,1.476436,0.006968,UNIT_VALUE_ASTHMA_HA,11232.0,78.262489,2.170555e+05
7,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,82,1.404677,6180,0.000167,0.003324,1.004680,0.004658,1.030420,0.004800,UNIT_VALUE_ASTHMA_HA,11232.0,53.913247,1.495246e+05
8,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,83,1.424958,38621,0.000167,0.003324,1.004748,0.004725,6.439461,0.030429,UNIT_VALUE_ASTHMA_HA,11232.0,341.775979,9.478918e+05
9,NO2_ASTHMA,NO2,Escenario beta central,beta_base_corrida_principal,Hospital Admissions,Asthma,84,1.429318,86400,0.000167,0.003324,1.004762,0.004740,14.405878,0.068281,UNIT_VALUE_ASTHMA_HA,11232.0,766.929427,2.127025e+06



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\beta\sensibilidad_beta_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\beta\sensibilidad_beta_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\beta\sensibilidad_beta_resumen_comparado_central.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\14_sensibilidad_beta_completa.json


## Paso 15. Definición de escenarios de sensibilidad económica

En esta etapa se definen los escenarios de sensibilidad económica para evaluar cuánto cambian los resultados monetarios cuando se modifican los parámetros de valoración. La finalidad es aislar el efecto que tiene la elección del valor económico unitario sobre la estimación final de la carga económica, manteniendo constantes los resultados sanitarios ya calculados.

Para las corridas de mortalidad asociadas a PM2.5 y O3 se construirán tres escenarios del Valor Estadístico de la Vida (VSL): bajo, central y alto. Para las corridas de hospital admissions asociadas a NO2 se construirán igualmente tres escenarios para los valores unitarios por caso. En esta implementación inicial se utilizará una variación paramétrica de más o menos 20 por ciento respecto al valor central.

El resultado será una tabla maestra de escenarios económicos por corrida, la cual servirá posteriormente para recalcular la valoración económica bajo supuestos bajos, centrales y altos de monetización.

In [6]:
# ============================================================
# PASO 15: definir escenarios de sensibilidad económica
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetro de amplitud de sensibilidad
# ------------------------------------------------------------
SENS_FACTOR_LOW = 0.80
SENS_FACTOR_CENTRAL = 1.00
SENS_FACTOR_HIGH = 1.20

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
base_root = salidas_sens / "02_base_analitica"
sens_root = salidas_sens / "04_sensibilidad" / "valoracion"
logs_dir = salidas_sens / "00_logs_validacion"

corridas_path = base_root / "corridas_base_definitivas.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not corridas_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{corridas_path}")

# ------------------------------------------------------------
# 3) Cargar corridas base
# ------------------------------------------------------------
corridas = pd.read_csv(corridas_path, encoding="utf-8-sig")
corridas.columns = [str(c).replace("\ufeff", "").strip() for c in corridas.columns]

for c in ["valuation_value_fixed"]:
    if c in corridas.columns:
        corridas[c] = pd.to_numeric(corridas[c], errors="coerce")

# ------------------------------------------------------------
# 4) Valor central por corrida
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0

rows = []

for _, row in corridas.iterrows():
    run_id = row["run_id"]
    pollutant = row["pollutant"]
    endpoint_group = row["endpoint_group"]
    endpoint = row["endpoint"]
    economic_method = str(row["economic_method"]).strip()

    if economic_method == "VSL":
        central_value = VSL_2015_USD
        parameter_name = "VSL_2015_USD"
    else:
        central_value = pd.to_numeric(row["valuation_value_fixed"], errors="coerce")
        parameter_name = str(row.get("valuation_parameter_name", "UNIT_VALUE")).strip()

    if pd.isna(central_value):
        raise ValueError(f"No se pudo determinar el valor central para {run_id}")

    scenarios = [
        ("Escenario económico bajo", central_value * SENS_FACTOR_LOW, f"{parameter_name}_x_0.80"),
        ("Escenario económico central", central_value * SENS_FACTOR_CENTRAL, f"{parameter_name}_x_1.00"),
        ("Escenario económico alto", central_value * SENS_FACTOR_HIGH, f"{parameter_name}_x_1.20"),
    ]

    for scenario_name, scenario_value, scenario_ref in scenarios:
        rows.append({
            "run_id": run_id,
            "pollutant": pollutant,
            "endpoint_group": endpoint_group,
            "endpoint": endpoint,
            "economic_method": economic_method,
            "economic_scenario": scenario_name,
            "valuation_value_usd_2015": float(scenario_value),
            "valuation_reference": scenario_ref
        })

econ_scenarios = pd.DataFrame(rows)

summary = (
    econ_scenarios.groupby(
        ["run_id", "pollutant", "endpoint", "economic_method", "economic_scenario"],
        as_index=False
    )
    .agg(
        valuation_value_usd_2015=("valuation_value_usd_2015", "first"),
        valuation_reference=("valuation_reference", "first")
    )
)

print("=" * 80)
print("ESCENARIOS DE SENSIBILIDAD ECONÓMICA")
print("=" * 80)
display(summary)

# ------------------------------------------------------------
# 5) Exportar
# ------------------------------------------------------------
econ_csv = sens_root / "escenarios_valoracion.csv"
econ_summary_csv = sens_root / "escenarios_valoracion_resumen.csv"
log_json = logs_dir / "15_escenarios_valoracion.json"

econ_scenarios.to_csv(econ_csv, index=False, encoding="utf-8-sig")
summary.to_csv(econ_summary_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "sens_factor_low": SENS_FACTOR_LOW,
    "sens_factor_central": SENS_FACTOR_CENTRAL,
    "sens_factor_high": SENS_FACTOR_HIGH,
    "econ_scenarios_shape": list(econ_scenarios.shape),
    "summary_shape": list(summary.shape),
    "outputs": {
        "econ_csv": str(econ_csv),
        "econ_summary_csv": str(econ_summary_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", econ_csv)
print("-", econ_summary_csv)
print("-", log_json)

ESCENARIOS DE SENSIBILIDAD ECONÓMICA


,run_id,pollutant,endpoint,economic_method,economic_scenario,valuation_value_usd_2015,valuation_reference
0,NO2_ASTHMA,NO2,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,13478.4,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20
1,NO2_ASTHMA,NO2,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico bajo,8985.6,UNIT_VALUE_ASTHMA_HA_2015_USD_x_0.80
2,NO2_ASTHMA,NO2,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico central,11232.0,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.00
3,NO2_CLD,NO2,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico alto,18450.0,UNIT_VALUE_CLD_HA_2015_USD_x_1.20
4,NO2_CLD,NO2,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico bajo,12300.0,UNIT_VALUE_CLD_HA_2015_USD_x_0.80
5,NO2_CLD,NO2,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico central,15375.0,UNIT_VALUE_CLD_HA_2015_USD_x_1.00
6,O3_MAIN,O3,Respiratory mortality,VSL,Escenario económico alto,10446136.8,VSL_2015_USD_x_1.20
7,O3_MAIN,O3,Respiratory mortality,VSL,Escenario económico bajo,6964091.2,VSL_2015_USD_x_0.80
8,O3_MAIN,O3,Respiratory mortality,VSL,Escenario económico central,8705114.0,VSL_2015_USD_x_1.00
9,PM25_MAIN,PM25,All-cause mortality,VSL,Escenario económico alto,10446136.8,VSL_2015_USD_x_1.20



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\valoracion\escenarios_valoracion.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\valoracion\escenarios_valoracion_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\15_escenarios_valoracion.json


## Paso 16. Recalcular la valoración económica bajo escenarios bajos, centrales y altos

En esta etapa se recalcula la valoración económica variando únicamente el parámetro monetario de cada corrida. La finalidad es identificar cuánto cambia la carga económica estimada cuando se modifica el valor estadístico de la vida o el valor unitario por caso, manteniendo constantes los resultados sanitarios obtenidos en la corrida base.

La lógica de este paso consiste en combinar los impactos sanitarios ya calculados con los escenarios económicos definidos previamente. Así, para cada corrida se construyen tres resultados monetarios: uno bajo, uno central y uno alto. Esto permite medir la sensibilidad del componente económico sin alterar la exposición, la población, la incidencia ni el coeficiente epidemiológico.

El resultado será una tabla detallada por celda y una tabla resumen por corrida y escenario económico, junto con una comparación frente al escenario central. De esta manera se podrá evaluar qué tan dependiente es la carga económica final respecto a la monetización adoptada.

In [7]:
# ============================================================
# PASO 16: recalcular valoración económica con sensibilidad económica
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetro de conversión
# ------------------------------------------------------------
FX_COP_PER_USD_2015 = 2773.43

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_main = salidas_sens / "03_corrida_principal"
sens_root = salidas_sens / "04_sensibilidad" / "valoracion"
logs_dir = salidas_sens / "00_logs_validacion"

health_path = out_main / "impactos_sanitarios_por_celda.csv"
econ_scenarios_path = sens_root / "escenarios_valoracion.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

if not health_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{health_path}")
if not econ_scenarios_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{econ_scenarios_path}")

# ------------------------------------------------------------
# 3) Cargar datos
# ------------------------------------------------------------
health = pd.read_csv(health_path, encoding="utf-8-sig")
econ_scenarios = pd.read_csv(econ_scenarios_path, encoding="utf-8-sig")

health.columns = [str(c).replace("\ufeff", "").strip() for c in health.columns]
econ_scenarios.columns = [str(c).replace("\ufeff", "").strip() for c in econ_scenarios.columns]

for c in ["avoided_cases"]:
    if c in health.columns:
        health[c] = pd.to_numeric(health[c], errors="coerce")

for c in ["valuation_value_usd_2015"]:
    if c in econ_scenarios.columns:
        econ_scenarios[c] = pd.to_numeric(econ_scenarios[c], errors="coerce")

# ------------------------------------------------------------
# 4) Cruzar impactos sanitarios con escenarios económicos
# ------------------------------------------------------------
sens_econ = health.merge(
    econ_scenarios[[
        "run_id", "economic_scenario", "valuation_value_usd_2015", "valuation_reference"
    ]],
    on="run_id",
    how="inner"
)

# ------------------------------------------------------------
# 5) Recalcular valoración económica
# ------------------------------------------------------------
sens_econ["economic_value_usd_2015"] = (
    sens_econ["avoided_cases"] * sens_econ["valuation_value_usd_2015"]
)

sens_econ["economic_value_cop_2015"] = (
    sens_econ["economic_value_usd_2015"] * FX_COP_PER_USD_2015
)

# ------------------------------------------------------------
# 6) Ordenar columnas
# ------------------------------------------------------------
sens_econ = sens_econ[[
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "economic_method", "economic_scenario", "valuation_reference",
    "cell_id", "Row", "Column",
    "avoided_cases",
    "valuation_value_usd_2015",
    "economic_value_usd_2015", "economic_value_cop_2015"
]].sort_values(["run_id", "economic_scenario", "cell_id"]).reset_index(drop=True)

# ------------------------------------------------------------
# 7) Resumen por corrida y escenario económico
# ------------------------------------------------------------
summary = (
    sens_econ.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method", "economic_scenario"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        avoided_cases_total=("avoided_cases", "sum"),
        valuation_value_usd_2015=("valuation_value_usd_2015", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 8) Comparación frente al escenario económico central
# ------------------------------------------------------------
central_ref = (
    summary[summary["economic_scenario"] == "Escenario económico central"][
        ["run_id", "economic_value_total_usd_2015", "economic_value_total_cop_2015"]
    ]
    .rename(columns={
        "economic_value_total_usd_2015": "economic_usd_central",
        "economic_value_total_cop_2015": "economic_cop_central"
    })
)

summary_compare = summary.merge(central_ref, on="run_id", how="left")

summary_compare["diferencia_usd_vs_central"] = (
    summary_compare["economic_value_total_usd_2015"] - summary_compare["economic_usd_central"]
)

summary_compare["razon_usd_vs_central"] = np.where(
    summary_compare["economic_usd_central"].abs() > 0,
    summary_compare["economic_value_total_usd_2015"] / summary_compare["economic_usd_central"],
    np.nan
)

summary_compare["diferencia_cop_vs_central"] = (
    summary_compare["economic_value_total_cop_2015"] - summary_compare["economic_cop_central"]
)

summary_compare["razon_cop_vs_central"] = np.where(
    summary_compare["economic_cop_central"].abs() > 0,
    summary_compare["economic_value_total_cop_2015"] / summary_compare["economic_cop_central"],
    np.nan
)

# ------------------------------------------------------------
# 9) Mostrar resultados
# ------------------------------------------------------------
print("=" * 80)
print("SENSIBILIDAD ECONÓMICA: VALORACIÓN BAJA, CENTRAL Y ALTA")
print("=" * 80)
display(summary_compare)

print("\nVista previa por celda:")
display(sens_econ.head(20))

# ------------------------------------------------------------
# 10) Exportar
# ------------------------------------------------------------
cells_csv = sens_root / "sensibilidad_valoracion_por_celda.csv"
summary_csv = sens_root / "sensibilidad_valoracion_resumen.csv"
compare_csv = sens_root / "sensibilidad_valoracion_resumen_comparado_central.csv"
log_json = logs_dir / "16_sensibilidad_valoracion_completa.json"

sens_econ.to_csv(cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "FX_COP_PER_USD_2015": FX_COP_PER_USD_2015,
    "sens_econ_shape": list(sens_econ.shape),
    "summary_shape": list(summary.shape),
    "summary_compare_shape": list(summary_compare.shape),
    "outputs": {
        "cells_csv": str(cells_csv),
        "summary_csv": str(summary_csv),
        "compare_csv": str(compare_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", cells_csv)
print("-", summary_csv)
print("-", compare_csv)
print("-", log_json)

SENSIBILIDAD ECONÓMICA: VALORACIÓN BAJA, CENTRAL Y ALTA


,run_id,pollutant,endpoint_group,endpoint,economic_method,economic_scenario,n_rows,n_unique_cells,avoided_cases_total,valuation_value_usd_2015,economic_value_total_usd_2015,economic_value_total_cop_2015,economic_usd_central,economic_cop_central,diferencia_usd_vs_central,razon_usd_vs_central,diferencia_cop_vs_central,razon_cop_vs_central
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,254,254,12.713526,13478.4,1.713580e+05,4.752494e+08,1.427983e+05,3.960412e+08,2.855967e+04,1.2,7.920823e+07,1.2
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico bajo,254,254,12.713526,8985.6,1.142387e+05,3.168329e+08,1.427983e+05,3.960412e+08,-2.855967e+04,0.8,-7.920823e+07,0.8
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico central,254,254,12.713526,11232.0,1.427983e+05,3.960412e+08,1.427983e+05,3.960412e+08,0.000000e+00,1.0,0.000000e+00,1.0
3,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico alto,254,254,9.454138,18450.0,1.744288e+05,4.837662e+08,1.453574e+05,4.031385e+08,2.907147e+04,1.2,8.062770e+07,1.2
4,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico bajo,254,254,9.454138,12300.0,1.162859e+05,3.225108e+08,1.453574e+05,4.031385e+08,-2.907147e+04,0.8,-8.062770e+07,0.8
5,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario económico central,254,254,9.454138,15375.0,1.453574e+05,4.031385e+08,1.453574e+05,4.031385e+08,0.000000e+00,1.0,0.000000e+00,1.0
6,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario económico alto,254,254,2.204553,10446136.8,2.302906e+07,6.386948e+10,1.919088e+07,5.322457e+10,3.838177e+06,1.2,1.064491e+10,1.2
7,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario económico bajo,254,254,2.204553,6964091.2,1.535271e+07,4.257966e+10,1.919088e+07,5.322457e+10,-3.838177e+06,0.8,-1.064491e+10,0.8
8,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario económico central,254,254,2.204553,8705114.0,1.919088e+07,5.322457e+10,1.919088e+07,5.322457e+10,0.000000e+00,1.0,0.000000e+00,1.0
9,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,Escenario económico alto,254,254,1.299799,10446136.8,1.357788e+07,3.765730e+10,1.131490e+07,3.138108e+10,2.262980e+06,1.2,6.276216e+09,1.2



Vista previa por celda:


,run_id,pollutant,endpoint_group,endpoint,economic_method,economic_scenario,valuation_reference,cell_id,Row,Column,avoided_cases,valuation_value_usd_2015,economic_value_usd_2015,economic_value_cop_2015
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,0,1,1,0.051939,13478.4,700.057288,1.941560e+06
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,1,2,1,0.028832,13478.4,388.614272,1.077794e+06
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,2,3,1,0.000528,13478.4,7.115416,1.973411e+04
3,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,41,1,2,0.042571,13478.4,573.791295,1.591370e+06
4,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,42,2,2,0.066629,13478.4,898.050296,2.490680e+06
5,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,43,3,2,0.051455,13478.4,693.533032,1.923465e+06
6,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,44,4,2,0.006968,13478.4,93.914987,2.604666e+05
7,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,82,1,3,0.004800,13478.4,64.695896,1.794295e+05
8,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,83,2,3,0.030429,13478.4,410.131175,1.137470e+06
9,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario económico alto,UNIT_VALUE_ASTHMA_HA_2015_USD_x_1.20,84,3,3,0.068281,13478.4,920.315313,2.552430e+06



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\valoracion\sensibilidad_valoracion_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\valoracion\sensibilidad_valoracion_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\valoracion\sensibilidad_valoracion_resumen_comparado_central.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\16_sensibilidad_valoracion_completa.json


## Paso 17. Construcción de la tabla resumen integrada de resultados y sensibilidad

En esta etapa se integran en una sola tabla los principales resultados del pipeline: la corrida base, la sensibilidad asociada a la incertidumbre del HBM, la sensibilidad por beta y la sensibilidad económica. La finalidad es disponer de una visión consolidada que permita comparar, para cada corrida, cómo cambia la carga sanitaria y económica bajo diferentes fuentes de incertidumbre y de variación paramétrica.

La tabla se organiza en cuatro bloques. El primero corresponde a la corrida base. El segundo resume la sensibilidad HBM mediante escenarios bajo, central y alto. El tercero resume la sensibilidad epidemiológica asociada a beta. El cuarto resume la sensibilidad económica asociada a la monetización. En todos los casos se conservan los resultados clave: casos evitados, valor económico en dólares de 2015 y valor económico en pesos colombianos de 2015.

El resultado esperado es una tabla integrada, clara y trazable, útil tanto para la interpretación analítica como para la construcción posterior de tablas finales, gráficos y redacción del documento.

## Paso 17. Construcción de la tabla resumen integrada de resultados y sensibilidad

En esta etapa se integran en una sola tabla los principales resultados del pipeline: la corrida base, la sensibilidad asociada a la incertidumbre del HBM, la sensibilidad por beta y la sensibilidad económica. La finalidad es disponer de una visión consolidada que permita comparar, para cada corrida, cómo cambia la carga sanitaria y económica bajo diferentes fuentes de incertidumbre y de variación paramétrica.

La tabla se organiza en cuatro bloques. El primero corresponde a la corrida base. El segundo resume la sensibilidad HBM mediante escenarios bajo, central y alto. El tercero resume la sensibilidad epidemiológica asociada a beta. El cuarto resume la sensibilidad económica asociada a la monetización. En todos los casos se conservan los resultados clave: casos evitados, valor económico en dólares de 2015 y valor económico en pesos colombianos de 2015.

El resultado esperado es una tabla integrada, clara y trazable, útil tanto para la interpretación analítica como para la construcción posterior de tablas finales, gráficos y redacción del documento.

In [8]:
# ============================================================
# PASO 17: tabla resumen integrada final
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_main = salidas_sens / "03_corrida_principal"
sens_hbm = salidas_sens / "04_sensibilidad" / "percentiles_hbm"
sens_beta = salidas_sens / "04_sensibilidad" / "beta"
sens_val = salidas_sens / "04_sensibilidad" / "valoracion"
out_tables = salidas_sens / "07_tablas"
out_final = salidas_sens / "08_resumenes_finales"
logs_dir = salidas_sens / "00_logs_validacion"

for folder in [out_tables, out_final, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

base_path = out_main / "valoracion_economica_resumen_usd2015.csv"
base_cop_path = out_tables / "valoracion_economica_resumen_cop.csv"
hbm_path = sens_hbm / "sensibilidad_hbm_resumen_comparado_p50.csv"
beta_path = sens_beta / "sensibilidad_beta_resumen_comparado_central.csv"
val_path = sens_val / "sensibilidad_valoracion_resumen_comparado_central.csv"

# ------------------------------------------------------------
# 2) Lectura segura
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo:\n{path}")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

df_base_usd = read_csv_safe(base_path)
df_base_cop = read_csv_safe(base_cop_path)
df_hbm = read_csv_safe(hbm_path)
df_beta = read_csv_safe(beta_path)
df_val = read_csv_safe(val_path)

# ------------------------------------------------------------
# 3) Mapeos de etiquetas
# ------------------------------------------------------------
corrida_map = {
    "PM25_MAIN": "PM2.5 - Mortalidad por todas las causas",
    "O3_MAIN": "O3 - Mortalidad respiratoria",
    "NO2_ASTHMA": "NO2 - Hospitalizaciones por asma",
    "NO2_CLD": "NO2 - Hospitalizaciones por enfermedad pulmonar crónica"
}

# ------------------------------------------------------------
# 4) Corrida base
# ------------------------------------------------------------
base = df_base_usd.merge(
    df_base_cop[["run_id", "economic_value_total_cop_2015"]],
    on="run_id",
    how="left"
).copy()

base["Tipo de resultado"] = "Corrida base"
base["Escenario"] = "Escenario central"
base["Corrida"] = base["run_id"].map(corrida_map).fillna(base["run_id"])

base_final = base[[
    "Tipo de resultado", "Corrida", "run_id", "pollutant",
    "endpoint_group", "endpoint", "Escenario",
    "avoided_cases_total", "economic_value_total_usd_2015", "economic_value_total_cop_2015"
]].copy()

base_final["Casos evitados"] = pd.to_numeric(base_final["avoided_cases_total"], errors="coerce")
base_final["Valor económico total (USD 2015)"] = pd.to_numeric(base_final["economic_value_total_usd_2015"], errors="coerce")
base_final["Valor económico total (COP 2015)"] = pd.to_numeric(base_final["economic_value_total_cop_2015"], errors="coerce")
base_final["Cambio vs. escenario central"] = np.nan
base_final["Razón vs. escenario central"] = np.nan

base_final = base_final[[
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "Casos evitados", "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)", "Cambio vs. escenario central", "Razón vs. escenario central"
]]

# ------------------------------------------------------------
# 5) Sensibilidad HBM
# ------------------------------------------------------------
hbm = df_hbm.copy()
hbm["Tipo de resultado"] = "Sensibilidad HBM"
hbm["Corrida"] = hbm["run_id"].map(corrida_map).fillna(hbm["run_id"])

map_hbm = {
    "p05": "Escenario bajo (p05)",
    "p50": "Escenario central (p50)",
    "p95": "Escenario alto (p95)"
}
hbm["Escenario"] = hbm["percentile_hbm"].map(map_hbm).fillna(hbm["percentile_hbm"])

hbm_final = hbm[[
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "avoided_cases_total", "economic_value_total_usd_2015",
    "economic_value_total_cop_2015", "delta_vs_p50_cases", "ratio_vs_p50_cases"
]].copy()

hbm_final.columns = [
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "Casos evitados", "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)", "Cambio vs. escenario central", "Razón vs. escenario central"
]

# ------------------------------------------------------------
# 6) Sensibilidad beta
# ------------------------------------------------------------
beta = df_beta.copy()
beta["Tipo de resultado"] = "Sensibilidad beta"
beta["Corrida"] = beta["run_id"].map(corrida_map).fillna(beta["run_id"])
beta["Escenario"] = beta["beta_scenario"]

beta_final = beta[[
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "avoided_cases_total", "economic_value_total_usd_2015",
    "economic_value_total_cop_2015", "diferencia_casos_vs_central", "razon_casos_vs_central"
]].copy()

beta_final.columns = [
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "Casos evitados", "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)", "Cambio vs. escenario central", "Razón vs. escenario central"
]

# ------------------------------------------------------------
# 7) Sensibilidad económica
# ------------------------------------------------------------
val = df_val.copy()
val["Tipo de resultado"] = "Sensibilidad económica"
val["Corrida"] = val["run_id"].map(corrida_map).fillna(val["run_id"])
val["Escenario"] = val["economic_scenario"]

val["Casos evitados"] = pd.to_numeric(val["avoided_cases_total"], errors="coerce")
val["Valor económico total (USD 2015)"] = pd.to_numeric(val["economic_value_total_usd_2015"], errors="coerce")
val["Valor económico total (COP 2015)"] = pd.to_numeric(val["economic_value_total_cop_2015"], errors="coerce")
val["Cambio vs. escenario central"] = pd.to_numeric(val["diferencia_usd_vs_central"], errors="coerce")
val["Razón vs. escenario central"] = pd.to_numeric(val["razon_usd_vs_central"], errors="coerce")

val_final = val[[
    "Tipo de resultado", "Corrida", "run_id", "pollutant", "endpoint_group", "endpoint",
    "Escenario", "Casos evitados", "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)", "Cambio vs. escenario central", "Razón vs. escenario central"
]].copy()

# ------------------------------------------------------------
# 8) Unir todo
# ------------------------------------------------------------
tabla_integrada = pd.concat(
    [base_final, hbm_final, beta_final, val_final],
    ignore_index=True
)

# ------------------------------------------------------------
# 9) Orden y formato
# ------------------------------------------------------------
orden_tipo = {
    "Corrida base": 1,
    "Sensibilidad HBM": 2,
    "Sensibilidad beta": 3,
    "Sensibilidad económica": 4
}

orden_corrida = {
    "PM2.5 - Mortalidad por todas las causas": 1,
    "O3 - Mortalidad respiratoria": 2,
    "NO2 - Hospitalizaciones por asma": 3,
    "NO2 - Hospitalizaciones por enfermedad pulmonar crónica": 4
}

tabla_integrada["orden_tipo"] = tabla_integrada["Tipo de resultado"].map(orden_tipo).fillna(999)
tabla_integrada["orden_corrida"] = tabla_integrada["Corrida"].map(orden_corrida).fillna(999)

tabla_integrada = (
    tabla_integrada
    .sort_values(["orden_tipo", "orden_corrida", "Escenario"])
    .drop(columns=["orden_tipo", "orden_corrida"])
    .reset_index(drop=True)
)

for col in ["Casos evitados", "Cambio vs. escenario central", "Razón vs. escenario central"]:
    tabla_integrada[col] = pd.to_numeric(tabla_integrada[col], errors="coerce").round(3)

for col in ["Valor económico total (USD 2015)", "Valor económico total (COP 2015)"]:
    tabla_integrada[col] = pd.to_numeric(tabla_integrada[col], errors="coerce").round(0)

# ------------------------------------------------------------
# 10) Mostrar
# ------------------------------------------------------------
print("=" * 90)
print("TABLA RESUMEN INTEGRADA FINAL")
print("=" * 90)
display(tabla_integrada)

# ------------------------------------------------------------
# 11) Exportar
# ------------------------------------------------------------
tabla_csv = out_tables / "tabla_resumen_integrada_final.csv"
tabla_xlsx = out_final / "tabla_resumen_integrada_final.xlsx"
log_json = logs_dir / "17_tabla_resumen_integrada_final.json"

tabla_integrada.to_csv(tabla_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(tabla_xlsx, engine="openpyxl") as writer:
    tabla_integrada.to_excel(writer, index=False, sheet_name="Resumen_integrado")

payload = {
    "timestamp": datetime.now().isoformat(),
    "tabla_shape": list(tabla_integrada.shape),
    "outputs": {
        "tabla_csv": str(tabla_csv),
        "tabla_xlsx": str(tabla_xlsx)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", tabla_csv)
print("-", tabla_xlsx)
print("-", log_json)

TABLA RESUMEN INTEGRADA FINAL


,Tipo de resultado,Corrida,run_id,pollutant,endpoint_group,endpoint,Escenario,Casos evitados,Valor económico total (USD 2015),Valor económico total (COP 2015),Cambio vs. escenario central,Razón vs. escenario central
0,Corrida base,PM2.5 - Mortalidad por todas las causas,PM25_MAIN,PM25,Mortality,All-cause mortality,Escenario central,1.300,11314899.0,3.138108e+10,NaN,NaN
1,Corrida base,O3 - Mortalidad respiratoria,O3_MAIN,O3,Mortality,Respiratory mortality,Escenario central,2.205,19190883.0,5.322457e+10,NaN,NaN
2,Corrida base,NO2 - Hospitalizaciones por asma,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,Escenario central,12.714,142798.0,3.960412e+08,NaN,NaN
3,Corrida base,NO2 - Hospitalizaciones por enfermedad pulmona...,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,Escenario central,9.454,145357.0,4.031385e+08,NaN,NaN
4,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,PM25_MAIN,PM25,Mortality,All-cause mortality,Escenario alto (p95),42.675,371489563.0,1.030300e+12,41.375,32.832
5,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,PM25_MAIN,PM25,Mortality,All-cause mortality,Escenario bajo (p05),-9.035,-78647386.0,-2.181230e+11,-10.334,-6.951
6,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,PM25_MAIN,PM25,Mortality,All-cause mortality,Escenario central (p50),1.300,11314899.0,3.138108e+10,0.000,1.000
7,Sensibilidad HBM,O3 - Mortalidad respiratoria,O3_MAIN,O3,Mortality,Respiratory mortality,Escenario alto (p95),91.848,799542966.0,2.217476e+12,89.643,41.663
8,Sensibilidad HBM,O3 - Mortalidad respiratoria,O3_MAIN,O3,Mortality,Respiratory mortality,Escenario bajo (p05),-14.718,-128124176.0,-3.553434e+11,-16.923,-6.676
9,Sensibilidad HBM,O3 - Mortalidad respiratoria,O3_MAIN,O3,Mortality,Respiratory mortality,Escenario central (p50),2.205,19190883.0,5.322457e+10,0.000,1.000



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\tabla_resumen_integrada_final.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\08_resumenes_finales\tabla_resumen_integrada_final.xlsx
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\17_tabla_resumen_integrada_final.json


## Paso 18. Formato legible de valores monetarios y sanitarios

En esta etapa se ajusta la tabla resumen integrada para que los resultados se vean en un formato más comprensible. La finalidad es mantener las columnas numéricas originales para análisis y trazabilidad, pero añadir columnas de lectura con expresiones más claras, por ejemplo en millones o en miles de millones.

La lógica del formato será la siguiente. Los casos evitados se presentarán con un número reducido de decimales. Los valores monetarios en dólares y en pesos se expresarán en unidades legibles, usando millones cuando la magnitud sea moderada y miles de millones cuando el valor sea más alto. De esta manera, la tabla conserva rigor cuantitativo y al mismo tiempo gana claridad para interpretación y redacción.

El resultado será una versión más amigable de la tabla integrada final, útil para revisión, discusión de resultados y preparación posterior de tablas para el documento.

In [9]:
# ============================================================
# PASO 18: formato legible de números en la tabla integrada
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_tables = salidas_sens / "07_tablas"
out_final = salidas_sens / "08_resumenes_finales"
logs_dir = salidas_sens / "00_logs_validacion"

tabla_path = out_tables / "tabla_resumen_integrada_final.csv"

for folder in [out_tables, out_final, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not tabla_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{tabla_path}")

# ------------------------------------------------------------
# 2) Cargar tabla
# ------------------------------------------------------------
df = pd.read_csv(tabla_path, encoding="utf-8-sig")
df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

# ------------------------------------------------------------
# 3) Funciones de formato
# ------------------------------------------------------------
def format_number_es(x, decimals=2):
    if pd.isna(x):
        return ""
    s = f"{x:,.{decimals}f}"
    # 1,234,567.89 -> 1.234.567,89
    s = s.replace(",", "X").replace(".", ",").replace("X", ".")
    return s

def format_money_readable(x, currency="COP"):
    if pd.isna(x):
        return ""
    x = float(x)
    sign = "-" if x < 0 else ""
    x_abs = abs(x)

    if x_abs >= 1_000_000_000_000:
        value = x_abs / 1_000_000_000_000
        unit = "billones"
    elif x_abs >= 1_000_000_000:
        value = x_abs / 1_000_000_000
        unit = "mil millones"
    elif x_abs >= 1_000_000:
        value = x_abs / 1_000_000
        unit = "millones"
    elif x_abs >= 1_000:
        value = x_abs / 1_000
        unit = "mil"
    else:
        return f"{sign}{format_number_es(x_abs, 2)} {currency}"

    return f"{sign}{format_number_es(value, 2)} {unit} {currency}"

def format_cases(x):
    if pd.isna(x):
        return ""
    return format_number_es(float(x), 3)

def format_change_by_type(row):
    x = row.get("Cambio vs. escenario central", np.nan)
    if pd.isna(x):
        return ""
    tipo = str(row.get("Tipo de resultado", "")).strip()

    # En sensibilidad económica, el cambio quedó en USD 2015
    if tipo == "Sensibilidad económica":
        return format_money_readable(x, currency="USD")
    else:
        return format_number_es(float(x), 3)

# ------------------------------------------------------------
# 4) Crear columnas legibles
# ------------------------------------------------------------
for c in [
    "Casos evitados",
    "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)",
    "Cambio vs. escenario central",
    "Razón vs. escenario central"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["Casos evitados (lectura)"] = df["Casos evitados"].apply(format_cases)
df["Valor económico USD 2015 (lectura)"] = df["Valor económico total (USD 2015)"].apply(
    lambda x: format_money_readable(x, currency="USD")
)
df["Valor económico COP 2015 (lectura)"] = df["Valor económico total (COP 2015)"].apply(
    lambda x: format_money_readable(x, currency="COP")
)
df["Cambio vs. escenario central (lectura)"] = df.apply(format_change_by_type, axis=1)
df["Razón vs. escenario central (lectura)"] = df["Razón vs. escenario central"].apply(
    lambda x: "" if pd.isna(x) else format_number_es(float(x), 3)
)

# ------------------------------------------------------------
# 5) Tabla amigable final
# ------------------------------------------------------------
tabla_legible = df[[
    "Tipo de resultado",
    "Corrida",
    "Escenario",
    "Casos evitados (lectura)",
    "Valor económico USD 2015 (lectura)",
    "Valor económico COP 2015 (lectura)",
    "Cambio vs. escenario central (lectura)",
    "Razón vs. escenario central (lectura)"
]].copy()

print("=" * 90)
print("TABLA INTEGRADA CON FORMATO LEGIBLE")
print("=" * 90)
display(tabla_legible)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
tabla_csv = out_tables / "tabla_resumen_integrada_legible.csv"
tabla_xlsx = out_final / "tabla_resumen_integrada_legible.xlsx"
log_json = logs_dir / "18_tabla_resumen_integrada_legible.json"

tabla_legible.to_csv(tabla_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(tabla_xlsx, engine="openpyxl") as writer:
    tabla_legible.to_excel(writer, index=False, sheet_name="Resumen_legible")

payload = {
    "timestamp": datetime.now().isoformat(),
    "tabla_shape": list(tabla_legible.shape),
    "outputs": {
        "tabla_csv": str(tabla_csv),
        "tabla_xlsx": str(tabla_xlsx)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", tabla_csv)
print("-", tabla_xlsx)
print("-", log_json)

TABLA INTEGRADA CON FORMATO LEGIBLE


,Tipo de resultado,Corrida,Escenario,Casos evitados (lectura),Valor económico USD 2015 (lectura),Valor económico COP 2015 (lectura),Cambio vs. escenario central (lectura),Razón vs. escenario central (lectura)
0,Corrida base,PM2.5 - Mortalidad por todas las causas,Escenario central,"1,300","11,31 millones USD","31,38 mil millones COP",,
1,Corrida base,O3 - Mortalidad respiratoria,Escenario central,"2,205","19,19 millones USD","53,22 mil millones COP",,
2,Corrida base,NO2 - Hospitalizaciones por asma,Escenario central,"12,714","142,80 mil USD","396,04 millones COP",,
3,Corrida base,NO2 - Hospitalizaciones por enfermedad pulmona...,Escenario central,"9,454","145,36 mil USD","403,14 millones COP",,
4,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario alto (p95),"42,675","371,49 millones USD","1,03 billones COP","41,375","32,832"
5,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario bajo (p05),"-9,035","-78,65 millones USD","-218,12 mil millones COP","-10,334","-6,951"
6,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario central (p50),"1,300","11,31 millones USD","31,38 mil millones COP","0,000","1,000"
7,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario alto (p95),"91,848","799,54 millones USD","2,22 billones COP","89,643","41,663"
8,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario bajo (p05),"-14,718","-128,12 millones USD","-355,34 mil millones COP","-16,923","-6,676"
9,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario central (p50),"2,205","19,19 millones USD","53,22 mil millones COP","0,000","1,000"



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\tabla_resumen_integrada_legible.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\08_resumenes_finales\tabla_resumen_integrada_legible.xlsx
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\18_tabla_resumen_integrada_legible.json


## Paso 18b. Ajuste del formato legible para mostrar solo millones y mil millones en pesos colombianos

En esta etapa se ajusta el formato de lectura de la tabla integrada para que los valores monetarios en pesos colombianos se expresen únicamente en millones o en mil millones. La finalidad es evitar el uso de billones, que puede generar confusión visual o interpretativa, y dejar una salida más homogénea para revisión y para posible uso en el documento.

La lógica será mantener intactas las columnas numéricas originales, pero modificar las columnas de lectura. En el caso de los valores en dólares se conservará el formato legible habitual. En el caso de los valores en pesos colombianos, toda magnitud grande se expresará solamente en millones o en mil millones, incluso cuando el valor sea muy alto.

El resultado será una nueva versión legible de la tabla integrada final, más clara y más consistente para presentar resultados económicos en el contexto del estudio.

In [10]:
# ============================================================
# PASO 18b: formato legible usando solo millones y mil millones en COP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name.upper() == "CODIGO" else current_dir

salidas_sens = project_root / "SALIDAS_SENSIBILIDAD"
out_tables = salidas_sens / "07_tablas"
out_final = salidas_sens / "08_resumenes_finales"
logs_dir = salidas_sens / "00_logs_validacion"

tabla_path = out_tables / "tabla_resumen_integrada_final.csv"

for folder in [out_tables, out_final, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if not tabla_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{tabla_path}")

# ------------------------------------------------------------
# 2) Cargar tabla
# ------------------------------------------------------------
df = pd.read_csv(tabla_path, encoding="utf-8-sig")
df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

# ------------------------------------------------------------
# 3) Funciones de formato
# ------------------------------------------------------------
def format_number_es(x, decimals=2):
    if pd.isna(x):
        return ""
    s = f"{x:,.{decimals}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def format_money_readable_usd(x):
    if pd.isna(x):
        return ""
    x = float(x)
    sign = "-" if x < 0 else ""
    x_abs = abs(x)

    if x_abs >= 1_000_000:
        return f"{sign}{format_number_es(x_abs / 1_000_000, 2)} millones USD"
    elif x_abs >= 1_000:
        return f"{sign}{format_number_es(x_abs / 1_000, 2)} mil USD"
    else:
        return f"{sign}{format_number_es(x_abs, 2)} USD"

def format_money_readable_cop(x):
    if pd.isna(x):
        return ""
    x = float(x)
    sign = "-" if x < 0 else ""
    x_abs = abs(x)

    # SOLO millones y mil millones
    if x_abs >= 1_000_000_000:
        return f"{sign}{format_number_es(x_abs / 1_000_000_000, 2)} mil millones COP"
    elif x_abs >= 1_000_000:
        return f"{sign}{format_number_es(x_abs / 1_000_000, 2)} millones COP"
    elif x_abs >= 1_000:
        return f"{sign}{format_number_es(x_abs / 1_000, 2)} mil COP"
    else:
        return f"{sign}{format_number_es(x_abs, 2)} COP"

def format_cases(x):
    if pd.isna(x):
        return ""
    return format_number_es(float(x), 3)

def format_change_by_type(row):
    x = row.get("Cambio vs. escenario central", np.nan)
    if pd.isna(x):
        return ""

    tipo = str(row.get("Tipo de resultado", "")).strip()

    if tipo == "Sensibilidad económica":
        return format_money_readable_usd(x)
    else:
        return format_number_es(float(x), 3)

# ------------------------------------------------------------
# 4) Normalizar columnas numéricas
# ------------------------------------------------------------
for c in [
    "Casos evitados",
    "Valor económico total (USD 2015)",
    "Valor económico total (COP 2015)",
    "Cambio vs. escenario central",
    "Razón vs. escenario central"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ------------------------------------------------------------
# 5) Crear columnas legibles nuevas
# ------------------------------------------------------------
df["Casos evitados (lectura)"] = df["Casos evitados"].apply(format_cases)
df["Valor económico USD 2015 (lectura)"] = df["Valor económico total (USD 2015)"].apply(format_money_readable_usd)
df["Valor económico COP 2015 (lectura)"] = df["Valor económico total (COP 2015)"].apply(format_money_readable_cop)
df["Cambio vs. escenario central (lectura)"] = df.apply(format_change_by_type, axis=1)
df["Razón vs. escenario central (lectura)"] = df["Razón vs. escenario central"].apply(
    lambda x: "" if pd.isna(x) else format_number_es(float(x), 3)
)

# ------------------------------------------------------------
# 6) Tabla legible final
# ------------------------------------------------------------
tabla_legible = df[[
    "Tipo de resultado",
    "Corrida",
    "Escenario",
    "Casos evitados (lectura)",
    "Valor económico USD 2015 (lectura)",
    "Valor económico COP 2015 (lectura)",
    "Cambio vs. escenario central (lectura)",
    "Razón vs. escenario central (lectura)"
]].copy()

print("=" * 90)
print("TABLA INTEGRADA LEGIBLE AJUSTADA")
print("=" * 90)
display(tabla_legible)

# ------------------------------------------------------------
# 7) Exportar
# ------------------------------------------------------------
tabla_csv = out_tables / "tabla_resumen_integrada_legible.csv"
tabla_xlsx = out_final / "tabla_resumen_integrada_legible.xlsx"
log_json = logs_dir / "18b_tabla_resumen_integrada_legible_ajustada.json"

tabla_legible.to_csv(tabla_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(tabla_xlsx, engine="openpyxl") as writer:
    tabla_legible.to_excel(writer, index=False, sheet_name="Resumen_legible")

payload = {
    "timestamp": datetime.now().isoformat(),
    "tabla_shape": list(tabla_legible.shape),
    "outputs": {
        "tabla_csv": str(tabla_csv),
        "tabla_xlsx": str(tabla_xlsx)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", tabla_csv)
print("-", tabla_xlsx)
print("-", log_json)

TABLA INTEGRADA LEGIBLE AJUSTADA


,Tipo de resultado,Corrida,Escenario,Casos evitados (lectura),Valor económico USD 2015 (lectura),Valor económico COP 2015 (lectura),Cambio vs. escenario central (lectura),Razón vs. escenario central (lectura)
0,Corrida base,PM2.5 - Mortalidad por todas las causas,Escenario central,"1,300","11,31 millones USD","31,38 mil millones COP",,
1,Corrida base,O3 - Mortalidad respiratoria,Escenario central,"2,205","19,19 millones USD","53,22 mil millones COP",,
2,Corrida base,NO2 - Hospitalizaciones por asma,Escenario central,"12,714","142,80 mil USD","396,04 millones COP",,
3,Corrida base,NO2 - Hospitalizaciones por enfermedad pulmona...,Escenario central,"9,454","145,36 mil USD","403,14 millones COP",,
4,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario alto (p95),"42,675","371,49 millones USD","1.030,30 mil millones COP","41,375","32,832"
5,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario bajo (p05),"-9,035","-78,65 millones USD","-218,12 mil millones COP","-10,334","-6,951"
6,Sensibilidad HBM,PM2.5 - Mortalidad por todas las causas,Escenario central (p50),"1,300","11,31 millones USD","31,38 mil millones COP","0,000","1,000"
7,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario alto (p95),"91,848","799,54 millones USD","2.217,48 mil millones COP","89,643","41,663"
8,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario bajo (p05),"-14,718","-128,12 millones USD","-355,34 mil millones COP","-16,923","-6,676"
9,Sensibilidad HBM,O3 - Mortalidad respiratoria,Escenario central (p50),"2,205","19,19 millones USD","53,22 mil millones COP","0,000","1,000"



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\tabla_resumen_integrada_legible.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\08_resumenes_finales\tabla_resumen_integrada_legible.xlsx
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\18b_tabla_resumen_integrada_legible_ajustada.json


## Búsqueda de la tabla o capa que relaciona celdas con localidades

Esta celda recorre los archivos más probables del proyecto y revisa si contienen columnas asociadas a la identificación de celdas y a la localidad. El objetivo es encontrar rápidamente si ya existe una tabla o capa con la correspondencia entre `cell_id`, `Row`, `Column` y `Localidad`, sin volver a construirla desde cero.

Se revisan archivos tabulares y geográficos comunes, como CSV, GPKG, SHP y GeoJSON. Para cada archivo encontrado, la celda muestra su ruta, el tipo de archivo y las columnas relevantes detectadas. Con esto será posible identificar cuál archivo ya contiene la relación administrativa por celda.

In [12]:
from pathlib import Path
import pandas as pd

# geopandas solo si está disponible
try:
    import geopandas as gpd
    HAS_GPD = True
except Exception:
    HAS_GPD = False

# --------------------------------------------------
# 1) Ruta del proyecto
# --------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

# --------------------------------------------------
# 2) Patrones a revisar
# --------------------------------------------------
patterns = ["*.csv", "*.gpkg", "*.geojson", "*.shp"]

# columnas clave
keywords = [
    "cell_id", "cellid", "row", "column", "col",
    "localidad", "locality", "loc_name", "nombre_loc", "nombre"
]

results = []

# --------------------------------------------------
# 3) Función para extraer columnas
# --------------------------------------------------
def get_cols(path: Path):
    suffix = path.suffix.lower()
    try:
        if suffix == ".csv":
            df = pd.read_csv(path, encoding="utf-8-sig", nrows=5)
            return list(df.columns)
        elif suffix in [".gpkg", ".geojson", ".shp"] and HAS_GPD:
            gdf = gpd.read_file(path, rows=5)
            return list(gdf.columns)
        else:
            return None
    except Exception:
        return None

# --------------------------------------------------
# 4) Buscar archivos y columnas relevantes
# --------------------------------------------------
files = []
for pat in patterns:
    files.extend(project_root.rglob(pat))

for f in files:
    cols = get_cols(f)
    if not cols:
        continue

    cols_clean = [str(c).strip() for c in cols]
    cols_lower = [c.lower() for c in cols_clean]

    hits = [c for c in cols_clean if c.lower() in keywords]

    # también capturar coincidencias parciales útiles
    partial_hits = []
    for c in cols_clean:
        cl = c.lower()
        if any(k in cl for k in ["local", "loc", "cell", "row", "col"]):
            partial_hits.append(c)

    partial_hits = sorted(set(partial_hits))
    hits = sorted(set(hits + partial_hits))

    if hits:
        results.append({
            "archivo": f.name,
            "tipo": f.suffix.lower(),
            "ruta": str(f),
            "columnas_relevantes": ", ".join(hits),
            "n_columnas": len(cols_clean)
        })

res_df = pd.DataFrame(results).sort_values(["archivo", "tipo"]).reset_index(drop=True)

print("=" * 100)
print("ARCHIVOS CANDIDATOS PARA CELDA -> LOCALIDAD")
print("=" * 100)

if res_df.empty:
    print("No se encontraron archivos con columnas relevantes.")
else:
    display(res_df)

    print("\nSUGERENCIAS DE REVISIÓN PRIMERO:")
    prioridad = res_df[
        res_df["archivo"].str.contains("grid|grilla|local|bogota", case=False, na=False)
    ].copy()

    if not prioridad.empty:
        display(prioridad.head(15))

d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(
d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'grid_3km_time_idw_2020_2024.gpkg': 'grid_3km_static' (default), 'localidades_bogota', 'grid_3km_time_idw'. Specify layer parameter to avoid this warning.
  result = read_func(
d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'prueba.gpkg': 'grid_3km_static' (default), 'localidades_bogota', 'grid_3km_time_idw'. Specify layer parameter to avoid this warning.
  result = read_func(


ARCHIVOS CANDIDATOS PARA CELDA -> LOCALIDAD


,archivo,tipo,ruta,columnas_relevantes,n_columnas
0,EEVV Mortalidad 2020-24.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MOR...,"LOCALOCUHE, codigo_localidad",84
1,HBM_NO2_M1_fold_1_pred_test.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",13
2,HBM_NO2_M1_fold_2_pred_test.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",13
3,HBM_NO2_M1_fold_3_pred_test.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",13
4,HBM_NO2_M1_pred_test_all_folds.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",13
...,...,...,...,...,...
206,tokyo_GS_NN_listwise.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-pac...,localpdev,23
207,valoracion_economica_por_celda_cop.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,"Column, Row, cell_id, economic_value_cop_local...",30
208,valoracion_economica_por_celda_usd2015.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,"Column, Row, cell_id",25
209,valoracion_economica_resumen_cop.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,"economic_value_total_cop_local_base, local_bas...",14



SUGERENCIAS DE REVISIÓN PRIMERO:


,archivo,tipo,ruta,columnas_relevantes,n_columnas
14,HBM_NO2_grid_audit_extreme_rows.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,cell_id,19
15,HBM_NO2_grid_base_clean_v1.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",9
16,HBM_NO2_grid_base_v2.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,"cell_id, cell_idx",10
17,HBM_NO2_grid_clean_v1.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_...,cell_id,22
34,HBM_O3_grid_audit_extreme_rows.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_O...,cell_id,19
35,HBM_O3_grid_base_clean_v1.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_O...,"cell_id, cell_idx",9
36,HBM_O3_grid_base_clean_v3.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V3_S...,"cell_id, cell_idx",10
37,HBM_O3_grid_base_v2.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_O...,"cell_id, cell_idx",10
38,HBM_O3_grid_base_v3.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V3_S...,"cell_id, cell_idx",11
39,HBM_O3_grid_clean_v1.csv,.csv,D:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_O...,cell_id,22


## Paso 19a. Inspección de capas geográficas de grilla y localidades

En esta etapa se revisan las capas geográficas disponibles en el proyecto para identificar cuál contiene la grilla del análisis y cuál contiene la división administrativa de Bogotá. La finalidad es reconocer las rutas, capas, sistemas de referencia y columnas relevantes antes de construir la tabla de correspondencia entre celdas y localidades.

La lógica de esta celda es explorar los archivos geográficos más probables del proyecto, especialmente los de tipo GPKG y SHP. Para cada uno se listan las capas disponibles, las columnas detectadas y la geometría que contienen. Esto permitirá verificar si la capa de grilla trae identificadores de celda como `cell_id`, `Row` y `Column`, y si la capa administrativa trae una columna con el nombre de la localidad.

El resultado esperado es un diagnóstico claro de las capas que deben utilizarse en la unión espacial posterior. Con esta información se podrá construir de forma segura la relación entre celdas y localidades de Bogotá.

In [13]:
# ============================================================
# PASO 19a: inspeccionar capas geográficas candidatas
# ============================================================

from pathlib import Path
import pandas as pd

try:
    import geopandas as gpd
except Exception as e:
    raise ImportError(f"No se pudo importar geopandas: {e}")

try:
    import fiona
except Exception as e:
    raise ImportError(f"No se pudo importar fiona: {e}")

# ------------------------------------------------------------
# 1) Ruta del proyecto
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

# archivos/carpetas probables según tu estructura
candidate_paths = [
    project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg",
    project_root / "Qgis" / "prueba.gpkg",
    project_root / "GRILLA-GEOVISOR MAPAS BEN-MAP" / "bogota",
    project_root / "GRILLA-GEOVISOR MAPAS BEN-MAP" / "gadm41_COL_shp",
]

# ------------------------------------------------------------
# 2) Utilidades
# ------------------------------------------------------------
def list_vector_files(folder: Path):
    if not folder.exists():
        return []
    out = []
    for ext in ["*.gpkg", "*.shp", "*.geojson"]:
        out.extend(folder.rglob(ext))
    return sorted(set(out))

def inspect_vector(path: Path):
    records = []

    if path.suffix.lower() == ".gpkg":
        try:
            layers = fiona.listlayers(path)
        except Exception:
            layers = []

        if not layers:
            records.append({
                "archivo": path.name,
                "ruta": str(path),
                "layer": "(sin detectar)",
                "geom_type": "",
                "crs": "",
                "n_features_sample": "",
                "columnas": "",
                "columnas_relevantes": ""
            })
        else:
            for lyr in layers:
                try:
                    gdf = gpd.read_file(path, layer=lyr, rows=5)
                    cols = [str(c).strip() for c in gdf.columns]
                    geom_type = ", ".join(sorted(gdf.geometry.geom_type.astype(str).unique().tolist())) if "geometry" in gdf.columns else ""
                    rel = [c for c in cols if any(k in c.lower() for k in [
                        "cell", "row", "col", "local", "loc", "bogota", "nombre"
                    ])]
                    records.append({
                        "archivo": path.name,
                        "ruta": str(path),
                        "layer": lyr,
                        "geom_type": geom_type,
                        "crs": str(gdf.crs),
                        "n_features_sample": len(gdf),
                        "columnas": ", ".join(cols[:20]),
                        "columnas_relevantes": ", ".join(rel)
                    })
                except Exception as e:
                    records.append({
                        "archivo": path.name,
                        "ruta": str(path),
                        "layer": lyr,
                        "geom_type": "",
                        "crs": "",
                        "n_features_sample": "",
                        "columnas": f"ERROR: {e}",
                        "columnas_relevantes": ""
                    })

    elif path.suffix.lower() in [".shp", ".geojson"]:
        try:
            gdf = gpd.read_file(path, rows=5)
            cols = [str(c).strip() for c in gdf.columns]
            geom_type = ", ".join(sorted(gdf.geometry.geom_type.astype(str).unique().tolist())) if "geometry" in gdf.columns else ""
            rel = [c for c in cols if any(k in c.lower() for k in [
                "cell", "row", "col", "local", "loc", "bogota", "nombre"
            ])]
            records.append({
                "archivo": path.name,
                "ruta": str(path),
                "layer": "(archivo único)",
                "geom_type": geom_type,
                "crs": str(gdf.crs),
                "n_features_sample": len(gdf),
                "columnas": ", ".join(cols[:20]),
                "columnas_relevantes": ", ".join(rel)
            })
        except Exception as e:
            records.append({
                "archivo": path.name,
                "ruta": str(path),
                "layer": "(archivo único)",
                "geom_type": "",
                "crs": "",
                "n_features_sample": "",
                "columnas": f"ERROR: {e}",
                "columnas_relevantes": ""
            })

    return records

# ------------------------------------------------------------
# 3) Recolectar candidatos reales
# ------------------------------------------------------------
real_files = []

for p in candidate_paths:
    if p.is_file():
        real_files.append(p)
    elif p.is_dir():
        real_files.extend(list_vector_files(p))

real_files = sorted(set(real_files))

if not real_files:
    raise FileNotFoundError(
        "No encontré archivos geográficos en las rutas esperadas. "
        "Revisa si las rutas del proyecto coinciden."
    )

# ------------------------------------------------------------
# 4) Inspección
# ------------------------------------------------------------
rows = []
for vf in real_files:
    rows.extend(inspect_vector(vf))

res = pd.DataFrame(rows)

print("=" * 110)
print("INSPECCIÓN DE CAPAS GEOGRÁFICAS CANDIDATAS")
print("=" * 110)
display(res)

# ------------------------------------------------------------
# 5) Sugerencias automáticas
# ------------------------------------------------------------
def score_row(r):
    score = 0
    txt = f"{r['archivo']} {r['layer']} {r['columnas_relevantes']}".lower()
    if "grid" in txt or "grilla" in txt:
        score += 3
    if "cell" in txt:
        score += 3
    if "row" in txt:
        score += 2
    if "col" in txt:
        score += 2
    if "local" in txt or "bogota" in txt or "nombre" in txt:
        score += 2
    if "polygon" in str(r["geom_type"]).lower():
        score += 1
    return score

res["score"] = res.apply(score_row, axis=1)

print("\n" + "=" * 110)
print("CAPAS MÁS PROMETEDORAS")
print("=" * 110)
display(
    res.sort_values(["score", "archivo", "layer"], ascending=[False, True, True])
       [["archivo", "layer", "geom_type", "crs", "columnas_relevantes", "ruta", "score"]]
       .head(20)
)

print("\nRevisa especialmente:")
print("- la capa de grilla que tenga cell_id, Row y Column")
print("- la capa administrativa que tenga Localidad o Nombre")

INSPECCIÓN DE CAPAS GEOGRÁFICAS CANDIDATAS


,archivo,ruta,layer,geom_type,crs,n_features_sample,columnas,columnas_relevantes
0,Loca.shp,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,(archivo único),Polygon,EPSG:4686,5,"LocNombre, LocAAdmini, LocArea, LocCodigo, SHA...","LocNombre, LocAAdmini, LocArea, LocCodigo"
1,Loca_WGS1984.shp,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,(archivo único),Polygon,EPSG:4326,5,"LocNombre, LocAAdmini, LocArea, LocCodigo, SHA...","LocNombre, LocAAdmini, LocArea, LocCodigo, COL..."
2,gadm41_COL_0.shp,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,(archivo único),MultiPolygon,EPSG:4326,1,"GID_0, COUNTRY, geometry",
3,gadm41_COL_1.shp,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,(archivo único),"MultiPolygon, Polygon",EPSG:4326,5,"GID_1, GID_0, COUNTRY, NAME_1, VARNAME_1, NL_N...",
4,gadm41_COL_2.shp,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,(archivo único),Polygon,EPSG:4326,5,"GID_2, GID_0, COUNTRY, GID_1, NAME_1, NL_NAME_...","COL, ROW"
5,grid_3km_time_idw_2020_2024.gpkg,D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,grid_3km_static,MultiPolygon,EPSG:3116,5,"cell_id, LocNombre, geometry","cell_id, LocNombre"
6,grid_3km_time_idw_2020_2024.gpkg,D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,localidades_bogota,Polygon,EPSG:3116,5,"LocNombre, geometry",LocNombre
7,grid_3km_time_idw_2020_2024.gpkg,D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,grid_3km_time_idw,MultiPolygon,EPSG:3116,5,"cell_id, LocNombre, fecha, PM25_idw, NO2_idw, ...","cell_id, LocNombre"
8,prueba.gpkg,D:\TRABAJO DE GRADO BEN-MAP\Qgis\prueba.gpkg,grid_3km_static,MultiPolygon,EPSG:3116,5,"cell_id, LocNombre, geometry","cell_id, LocNombre"
9,prueba.gpkg,D:\TRABAJO DE GRADO BEN-MAP\Qgis\prueba.gpkg,localidades_bogota,Polygon,EPSG:3116,5,"LocNombre, geometry",LocNombre



CAPAS MÁS PROMETEDORAS


,archivo,layer,geom_type,crs,columnas_relevantes,ruta,score
5,grid_3km_time_idw_2020_2024.gpkg,grid_3km_static,MultiPolygon,EPSG:3116,"cell_id, LocNombre",D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,9
7,grid_3km_time_idw_2020_2024.gpkg,grid_3km_time_idw,MultiPolygon,EPSG:3116,"cell_id, LocNombre",D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,9
8,prueba.gpkg,grid_3km_static,MultiPolygon,EPSG:3116,"cell_id, LocNombre",D:\TRABAJO DE GRADO BEN-MAP\Qgis\prueba.gpkg,9
10,prueba.gpkg,grid_3km_time_idw,MultiPolygon,EPSG:3116,"cell_id, LocNombre",D:\TRABAJO DE GRADO BEN-MAP\Qgis\prueba.gpkg,9
1,Loca_WGS1984.shp,(archivo único),Polygon,EPSG:4326,"LocNombre, LocAAdmini, LocArea, LocCodigo, COL...",D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,7
6,grid_3km_time_idw_2020_2024.gpkg,localidades_bogota,Polygon,EPSG:3116,LocNombre,D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time...,6
4,gadm41_COL_2.shp,(archivo único),Polygon,EPSG:4326,"COL, ROW",D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,5
0,Loca.shp,(archivo único),Polygon,EPSG:4686,"LocNombre, LocAAdmini, LocArea, LocCodigo",D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,3
2,gadm41_COL_0.shp,(archivo único),MultiPolygon,EPSG:4326,,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,3
3,gadm41_COL_1.shp,(archivo único),"MultiPolygon, Polygon",EPSG:4326,,D:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MA...,3



Revisa especialmente:
- la capa de grilla que tenga cell_id, Row y Column
- la capa administrativa que tenga Localidad o Nombre


## Paso 19b. Construcción directa de la tabla celda-localidad desde la grilla final

En esta etapa se construye la tabla de correspondencia entre celdas y localidades utilizando directamente la capa geográfica de la grilla final del proyecto. La finalidad es aprovechar que la capa `grid_3km_static` ya contiene tanto el identificador de la celda como el nombre de la localidad, evitando así una unión espacial adicional.

La lógica consiste en leer la capa estática de la grilla, extraer las columnas relevantes para la identificación espacial y estandarizar sus nombres. Luego, esa información se armoniza con la grilla tabular usada en el pipeline para conservar los campos `cell_id`, `Row` y `Column`, junto con la columna administrativa `Localidad`.

El resultado esperado es una tabla final con la relación entre cada celda del análisis y su localidad correspondiente, lista para ser utilizada en la agregación administrativa de resultados sanitarios y económicos.

In [14]:
# ============================================================
# PASO 19b: construir cell_id -> Localidad desde grid_3km_static
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

gpkg_path = project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg"
layer_name = "grid_3km_static"

grid_csv_path = project_root / "SALIDAS_SENSIBILIDAD" / "01_insumos" / "grilla" / "benmap_grid_definition_final_ok_debug.csv"
out_csv = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica" / "cell_to_localidad.csv"
log_json = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion" / "19b_cell_to_localidad_desde_gpkg.json"

out_csv.parent.mkdir(parents=True, exist_ok=True)
log_json.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Leer grilla geográfica y grilla tabular
# ------------------------------------------------------------
if not gpkg_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo GPKG:\n{gpkg_path}")

if not grid_csv_path.exists():
    raise FileNotFoundError(f"No se encontró la grilla tabular:\n{grid_csv_path}")

gdf = gpd.read_file(gpkg_path, layer=layer_name)
gdf.columns = [str(c).strip() for c in gdf.columns]

grid_df = pd.read_csv(grid_csv_path, encoding="utf-8-sig")
grid_df.columns = [str(c).replace("\ufeff", "").strip() for c in grid_df.columns]

# ------------------------------------------------------------
# 3) Validar columnas mínimas
# ------------------------------------------------------------
required_geo = {"cell_id", "LocNombre"}
missing_geo = required_geo - set(gdf.columns)
if missing_geo:
    raise ValueError(f"Faltan columnas en la capa geográfica: {sorted(missing_geo)}")

rename_grid = {}
for c in grid_df.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl in ["col", "column"]:
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

grid_df = grid_df.rename(columns=rename_grid)

required_tab = {"cell_id", "Row", "Column"}
missing_tab = required_tab - set(grid_df.columns)
if missing_tab:
    raise ValueError(f"Faltan columnas en la grilla tabular: {sorted(missing_tab)}")

# ------------------------------------------------------------
# 4) Construir correspondencia
# ------------------------------------------------------------
gdf["cell_id"] = pd.to_numeric(gdf["cell_id"], errors="coerce")
grid_df["cell_id"] = pd.to_numeric(grid_df["cell_id"], errors="coerce")
grid_df["Row"] = pd.to_numeric(grid_df["Row"], errors="coerce")
grid_df["Column"] = pd.to_numeric(grid_df["Column"], errors="coerce")

mapping = (
    gdf[["cell_id", "LocNombre"]]
    .dropna(subset=["cell_id"])
    .copy()
)

mapping["cell_id"] = mapping["cell_id"].astype(int)
mapping["Localidad"] = mapping["LocNombre"].astype(str).str.strip()
mapping = mapping.drop(columns=["LocNombre"]).drop_duplicates(subset=["cell_id"])

mapping = (
    grid_df[["cell_id", "Row", "Column"]]
    .dropna(subset=["cell_id", "Row", "Column"])
    .copy()
)
mapping["cell_id"] = mapping["cell_id"].astype(int)
mapping["Row"] = mapping["Row"].astype(int)
mapping["Column"] = mapping["Column"].astype(int)

mapping = (
    mapping.merge(
        gdf[["cell_id", "LocNombre"]].dropna(subset=["cell_id"]).assign(
            cell_id=lambda d: pd.to_numeric(d["cell_id"], errors="coerce").astype("Int64"),
            Localidad=lambda d: d["LocNombre"].astype(str).str.strip()
        )[["cell_id", "Localidad"]].drop_duplicates(subset=["cell_id"]),
        on="cell_id",
        how="left"
    )
    .sort_values("cell_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5) Resumen y validación
# ------------------------------------------------------------
summary = {
    "n_celdas_total": int(len(mapping)),
    "n_celdas_con_localidad": int(mapping["Localidad"].notna().sum()),
    "n_celdas_sin_localidad": int(mapping["Localidad"].isna().sum()),
    "n_localidades": int(mapping["Localidad"].dropna().nunique())
}

print("=" * 90)
print("TABLA CELDA -> LOCALIDAD CONSTRUIDA DESDE grid_3km_static")
print("=" * 90)
print(f"Archivo fuente : {gpkg_path}")
print(f"Capa usada     : {layer_name}")
print("\nResumen:")
for k, v in summary.items():
    print(f"- {k}: {v}")

print("\nVista previa:")
display(mapping.head(20))

print("\nLocalidades detectadas:")
display(pd.DataFrame({"Localidad": sorted(mapping["Localidad"].dropna().unique().tolist())}))

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
mapping.to_csv(out_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "source_file": str(gpkg_path),
    "source_layer": layer_name,
    "summary": summary,
    "output_csv": str(out_csv)
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", out_csv)
print("-", log_json)

TABLA CELDA -> LOCALIDAD CONSTRUIDA DESDE grid_3km_static
Archivo fuente : D:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time_idw_2020_2024.gpkg
Capa usada     : grid_3km_static

Resumen:
- n_celdas_total: 254
- n_celdas_con_localidad: 254
- n_celdas_sin_localidad: 0
- n_localidades: 19

Vista previa:


,cell_id,Row,Column,Localidad
0,0,1,1,SUMAPAZ
1,1,2,1,SUMAPAZ
2,2,3,1,None
3,41,1,2,SUMAPAZ
4,42,2,2,SUMAPAZ
5,43,3,2,SUMAPAZ
6,44,4,2,SUMAPAZ
7,82,1,3,SUMAPAZ
8,83,2,3,SUMAPAZ
9,84,3,3,SUMAPAZ



Localidades detectadas:


,Localidad
0,BARRIOS UNIDOS
1,BOSA
2,CHAPINERO
3,CIUDAD BOLIVAR
4,ENGATIVA
5,FONTIBON
6,KENNEDY
7,LOS MARTIRES
8,None
9,PUENTE ARANDA



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\cell_to_localidad.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\19b_cell_to_localidad_desde_gpkg.json


## Paso 19c. Corrección de celdas con localidad faltante

En esta etapa se corrige la tabla de correspondencia entre celdas y localidades para identificar y reparar registros cuyo nombre de localidad haya quedado vacío o representado como texto no válido. La finalidad es asegurar que todas las celdas de la grilla queden asociadas correctamente a una unidad administrativa antes de realizar la agregación de resultados.

La lógica consiste en tomar la tabla preliminar de correspondencia, detectar valores faltantes o cadenas como `None`, y completar esos casos mediante una unión espacial entre la geometría de la celda y la capa de localidades de Bogotá. Con esto se evita trasladar errores administrativos a la etapa de agregación sanitaria y económica.

El resultado esperado es una tabla definitiva de relación `cell_id`–`Localidad`, sin celdas huérfanas y lista para usar en la agregación de resultados por localidad.

In [15]:
# ============================================================
# PASO 19c: corregir celdas con Localidad faltante o "None"
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

gpkg_path = project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg"
grid_layer = "grid_3km_static"
loc_layer = "localidades_bogota"

mapping_path = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica" / "cell_to_localidad.csv"
out_csv = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica" / "cell_to_localidad.csv"
log_json = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion" / "19c_cell_to_localidad_corregido.json"

if not gpkg_path.exists():
    raise FileNotFoundError(f"No se encontró:\n{gpkg_path}")
if not mapping_path.exists():
    raise FileNotFoundError(f"No se encontró:\n{mapping_path}")

# ------------------------------------------------------------
# 2) Cargar datos
# ------------------------------------------------------------
mapping = pd.read_csv(mapping_path, encoding="utf-8-sig")
mapping.columns = [str(c).replace("\ufeff", "").strip() for c in mapping.columns]

grid_gdf = gpd.read_file(gpkg_path, layer=grid_layer)
loc_gdf = gpd.read_file(gpkg_path, layer=loc_layer)

grid_gdf.columns = [str(c).strip() for c in grid_gdf.columns]
loc_gdf.columns = [str(c).strip() for c in loc_gdf.columns]

required_map = {"cell_id", "Row", "Column", "Localidad"}
missing_map = required_map - set(mapping.columns)
if missing_map:
    raise ValueError(f"Faltan columnas en mapping: {sorted(missing_map)}")

required_grid = {"cell_id", "geometry"}
missing_grid = required_grid - set(grid_gdf.columns)
if missing_grid:
    raise ValueError(f"Faltan columnas en grid_gdf: {sorted(missing_grid)}")

if "LocNombre" not in loc_gdf.columns:
    raise ValueError("La capa localidades_bogota no tiene la columna 'LocNombre'.")

# ------------------------------------------------------------
# 3) Normalizar y detectar faltantes reales
# ------------------------------------------------------------
mapping["cell_id"] = pd.to_numeric(mapping["cell_id"], errors="coerce")
mapping["Row"] = pd.to_numeric(mapping["Row"], errors="coerce")
mapping["Column"] = pd.to_numeric(mapping["Column"], errors="coerce")
mapping["Localidad"] = mapping["Localidad"].astype(str).str.strip()

invalid_tokens = {"", "NONE", "NAN", "NULL"}

mask_invalid = (
    mapping["Localidad"].str.upper().isin(invalid_tokens)
)

before_missing = int(mask_invalid.sum())

print("=" * 90)
print("CORRECCIÓN DE LOCALIDADES FALTANTES")
print("=" * 90)
print(f"Celdas con localidad inválida antes de corregir: {before_missing}")

# ------------------------------------------------------------
# 4) Reparar mediante unión espacial
# ------------------------------------------------------------
if before_missing > 0:
    grid_gdf["cell_id"] = pd.to_numeric(grid_gdf["cell_id"], errors="coerce")
    grid_fix = grid_gdf[grid_gdf["cell_id"].isin(mapping.loc[mask_invalid, "cell_id"])].copy()

    # CRS
    if grid_fix.crs != loc_gdf.crs:
        loc_gdf = loc_gdf.to_crs(grid_fix.crs)

    # usar centroides si la geometría es polígono
    if grid_fix.geometry.geom_type.astype(str).str.contains("Polygon", case=False).any():
        grid_fix = grid_fix.copy()
        grid_fix["geometry"] = grid_fix.geometry.centroid

    joined = gpd.sjoin(
        grid_fix[["cell_id", "geometry"]],
        loc_gdf[["LocNombre", "geometry"]],
        how="left",
        predicate="intersects"
    )

    repair = (
        joined[["cell_id", "LocNombre"]]
        .drop_duplicates(subset=["cell_id"])
        .rename(columns={"LocNombre": "Localidad_new"})
    )

    mapping = mapping.merge(repair, on="cell_id", how="left")
    mapping["Localidad"] = mapping["Localidad"].mask(
        mask_invalid,
        mapping["Localidad_new"]
    )
    mapping = mapping.drop(columns=["Localidad_new"])

# ------------------------------------------------------------
# 5) Limpieza final
# ------------------------------------------------------------
mapping["Localidad"] = mapping["Localidad"].astype(str).str.strip()
mask_invalid_after = mapping["Localidad"].str.upper().isin(invalid_tokens)

after_missing = int(mask_invalid_after.sum())
n_localidades = int(mapping.loc[~mask_invalid_after, "Localidad"].nunique())

print(f"Celdas con localidad inválida después de corregir: {after_missing}")
print(f"Número de localidades válidas detectadas: {n_localidades}")

print("\nVista previa:")
display(mapping.head(20))

print("\nLocalidades válidas:")
display(
    pd.DataFrame(
        {"Localidad": sorted(mapping.loc[~mask_invalid_after, "Localidad"].unique().tolist())}
    )
)

# ------------------------------------------------------------
# 6) Guardar
# ------------------------------------------------------------
mapping.to_csv(out_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "before_missing": before_missing,
    "after_missing": after_missing,
    "n_localidades_validas": n_localidades,
    "output_csv": str(out_csv)
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", out_csv)
print("-", log_json)

CORRECCIÓN DE LOCALIDADES FALTANTES
Celdas con localidad inválida antes de corregir: 8
Celdas con localidad inválida después de corregir: 8
Número de localidades válidas detectadas: 18

Vista previa:


,cell_id,Row,Column,Localidad
0,0,1,1,SUMAPAZ
1,1,2,1,SUMAPAZ
2,2,3,1,nan
3,41,1,2,SUMAPAZ
4,42,2,2,SUMAPAZ
5,43,3,2,SUMAPAZ
6,44,4,2,SUMAPAZ
7,82,1,3,SUMAPAZ
8,83,2,3,SUMAPAZ
9,84,3,3,SUMAPAZ



Localidades válidas:


,Localidad
0,BARRIOS UNIDOS
1,BOSA
2,CHAPINERO
3,CIUDAD BOLIVAR
4,ENGATIVA
5,FONTIBON
6,KENNEDY
7,LOS MARTIRES
8,PUENTE ARANDA
9,RAFAEL URIBE URIBE



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\cell_to_localidad.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\19c_cell_to_localidad_corregido.json


## Paso 19d. Asignación definitiva de localidad por máxima intersección espacial

En esta etapa se corrigen de forma definitiva las celdas que aún no tienen localidad asignada. La finalidad es evitar que queden celdas sin unidad administrativa antes de agregar los resultados sanitarios y económicos a nivel de localidad.

La lógica del procedimiento consiste en trabajar directamente con los polígonos de la grilla y de las localidades. Para cada celda faltante se calcula la intersección espacial con las localidades de Bogotá y se mide el área de traslape. Luego se asigna la localidad que tenga la mayor área de intersección, ya que esta representa la unidad administrativa dominante dentro de la celda.

Como salvaguarda adicional, si alguna celda siguiera sin localidad después de este proceso, se utiliza una asignación por vecindad espacial cercana. El resultado esperado es una tabla `cell_id`–`Localidad` completamente cerrada y lista para la agregación administrativa de resultados.## Paso 19d. Asignación definitiva de localidad por máxima intersección espacial

En esta etapa se corrigen de forma definitiva las celdas que aún no tienen localidad asignada. La finalidad es evitar que queden celdas sin unidad administrativa antes de agregar los resultados sanitarios y económicos a nivel de localidad.

La lógica del procedimiento consiste en trabajar directamente con los polígonos de la grilla y de las localidades. Para cada celda faltante se calcula la intersección espacial con las localidades de Bogotá y se mide el área de traslape. Luego se asigna la localidad que tenga la mayor área de intersección, ya que esta representa la unidad administrativa dominante dentro de la celda.

Como salvaguarda adicional, si alguna celda siguiera sin localidad después de este proceso, se utiliza una asignación por vecindad espacial cercana. El resultado esperado es una tabla `cell_id`–`Localidad` completamente cerrada y lista para la agregación administrativa de resultados.

In [16]:
# ============================================================
# PASO 19d: corregir celdas faltantes por máxima intersección
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

gpkg_path = project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg"
grid_layer = "grid_3km_static"
loc_layer = "localidades_bogota"

mapping_path = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica" / "cell_to_localidad.csv"
out_csv = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica" / "cell_to_localidad.csv"
log_json = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion" / "19d_cell_to_localidad_max_interseccion.json"

if not gpkg_path.exists():
    raise FileNotFoundError(f"No se encontró:\n{gpkg_path}")
if not mapping_path.exists():
    raise FileNotFoundError(f"No se encontró:\n{mapping_path}")

# ------------------------------------------------------------
# 2) Cargar datos
# ------------------------------------------------------------
mapping = pd.read_csv(mapping_path, encoding="utf-8-sig")
mapping.columns = [str(c).replace("\ufeff", "").strip() for c in mapping.columns]

grid_gdf = gpd.read_file(gpkg_path, layer=grid_layer)
loc_gdf = gpd.read_file(gpkg_path, layer=loc_layer)

grid_gdf.columns = [str(c).strip() for c in grid_gdf.columns]
loc_gdf.columns = [str(c).strip() for c in loc_gdf.columns]

if "cell_id" not in grid_gdf.columns:
    raise ValueError("La capa de grilla no tiene 'cell_id'.")
if "LocNombre" not in loc_gdf.columns:
    raise ValueError("La capa de localidades no tiene 'LocNombre'.")

# ------------------------------------------------------------
# 3) Normalizar faltantes
# ------------------------------------------------------------
mapping["cell_id"] = pd.to_numeric(mapping["cell_id"], errors="coerce")
mapping["Row"] = pd.to_numeric(mapping["Row"], errors="coerce")
mapping["Column"] = pd.to_numeric(mapping["Column"], errors="coerce")

mapping["Localidad"] = mapping["Localidad"].astype(str).str.strip()
mapping["Localidad"] = mapping["Localidad"].replace({
    "None": pd.NA,
    "NONE": pd.NA,
    "nan": pd.NA,
    "NaN": pd.NA,
    "NAN": pd.NA,
    "": pd.NA
})

before_missing = int(mapping["Localidad"].isna().sum())

print("=" * 90)
print("ASIGNACIÓN DEFINITIVA DE LOCALIDADES POR MÁXIMA INTERSECCIÓN")
print("=" * 90)
print(f"Celdas faltantes antes de corregir: {before_missing}")

# ------------------------------------------------------------
# 4) Preparar geometrías faltantes
# ------------------------------------------------------------
grid_gdf["cell_id"] = pd.to_numeric(grid_gdf["cell_id"], errors="coerce")
grid_gdf = grid_gdf.dropna(subset=["cell_id"]).copy()
grid_gdf["cell_id"] = grid_gdf["cell_id"].astype(int)

missing_ids = mapping.loc[mapping["Localidad"].isna(), "cell_id"].dropna().astype(int).tolist()
grid_missing = grid_gdf[grid_gdf["cell_id"].isin(missing_ids)].copy()

if grid_missing.empty:
    print("No hay celdas faltantes para corregir.")
else:
    # CRS consistente
    if grid_missing.crs != loc_gdf.crs:
        loc_gdf = loc_gdf.to_crs(grid_missing.crs)

    # --------------------------------------------------------
    # 5) Overlay polígono-polígono
    # --------------------------------------------------------
    grid_missing = grid_missing[["cell_id", "geometry"]].copy()
    loc_base = loc_gdf[["LocNombre", "geometry"]].copy()

    inter = gpd.overlay(grid_missing, loc_base, how="intersection")

    if not inter.empty:
        inter["inter_area"] = inter.geometry.area

        best_match = (
            inter.sort_values(["cell_id", "inter_area"], ascending=[True, False])
                 .drop_duplicates(subset=["cell_id"])
                 .rename(columns={"LocNombre": "Localidad_new"})
        )[
            ["cell_id", "Localidad_new", "inter_area"]
        ]
    else:
        best_match = pd.DataFrame(columns=["cell_id", "Localidad_new", "inter_area"])

    # --------------------------------------------------------
    # 6) Fallback con nearest para las que sigan vacías
    # --------------------------------------------------------
    unresolved_ids = sorted(set(missing_ids) - set(best_match["cell_id"].astype(int).tolist()))

    nearest_match = pd.DataFrame(columns=["cell_id", "Localidad_new"])
    if unresolved_ids:
        grid_unresolved = grid_missing[grid_missing["cell_id"].isin(unresolved_ids)].copy()

        # usar centroides para nearest
        grid_unresolved = grid_unresolved.copy()
        grid_unresolved["geometry"] = grid_unresolved.geometry.centroid

        near = gpd.sjoin_nearest(
            grid_unresolved,
            loc_base,
            how="left",
            distance_col="dist_to_loc"
        )

        nearest_match = (
            near.rename(columns={"LocNombre": "Localidad_new"})
                [["cell_id", "Localidad_new", "dist_to_loc"]]
                .drop_duplicates(subset=["cell_id"])
        )

    # combinar mejores asignaciones
    repair = pd.concat(
        [
            best_match[["cell_id", "Localidad_new"]],
            nearest_match[["cell_id", "Localidad_new"]],
        ],
        ignore_index=True
    ).drop_duplicates(subset=["cell_id"], keep="first")

    # aplicar reparación
    mapping = mapping.merge(repair, on="cell_id", how="left")
    mapping["Localidad"] = mapping["Localidad"].fillna(mapping["Localidad_new"])
    mapping = mapping.drop(columns=["Localidad_new"])

# ------------------------------------------------------------
# 7) Limpieza final
# ------------------------------------------------------------
mapping["Localidad"] = mapping["Localidad"].astype(str).str.strip()
mapping["Localidad"] = mapping["Localidad"].replace({
    "None": pd.NA,
    "NONE": pd.NA,
    "nan": pd.NA,
    "NaN": pd.NA,
    "NAN": pd.NA,
    "": pd.NA
})

after_missing = int(mapping["Localidad"].isna().sum())
n_localidades = int(mapping["Localidad"].dropna().nunique())

print(f"Celdas faltantes después de corregir: {after_missing}")
print(f"Número de localidades válidas detectadas: {n_localidades}")

print("\nVista previa:")
display(mapping.head(20))

print("\nConteo por localidad:")
display(
    mapping.groupby("Localidad", dropna=False)
           .size()
           .reset_index(name="n_celdas")
           .sort_values("n_celdas", ascending=False)
           .reset_index(drop=True)
)

# ------------------------------------------------------------
# 8) Guardar
# ------------------------------------------------------------
mapping.to_csv(out_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "before_missing": before_missing,
    "after_missing": after_missing,
    "n_localidades_validas": n_localidades,
    "output_csv": str(out_csv)
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos actualizados:")
print("-", out_csv)
print("-", log_json)

ASIGNACIÓN DEFINITIVA DE LOCALIDADES POR MÁXIMA INTERSECCIÓN
Celdas faltantes antes de corregir: 8
Celdas faltantes después de corregir: 0
Número de localidades válidas detectadas: 18

Vista previa:


,cell_id,Row,Column,Localidad
0,0,1,1,SUMAPAZ
1,1,2,1,SUMAPAZ
2,2,3,1,SUMAPAZ
3,41,1,2,SUMAPAZ
4,42,2,2,SUMAPAZ
5,43,3,2,SUMAPAZ
6,44,4,2,SUMAPAZ
7,82,1,3,SUMAPAZ
8,83,2,3,SUMAPAZ
9,84,3,3,SUMAPAZ



Conteo por localidad:


,Localidad,n_celdas
0,SUMAPAZ,122
1,USME,34
2,CIUDAD BOLIVAR,21
3,SUBA,15
4,USAQUEN,13
5,SAN CRISTOBAL,8
6,SANTA FE,7
7,CHAPINERO,7
8,FONTIBON,5
9,BOSA,5



Archivos actualizados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\02_base_analitica\cell_to_localidad.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\19d_cell_to_localidad_max_interseccion.json


## Paso 20. Agregación de la corrida base por localidad

En esta etapa se agregan los resultados de la corrida base a nivel de localidad, con el fin de traducir los impactos calculados por celda a una escala administrativa más útil para la interpretación territorial del estudio. La finalidad es obtener, para cada contaminante y endpoint, el total de casos evitados y el valor económico asociado dentro de cada localidad de Bogotá efectivamente cubierta por la grilla.

La lógica consiste en unir la tabla de correspondencia entre celdas y localidades con los resultados sanitarios y económicos ya calculados en la corrida base. Después, los impactos se agregan por localidad y por corrida, sumando los casos evitados y los valores monetarios. De forma complementaria, también se calcula la participación porcentual de cada localidad dentro del total de la corrida, para facilitar la lectura comparativa.

El resultado esperado es una tabla detallada por localidad y una tabla resumen ordenada para cada corrida base, listas para usarse en tablas del documento, gráficos comparativos y mapas temáticos posteriores.

In [19]:
# ============================================================
# PASO 20: agregar corrida base por localidad (CORREGIDO)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

base_root = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica"
main_root = project_root / "SALIDAS_SENSIBILIDAD" / "03_corrida_principal"
comp_root = project_root / "SALIDAS_SENSIBILIDAD" / "05_complementarios" / "localidades"
tables_root = project_root / "SALIDAS_SENSIBILIDAD" / "07_tablas"
logs_root = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion"

mapping_path = base_root / "cell_to_localidad.csv"
health_path = main_root / "impactos_sanitarios_por_celda.csv"
econ_path = main_root / "valoracion_economica_por_celda_cop.csv"

for folder in [comp_root, tables_root, logs_root]:
    folder.mkdir(parents=True, exist_ok=True)

for p in [mapping_path, health_path, econ_path]:
    if not p.exists():
        raise FileNotFoundError(f"No se encontró el archivo:\n{p}")

# ------------------------------------------------------------
# 2) Cargar archivos
# ------------------------------------------------------------
mapping = pd.read_csv(mapping_path, encoding="utf-8-sig")
health = pd.read_csv(health_path, encoding="utf-8-sig")
econ = pd.read_csv(econ_path, encoding="utf-8-sig")

for df in [mapping, health, econ]:
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

# ------------------------------------------------------------
# 3) Normalizar columnas
# ------------------------------------------------------------
mapping["cell_id"] = pd.to_numeric(mapping["cell_id"], errors="coerce")
mapping["Row"] = pd.to_numeric(mapping["Row"], errors="coerce")
mapping["Column"] = pd.to_numeric(mapping["Column"], errors="coerce")
mapping["Localidad"] = mapping["Localidad"].astype(str).str.strip()

health["cell_id"] = pd.to_numeric(health["cell_id"], errors="coerce")
health["avoided_cases"] = pd.to_numeric(health["avoided_cases"], errors="coerce")
health["baseline_cases"] = pd.to_numeric(health["baseline_cases"], errors="coerce")
health["population_target"] = pd.to_numeric(health["population_target"], errors="coerce")
health["incidence_target"] = pd.to_numeric(health["incidence_target"], errors="coerce")

econ["cell_id"] = pd.to_numeric(econ["cell_id"], errors="coerce")
econ["economic_value_usd_2015"] = pd.to_numeric(econ["economic_value_usd_2015"], errors="coerce")
econ["economic_value_cop_2015"] = pd.to_numeric(econ["economic_value_cop_2015"], errors="coerce")

# ------------------------------------------------------------
# 4) Verificar columnas disponibles
# ------------------------------------------------------------
required_health = [
    "run_id", "pollutant", "endpoint_group", "endpoint",
    "economic_method", "cell_id",
    "population_target", "baseline_cases", "avoided_cases"
]
missing_health = [c for c in required_health if c not in health.columns]
if missing_health:
    raise ValueError(f"Faltan columnas en health: {missing_health}")

required_econ = [
    "run_id", "cell_id",
    "valuation_parameter_name", "valuation_value_used_usd_2015",
    "economic_value_usd_2015", "economic_value_cop_2015"
]
missing_econ = [c for c in required_econ if c not in econ.columns]
if missing_econ:
    raise ValueError(f"Faltan columnas en econ: {missing_econ}")

# ------------------------------------------------------------
# 5) Unir sanitario + económico + localidad
#    NOTA: economic_method ya viene en health, no lo traemos de econ
# ------------------------------------------------------------
base_cells = (
    health.merge(
        econ[[
            "run_id", "cell_id",
            "valuation_parameter_name",
            "valuation_value_used_usd_2015",
            "economic_value_usd_2015",
            "economic_value_cop_2015"
        ]],
        on=["run_id", "cell_id"],
        how="left"
    )
    .merge(
        mapping[["cell_id", "Row", "Column", "Localidad"]],
        on="cell_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 6) Validaciones rápidas
# ------------------------------------------------------------
missing_localidad = int(base_cells["Localidad"].isna().sum())
if missing_localidad > 0:
    raise ValueError(f"Quedaron {missing_localidad} filas sin localidad al unir resultados base.")

missing_economic = int(base_cells["economic_value_cop_2015"].isna().sum())
if missing_economic > 0:
    raise ValueError(f"Quedaron {missing_economic} filas sin valoración económica al unir resultados base.")

# ------------------------------------------------------------
# 7) Agregación por localidad
# ------------------------------------------------------------
agg_localidad = (
    base_cells.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method", "Localidad"],
        as_index=False
    )
    .agg(
        n_celdas=("cell_id", "nunique"),
        population_total=("population_target", "sum"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 8) Participación dentro de cada corrida
# ------------------------------------------------------------
totals = (
    agg_localidad.groupby("run_id", as_index=False)
    .agg(
        total_cases_run=("avoided_cases_total", "sum"),
        total_cop_run=("economic_value_total_cop_2015", "sum")
    )
)

agg_localidad = agg_localidad.merge(totals, on="run_id", how="left")

agg_localidad["participacion_casos_pct"] = np.where(
    agg_localidad["total_cases_run"].abs() > 0,
    100 * agg_localidad["avoided_cases_total"] / agg_localidad["total_cases_run"],
    np.nan
)

agg_localidad["participacion_cop_pct"] = np.where(
    agg_localidad["total_cop_run"].abs() > 0,
    100 * agg_localidad["economic_value_total_cop_2015"] / agg_localidad["total_cop_run"],
    np.nan
)

# ------------------------------------------------------------
# 9) Ordenar y resumen corto
# ------------------------------------------------------------
agg_localidad = agg_localidad.sort_values(
    ["run_id", "economic_value_total_cop_2015"],
    ascending=[True, False]
).reset_index(drop=True)

resumen_top = (
    agg_localidad[[
        "run_id", "pollutant", "endpoint", "Localidad",
        "avoided_cases_total", "economic_value_total_cop_2015",
        "participacion_casos_pct", "participacion_cop_pct"
    ]]
    .copy()
)

for col in [
    "avoided_cases_total", "economic_value_total_cop_2015",
    "participacion_casos_pct", "participacion_cop_pct"
]:
    resumen_top[col] = pd.to_numeric(resumen_top[col], errors="coerce")

# ------------------------------------------------------------
# 10) Mostrar resultados
# ------------------------------------------------------------
print("=" * 90)
print("CORRIDA BASE AGREGADA POR LOCALIDAD")
print("=" * 90)

print("\nResumen por corrida:")
display(
    agg_localidad.groupby("run_id", as_index=False).agg(
        n_localidades=("Localidad", "nunique"),
        avoided_cases_total=("avoided_cases_total", "sum"),
        economic_value_total_cop_2015=("economic_value_total_cop_2015", "sum")
    )
)

print("\nVista previa detallada:")
display(agg_localidad.head(20))

print("\nTop localidades por valor económico dentro de cada corrida:")
display(
    resumen_top.groupby("run_id", group_keys=False)
              .head(5)
              .reset_index(drop=True)
)

# ------------------------------------------------------------
# 11) Exportar
# ------------------------------------------------------------
detail_csv = comp_root / "corrida_base_por_localidad_detalle.csv"
top_csv = tables_root / "corrida_base_por_localidad_top.csv"
log_json = logs_root / "20_corrida_base_por_localidad.json"

agg_localidad.to_csv(detail_csv, index=False, encoding="utf-8-sig")
resumen_top.to_csv(top_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "detail_shape": list(agg_localidad.shape),
    "top_shape": list(resumen_top.shape),
    "outputs": {
        "detail_csv": str(detail_csv),
        "top_csv": str(top_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", detail_csv)
print("-", top_csv)
print("-", log_json)

CORRIDA BASE AGREGADA POR LOCALIDAD

Resumen por corrida:


,run_id,n_localidades,avoided_cases_total,economic_value_total_cop_2015
0,NO2_ASTHMA,18,12.713526,3.960412e+08
1,NO2_CLD,18,9.454138,4.031385e+08
2,O3_MAIN,18,2.204553,5.322457e+10
3,PM25_MAIN,18,1.299799,3.138108e+10



Vista previa detallada:


,run_id,pollutant,endpoint_group,endpoint,economic_method,Localidad,n_celdas,population_total,baseline_cases_total,avoided_cases_total,economic_value_total_usd_2015,economic_value_total_cop_2015,total_cases_run,total_cop_run,participacion_casos_pct,participacion_cop_pct
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,SUMAPAZ,122,7684939,1281.345999,6.062825,68097.646879,1.888641e+08,12.713526,3.960412e+08,47.687987,47.687987
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,USME,34,2143977,357.475362,1.697184,19062.773437,5.286927e+07,12.713526,3.960412e+08,13.349438,13.349438
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,CIUDAD BOLIVAR,21,1258083,209.766092,0.993115,11154.666007,3.093669e+07,12.713526,3.960412e+08,7.811482,7.811482
3,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,SUBA,15,986037,164.406583,0.799094,8975.423584,2.489271e+07,12.713526,3.960412e+08,6.285384,6.285384
4,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,USAQUEN,13,616487,102.789775,0.448484,5037.369708,1.397079e+07,12.713526,3.960412e+08,3.527611,3.527611
5,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,SAN CRISTOBAL,8,540841,90.176962,0.411515,4622.137572,1.281918e+07,12.713526,3.960412e+08,3.236829,3.236829
6,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,SANTA FE,7,499559,83.293820,0.397137,4460.644219,1.237128e+07,12.713526,3.960412e+08,3.123737,3.123737
7,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,FONTIBON,5,368111,61.376877,0.315904,3548.234070,9.840779e+06,12.713526,3.960412e+08,2.484787,2.484787
8,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,KENNEDY,4,353434,58.929712,0.311293,3496.437966,9.697126e+06,12.713526,3.960412e+08,2.448515,2.448515
9,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,ENGATIVA,5,340054,56.698802,0.273963,3077.156621,8.534278e+06,12.713526,3.960412e+08,2.154897,2.154897



Top localidades por valor económico dentro de cada corrida:


,run_id,pollutant,endpoint,Localidad,avoided_cases_total,economic_value_total_cop_2015,participacion_casos_pct,participacion_cop_pct
0,NO2_ASTHMA,NO2,Asthma,SUMAPAZ,6.062825,1.888641e+08,47.687987,47.687987
1,NO2_ASTHMA,NO2,Asthma,USME,1.697184,5.286927e+07,13.349438,13.349438
2,NO2_ASTHMA,NO2,Asthma,CIUDAD BOLIVAR,0.993115,3.093669e+07,7.811482,7.811482
3,NO2_ASTHMA,NO2,Asthma,SUBA,0.799094,2.489271e+07,6.285384,6.285384
4,NO2_ASTHMA,NO2,Asthma,USAQUEN,0.448484,1.397079e+07,3.527611,3.527611
5,NO2_CLD,NO2,Chronic Lung Disease,SUMAPAZ,4.508445,1.922468e+08,47.687533,47.687533
6,NO2_CLD,NO2,Chronic Lung Disease,USME,1.262067,5.381642e+07,13.349364,13.349364
7,NO2_CLD,NO2,Chronic Lung Disease,CIUDAD BOLIVAR,0.738503,3.149087e+07,7.811428,7.811428
8,NO2_CLD,NO2,Chronic Lung Disease,SUBA,0.594247,2.533959e+07,6.285580,6.285580
9,NO2_CLD,NO2,Chronic Lung Disease,USAQUEN,0.333480,1.422009e+07,3.527347,3.527347



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\05_complementarios\localidades\corrida_base_por_localidad_detalle.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\07_tablas\corrida_base_por_localidad_top.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\20_corrida_base_por_localidad.json


## Paso 21a. Inventario de funciones concentración-respuesta disponibles

En esta etapa se realiza un inventario de las funciones concentración-respuesta actualmente disponibles en los archivos del proyecto. La finalidad es identificar cuáles funciones sanitarias pueden utilizarse para construir una sensibilidad explícita por función C–R, sin introducir supuestos arbitrarios ni funciones no trazables.

La lógica consiste en revisar los archivos de funciones de impacto almacenados en la carpeta de insumos de BenMAP, detectar las columnas relevantes asociadas con contaminante, endpoint, estudio, beta y demás metadatos, y consolidar esa información en una tabla resumen. Con esto será posible reconocer si existen funciones alternativas para un mismo contaminante y endpoint, por ejemplo una versión central y otra de referencia distinta, que puedan compararse como parte del análisis de sensibilidad.

El resultado esperado es una tabla de inventario de funciones C–R disponibles, junto con una vista preliminar de las filas relevantes para PM2.5, O3 y NO2. Esta salida servirá como base para definir los escenarios de sensibilidad por función C–R en el siguiente paso.

In [20]:
# ============================================================
# PASO 21a: inventario de funciones C-R disponibles
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")
benmap_root = project_root / "SALIDAS_SENSIBILIDAD" / "01_insumos" / "benmap"
out_root = project_root / "SALIDAS_SENSIBILIDAD" / "04_sensibilidad" / "funcion_cr"
logs_root = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion"

out_root.mkdir(parents=True, exist_ok=True)
logs_root.mkdir(parents=True, exist_ok=True)

if not benmap_root.exists():
    raise FileNotFoundError(f"No se encontró la carpeta:\n{benmap_root}")

# ------------------------------------------------------------
# 2) Utilidades
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

def normalize_colname(c: str) -> str:
    return str(c).strip().lower().replace(" ", "_").replace("-", "_")

def detect_col(df: pd.DataFrame, options):
    norm_map = {normalize_colname(c): c for c in df.columns}
    for opt in options:
        if opt in norm_map:
            return norm_map[opt]
    return None

# ------------------------------------------------------------
# 3) Buscar archivos candidatos
# ------------------------------------------------------------
candidate_files = sorted([
    p for p in benmap_root.rglob("*.csv")
    if any(k in p.name.lower() for k in ["health", "impact", "function", "cr", "endpoint"])
])

if not candidate_files:
    raise FileNotFoundError(
        "No se encontraron CSV candidatos de funciones C-R en:\n"
        f"{benmap_root}"
    )

inventory_rows = []
preview_rows = []

# ------------------------------------------------------------
# 4) Revisar cada archivo
# ------------------------------------------------------------
for path in candidate_files:
    try:
        df = read_csv_safe(path)
    except Exception as e:
        inventory_rows.append({
            "archivo": path.name,
            "ruta": str(path),
            "n_filas": None,
            "n_columnas": None,
            "col_contaminante": None,
            "col_endpoint": None,
            "col_estudio": None,
            "col_beta": None,
            "col_beta_se": None,
            "estado": f"error_lectura: {e}"
        })
        continue

    col_pollutant = detect_col(df, [
        "pollutant", "pollutant_name", "pollutante", "contaminant"
    ])
    col_endpoint = detect_col(df, [
        "endpoint", "health_endpoint", "endpoint_name"
    ])
    col_endpoint_group = detect_col(df, [
        "endpoint_group", "group", "health_group"
    ])
    col_study = detect_col(df, [
        "study", "study_name", "author", "authors", "reference", "citation"
    ])
    col_beta = detect_col(df, [
        "beta", "coef", "coefficient", "effect_estimate"
    ])
    col_beta_se = detect_col(df, [
        "beta_se", "se", "std_error", "standard_error"
    ])
    col_metric = detect_col(df, [
        "metric", "aq_metric", "measure"
    ])

    inventory_rows.append({
        "archivo": path.name,
        "ruta": str(path),
        "n_filas": int(len(df)),
        "n_columnas": int(len(df.columns)),
        "col_contaminante": col_pollutant,
        "col_endpoint": col_endpoint,
        "col_endpoint_group": col_endpoint_group,
        "col_estudio": col_study,
        "col_beta": col_beta,
        "col_beta_se": col_beta_se,
        "col_metric": col_metric,
        "estado": "ok"
    })

    # crear preview si el archivo parece útil
    if col_pollutant or col_endpoint or col_study or col_beta:
        tmp = df.copy()

        # filtrar por contaminantes relevantes si existe columna
        if col_pollutant is not None:
            tmp[col_pollutant] = tmp[col_pollutant].astype(str).str.upper().str.strip()
            tmp = tmp[tmp[col_pollutant].str.contains("PM|O3|NO2", na=False)]

        keep_cols = [c for c in [
            col_pollutant, col_metric, col_endpoint_group,
            col_endpoint, col_study, col_beta, col_beta_se
        ] if c is not None]

        if keep_cols:
            tmp = tmp[keep_cols].copy().head(20)
            tmp.insert(0, "archivo", path.name)
            preview_rows.append(tmp)

# ------------------------------------------------------------
# 5) Consolidar salidas
# ------------------------------------------------------------
inventory_df = pd.DataFrame(inventory_rows).sort_values(
    ["estado", "archivo"], ascending=[True, True]
).reset_index(drop=True)

preview_df = pd.concat(preview_rows, ignore_index=True) if preview_rows else pd.DataFrame()

print("=" * 100)
print("INVENTARIO DE FUNCIONES C-R DISPONIBLES")
print("=" * 100)
display(inventory_df)

print("\nVista preliminar de filas relevantes:")
if preview_df.empty:
    print("No se detectaron filas útiles para preview.")
else:
    display(preview_df)

# ------------------------------------------------------------
# 6) Exportar
# ------------------------------------------------------------
inventory_csv = out_root / "inventario_funciones_cr.csv"
preview_csv = out_root / "inventario_funciones_cr_preview.csv"
log_json = logs_root / "21a_inventario_funciones_cr.json"

inventory_df.to_csv(inventory_csv, index=False, encoding="utf-8-sig")
if not preview_df.empty:
    preview_df.to_csv(preview_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "n_archivos_candidatos": len(candidate_files),
    "inventory_shape": list(inventory_df.shape),
    "preview_shape": list(preview_df.shape) if not preview_df.empty else [0, 0],
    "outputs": {
        "inventory_csv": str(inventory_csv),
        "preview_csv": str(preview_csv) if not preview_df.empty else None
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", inventory_csv)
if not preview_df.empty:
    print("-", preview_csv)
print("-", log_json)

INVENTARIO DE FUNCIONES C-R DISPONIBLES


,archivo,ruta,n_filas,n_columnas,col_contaminante,col_endpoint,col_endpoint_group,col_estudio,col_beta,col_beta_se,col_metric,estado
0,benmap_health_impact_functions_import_full.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,33,Pollutant,Endpoint,Endpoint Group,Reference,Beta,None,Metric,ok
1,benmap_health_impact_functions_import_full_v2.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,33,Pollutant,Endpoint,Endpoint Group,Reference,Beta,None,Metric,ok
2,benmap_health_impact_functions_import_full_v3.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,33,Pollutant,Endpoint,Endpoint Group,Reference,Beta,None,Metric,ok
3,benmap_health_impact_functions_import_full_v4.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,33,Pollutant,Endpoint,Endpoint Group,Reference,Beta,None,Metric,ok
4,benmap_health_impact_functions_ready_only.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,15,Pollutant,Endpoint,Endpoint Group,Author,None,None,Metric,ok
5,benmap_health_impact_functions_ready_only_work...,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,2,26,Pollutant,Endpoint,Endpoint Group,Author,None,beta_se,Metric,ok
6,benmap_health_impact_functions_working.csv,D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILID...,4,26,Pollutant,Endpoint,Endpoint Group,Author,None,beta_se,Metric,ok



Vista preliminar de filas relevantes:


,archivo,Pollutant,Metric,Endpoint Group,Endpoint,Reference,Beta,Author,beta_se
0,benmap_health_impact_functions_import_full.csv,PM25,Annual (D8HourMax),Mortality,All-cause mortality,Di et al. (2017),0.001094,NaN,NaN
1,benmap_health_impact_functions_import_full.csv,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al. (2009),0.000867,NaN,NaN
2,benmap_health_impact_functions_import_full_v2.csv,PM25,AnnualMean,Mortality,All-cause mortality,Di et al. (2017),0.001094,NaN,NaN
3,benmap_health_impact_functions_import_full_v2.csv,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al. (2009),0.000867,NaN,NaN
4,benmap_health_impact_functions_import_full_v3.csv,PM25,AnnualMean,Mortality,All-cause mortality,Di et al. (2017),0.001094,NaN,NaN
5,benmap_health_impact_functions_import_full_v3.csv,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al. (2009),0.000867,NaN,NaN
6,benmap_health_impact_functions_import_full_v4.csv,PM25,AnnualMean,Mortality,All-cause mortality,Di et al. (2017),0.001094,NaN,NaN
7,benmap_health_impact_functions_import_full_v4.csv,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al. (2009),0.000867,NaN,NaN
8,benmap_health_impact_functions_ready_only.csv,PM25,Annual (D8HourMax),Mortality,All-cause mortality,NaN,NaN,Di et al.,NaN
9,benmap_health_impact_functions_ready_only.csv,O3,D8HourMax,Mortality,Respiratory mortality,NaN,NaN,Katsouyanni et al.,NaN



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\funcion_cr\inventario_funciones_cr.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\funcion_cr\inventario_funciones_cr_preview.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\21a_inventario_funciones_cr.json


## Paso 21b. Inspección detallada de las funciones C–R candidatas

En esta etapa se revisan en detalle las funciones concentración-respuesta candidatas identificadas en el inventario preliminar. La finalidad es verificar cuáles filas contienen realmente la información necesaria para construir una sensibilidad por función C–R, especialmente en términos de contaminante, endpoint, referencia y coeficiente beta.

La lógica consiste en abrir directamente los archivos más relevantes de funciones sanitarias y mostrar tanto sus columnas como sus registros completos. De esta manera se podrá confirmar si existen funciones alternativas válidas para PM2.5, O3 y NO2, y si cuentan con los parámetros mínimos necesarios para ser utilizadas en el pipeline.

El resultado esperado es una inspección clara de las funciones candidatas, a partir de la cual se definirá en el siguiente paso cuáles comparaciones se incluirán formalmente en la sensibilidad por función C–R.

In [21]:
# ============================================================
# PASO 21b: inspección detallada de funciones C-R candidatas
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")
benmap_root = project_root / "SALIDAS_SENSIBILIDAD" / "01_insumos" / "benmap"

candidate_paths = [
    benmap_root / "benmap_health_impact_functions_import_full.csv",
    benmap_root / "benmap_health_impact_functions_import_full_v2.csv",
    benmap_root / "benmap_health_impact_functions_import_full_v3.csv",
    benmap_root / "benmap_health_impact_functions_import_full_v4.csv",
    benmap_root / "benmap_health_impact_functions_ready_only.csv",
    benmap_root / "benmap_health_impact_functions_ready_only_working.csv",
    benmap_root / "benmap_health_impact_functions_working.csv",
]

# ------------------------------------------------------------
# 2) Utilidad
# ------------------------------------------------------------
def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        df = pd.read_csv(path, encoding="latin1")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

# ------------------------------------------------------------
# 3) Revisar uno por uno
# ------------------------------------------------------------
for path in candidate_paths:
    if not path.exists():
        print("\n" + "=" * 100)
        print(f"NO EXISTE: {path.name}")
        print("=" * 100)
        continue

    df = read_csv_safe(path)

    print("\n" + "=" * 100)
    print(f"ARCHIVO: {path.name}")
    print("=" * 100)
    print(f"Forma: {df.shape}")
    print("\nColumnas:")
    print(list(df.columns))

    # filtrar contaminantes relevantes si existe columna Pollutant
    if "Pollutant" in df.columns:
        tmp = df[df["Pollutant"].astype(str).str.upper().str.contains("PM|O3|NO2", na=False)].copy()
    else:
        tmp = df.copy()

    print("\nContenido relevante:")
    display(tmp)

    # mostrar un subconjunto útil si existen las columnas
    useful_cols = [c for c in [
        "Pollutant", "Metric", "Endpoint Group", "Endpoint",
        "Reference", "Author", "Beta", "beta_se"
    ] if c in tmp.columns]

    if useful_cols:
        print("\nResumen útil:")
        display(tmp[useful_cols])


ARCHIVO: benmap_health_impact_functions_import_full.csv
Forma: (2, 33)

Columnas:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Seasonal Metric', 'Metric Statistic', 'Study Year', 'Study Author', 'Study Location', 'Reference', 'Qualifier', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Geographic Area', 'Incidence DataSet', 'Prevalence DataSet', 'Variable DataSet', 'Beta', 'Distribution Beta', 'Parameter 1 Beta', 'Parameter 2 Beta', 'Other Pollutants', 'Function', 'Baseline Function', 'A', 'Name A', 'B', 'Name B', 'C', 'Name C']

Contenido relevante:


,Endpoint Group,Endpoint,Pollutant,Metric,Seasonal Metric,Metric Statistic,Study Year,Study Author,Study Location,Reference,...,Parameter 2 Beta,Other Pollutants,Function,Baseline Function,A,Name A,B,Name B,C,Name C
0,Mortality,All-cause mortality,PM25,Annual (D8HourMax),NaN,NaN,2017,Di et al.,Di et al. (2017),Di et al. (2017),...,NaN,NaN,log-linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Mortality,Respiratory mortality,O3,D8HourMax,NaN,NaN,2009,Katsouyanni et al.,Katsouyanni et al. (2009),Katsouyanni et al. (2009),...,NaN,NaN,log-linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Resumen útil:


,Pollutant,Metric,Endpoint Group,Endpoint,Reference,Beta
0,PM25,Annual (D8HourMax),Mortality,All-cause mortality,Di et al. (2017),0.001094
1,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al. (2009),0.000867



ARCHIVO: benmap_health_impact_functions_import_full_v2.csv
Forma: (2, 33)

Columnas:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Seasonal Metric', 'Metric Statistic', 'Study Year', 'Study Author', 'Study Location', 'Reference', 'Qualifier', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Geographic Area', 'Incidence DataSet', 'Prevalence DataSet', 'Variable DataSet', 'Beta', 'Distribution Beta', 'Parameter 1 Beta', 'Parameter 2 Beta', 'Other Pollutants', 'Function', 'Baseline Function', 'A', 'Name A', 'B', 'Name B', 'C', 'Name C']

Contenido relevante:


...

,Pollutant,Metric,Endpoint Group,Endpoint,Author,beta_se
0,PM25,Annual (D8HourMax),Mortality,All-cause mortality,Di et al.,0.000050
1,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al.,0.000304



ARCHIVO: benmap_health_impact_functions_working.csv
Forma: (4, 26)

Columnas:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Annual Statistic', 'Seasonal Metric', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Author', 'Apply Function To', 'Year of Publication', 'Qualifier', 'endpoint_code_local', 'reference_short', 'reference_full', 'effect_type', 'beta_per_unit', 'beta_se', 'unit_change', 'concentration_unit', 'function_form', 'status', 'notes']

Contenido relevante:


,Endpoint Group,Endpoint,Pollutant,Metric,Annual Statistic,Seasonal Metric,Race,Ethnicity,Gender,Start Age,...,reference_short,reference_full,effect_type,beta_per_unit,beta_se,unit_change,concentration_unit,function_form,status,notes
0,Mortality,All-cause mortality,PM25,Annual (D8HourMax),NaN,NaN,NaN,NaN,NaN,65,...,Di et al. (2017),"Pope, C. A., & Dockery, D. W. (2006). Health e...",mortality,0.001094,0.000050,1.0,ug/m3,log-linear,READY_NUMERIC,Revisar estudio epidemiológico final y rango e...
1,Mortality,Respiratory mortality,PM25,PM25 Annual Mean,NaN,NaN,NaN,NaN,NaN,0,...,Pope & Dockery (2006),"Pope, C. A., & Dockery, D. W. (2006). Health e...",mortality,NaN,NaN,10.0,ug/m3,log-linear,REVIEW_PM25_DUPLICATE,Revisar si el endpoint final se mantiene como ...
2,Mortality,Respiratory mortality,O3,D8HourMax,NaN,NaN,NaN,NaN,NaN,0,...,Katsouyanni et al. (2009),"Tagaris, E., et al. (2010). Sensitivity of air...",mortality,0.000867,0.000304,1.0,ppb_or_ugm3_EDIT,log-linear,READY_NUMERIC,Confirmar métrica de O3 y unidad usada en BenMAP.
3,Mortality,Circulatory mortality,NO2,NO2 Annual Mean,NaN,NaN,NaN,NaN,NaN,0,...,Custom NO2 reference,Seleccionar estudio epidemiológico final si se...,mortality,NaN,NaN,10.0,ug/m3,log-linear,NO2_CUSTOM_REVIEW,Fila opcional. Mantener solo si se cargará fun...



Resumen útil:


,Pollutant,Metric,Endpoint Group,Endpoint,Author,beta_se
0,PM25,Annual (D8HourMax),Mortality,All-cause mortality,Di et al.,0.000050
1,PM25,PM25 Annual Mean,Mortality,Respiratory mortality,Pope & Dockery,NaN
2,O3,D8HourMax,Mortality,Respiratory mortality,Katsouyanni et al.,0.000304
3,NO2,NO2 Annual Mean,Mortality,Circulatory mortality,Custom NO2 Function,NaN


## Paso 21c. Sensibilidad por función concentración-respuesta para PM2.5

En esta etapa se implementa la sensibilidad por función concentración-respuesta para PM2.5. La finalidad es evaluar cuánto cambian los resultados sanitarios y económicos cuando, en lugar de utilizar únicamente la función epidemiológica adoptada en la corrida principal, se utiliza una función alternativa identificada en los archivos del proyecto.

La comparación se construye entre dos escenarios. El primero corresponde a la función base de la corrida principal de PM2.5. El segundo corresponde a una función alternativa asociada a la referencia Di et al. (2017), con un coeficiente beta menor. De esta forma se mide la sensibilidad de los casos evitados y de la valoración económica frente a la elección del estudio epidemiológico de referencia.

El resultado esperado es una tabla por celda y una tabla resumen para PM2.5, con comparación directa entre función base y función alternativa. Esto permitirá cuantificar si la elección de la función C–R modifica sustancialmente la carga sanitaria y económica estimada para este contaminante.

In [1]:
# ============================================================
# PASO 21c: sensibilidad por función C-R para PM2.5
# compara función base vs función alternativa (Di et al. 2017)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetros económicos
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0
FX_COP_PER_USD_2015 = 2773.43

# ------------------------------------------------------------
# 2) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

base_root = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica"
sens_root = project_root / "SALIDAS_SENSIBILIDAD" / "04_sensibilidad" / "funcion_cr"
logs_root = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion"

base_path = base_root / "base_analitica_final_por_corrida.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_root.mkdir(parents=True, exist_ok=True)

if not base_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{base_path}")

# ------------------------------------------------------------
# 3) Cargar base analítica
# ------------------------------------------------------------
base = pd.read_csv(base_path, encoding="utf-8-sig")
base.columns = [str(c).replace("\ufeff", "").strip() for c in base.columns]

# Filtrar solo PM2.5
pm25 = base[base["run_id"].astype(str).str.strip() == "PM25_MAIN"].copy()

if pm25.empty:
    raise ValueError("No se encontraron filas para PM25_MAIN en la base analítica.")

for c in ["delta_concentration", "population_target", "incidence_target", "beta", "beta_se"]:
    if c in pm25.columns:
        pm25[c] = pd.to_numeric(pm25[c], errors="coerce")

# ------------------------------------------------------------
# 4) Definir escenarios C-R
# ------------------------------------------------------------
# Función base = la corrida principal actual
beta_base = float(pm25["beta"].dropna().iloc[0])

# Función alternativa identificada en tus archivos:
beta_alt = 0.001094

cr_scenarios = pd.DataFrame([
    {
        "run_id": "PM25_MAIN",
        "pollutant": "PM25",
        "cr_scenario": "Función base",
        "cr_reference": "Corrida principal PM2.5",
        "beta_value": beta_base
    },
    {
        "run_id": "PM25_MAIN",
        "pollutant": "PM25",
        "cr_scenario": "Función alternativa",
        "cr_reference": "Di et al. (2017)",
        "beta_value": beta_alt
    }
])

# ------------------------------------------------------------
# 5) Cruzar base con escenarios C-R
# ------------------------------------------------------------
sens_cr = pm25.merge(
    cr_scenarios[["run_id", "cr_scenario", "cr_reference", "beta_value"]],
    on="run_id",
    how="inner"
)

# ------------------------------------------------------------
# 6) Cálculo sanitario
# ------------------------------------------------------------
sens_cr["rr"] = np.exp(sens_cr["beta_value"] * sens_cr["delta_concentration"])
sens_cr["af_avoided"] = 1 - np.exp(-sens_cr["beta_value"] * sens_cr["delta_concentration"])
sens_cr["baseline_cases"] = sens_cr["incidence_target"] * sens_cr["population_target"]
sens_cr["avoided_cases"] = sens_cr["baseline_cases"] * sens_cr["af_avoided"]

# ------------------------------------------------------------
# 7) Valoración económica
# ------------------------------------------------------------
sens_cr["valuation_value_used_usd_2015"] = VSL_2015_USD
sens_cr["economic_value_usd_2015"] = sens_cr["avoided_cases"] * sens_cr["valuation_value_used_usd_2015"]
sens_cr["economic_value_cop_2015"] = sens_cr["economic_value_usd_2015"] * FX_COP_PER_USD_2015

# ------------------------------------------------------------
# 8) Resumen
# ------------------------------------------------------------
summary = (
    sens_cr.groupby(
        ["run_id", "pollutant", "cr_scenario", "cr_reference"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        delta_mean=("delta_concentration", "mean"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
        beta_value=("beta_value", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 9) Comparación contra la función base
# ------------------------------------------------------------
base_ref = (
    summary[summary["cr_scenario"] == "Función base"][
        ["run_id", "avoided_cases_total", "economic_value_total_usd_2015", "economic_value_total_cop_2015"]
    ]
    .rename(columns={
        "avoided_cases_total": "cases_base",
        "economic_value_total_usd_2015": "usd_base",
        "economic_value_total_cop_2015": "cop_base"
    })
)

summary_compare = summary.merge(base_ref, on="run_id", how="left")

summary_compare["diferencia_casos_vs_base"] = (
    summary_compare["avoided_cases_total"] - summary_compare["cases_base"]
)

summary_compare["razon_casos_vs_base"] = np.where(
    summary_compare["cases_base"].abs() > 0,
    summary_compare["avoided_cases_total"] / summary_compare["cases_base"],
    np.nan
)

summary_compare["diferencia_usd_vs_base"] = (
    summary_compare["economic_value_total_usd_2015"] - summary_compare["usd_base"]
)

summary_compare["razon_usd_vs_base"] = np.where(
    summary_compare["usd_base"].abs() > 0,
    summary_compare["economic_value_total_usd_2015"] / summary_compare["usd_base"],
    np.nan
)

# ------------------------------------------------------------
# 10) Mostrar
# ------------------------------------------------------------
print("=" * 90)
print("SENSIBILIDAD POR FUNCIÓN C-R PARA PM2.5")
print("=" * 90)
display(summary_compare)

print("\nVista previa por celda:")
display(
    sens_cr[[
        "run_id", "pollutant", "cr_scenario", "cr_reference",
        "cell_id", "delta_concentration", "population_target",
        "incidence_target", "beta_value", "avoided_cases",
        "economic_value_usd_2015", "economic_value_cop_2015"
    ]].head(20)
)

# ------------------------------------------------------------
# 11) Exportar
# ------------------------------------------------------------
cells_csv = sens_root / "sensibilidad_funcion_cr_pm25_por_celda.csv"
summary_csv = sens_root / "sensibilidad_funcion_cr_pm25_resumen.csv"
compare_csv = sens_root / "sensibilidad_funcion_cr_pm25_resumen_comparado_base.csv"
log_json = logs_root / "21c_sensibilidad_funcion_cr_pm25.json"

sens_cr.to_csv(cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "VSL_2015_USD": VSL_2015_USD,
    "FX_COP_PER_USD_2015": FX_COP_PER_USD_2015,
    "beta_base": beta_base,
    "beta_alt": beta_alt,
    "sens_cr_shape": list(sens_cr.shape),
    "summary_shape": list(summary.shape),
    "summary_compare_shape": list(summary_compare.shape),
    "outputs": {
        "cells_csv": str(cells_csv),
        "summary_csv": str(summary_csv),
        "compare_csv": str(compare_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", cells_csv)
print("-", summary_csv)
print("-", compare_csv)
print("-", log_json)

SENSIBILIDAD POR FUNCIÓN C-R PARA PM2.5


,run_id,pollutant,cr_scenario,cr_reference,n_rows,n_unique_cells,delta_mean,baseline_cases_total,avoided_cases_total,beta_value,economic_value_total_usd_2015,economic_value_total_cop_2015,cases_base,usd_base,cop_base,diferencia_casos_vs_base,razon_casos_vs_base,diferencia_usd_vs_base,razon_usd_vs_base
0,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),254,254,1.478043,109.744592,0.177207,0.001094,1.542609e+06,4.278319e+09,1.299799,1.131490e+07,3.138108e+10,-1.122592,0.136334,-9.772290e+06,0.136334
1,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,254,254,1.478043,109.744592,1.299799,0.008066,1.131490e+07,3.138108e+10,1.299799,1.131490e+07,3.138108e+10,0.000000,1.000000,0.000000e+00,1.000000



Vista previa por celda:


,run_id,pollutant,cr_scenario,cr_reference,cell_id,delta_concentration,population_target,incidence_target,beta_value,avoided_cases,economic_value_usd_2015,economic_value_cop_2015
0,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,0,1.505179,7006,0.000064,0.008066,0.005445,47397.500503,1.314536e+08
1,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),0,1.505179,7006,0.000064,0.001094,0.000742,6462.353220,1.792288e+07
2,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,41,1.473053,5661,0.000064,0.008066,0.004306,37485.646070,1.039638e+08
3,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),41,1.473053,5661,0.000064,0.001094,0.000587,5110.362726,1.417323e+07
4,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,82,1.485199,656,0.000064,0.008066,0.000503,4379.459829,1.214613e+07
5,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),82,1.485199,656,0.000064,0.001094,0.000069,597.070507,1.655933e+06
6,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,1,1.482113,3805,0.000064,0.008066,0.002912,25349.737447,7.030572e+07
7,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),1,1.482113,3805,0.000064,0.001094,0.000397,3456.001137,9.584977e+06
8,PM25_MAIN,PM25,Función base,Corrida principal PM2.5,42,1.405620,9380,0.000064,0.008066,0.006810,59284.617340,1.644217e+08
9,PM25_MAIN,PM25,Función alternativa,Di et al. (2017),42,1.405620,9380,0.000064,0.001094,0.000928,8080.288758,2.241012e+07



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\funcion_cr\sensibilidad_funcion_cr_pm25_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\funcion_cr\sensibilidad_funcion_cr_pm25_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\funcion_cr\sensibilidad_funcion_cr_pm25_resumen_comparado_base.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\21c_sensibilidad_funcion_cr_pm25.json


## Paso 22. Escenarios de sensibilidad climática para PM2.5, NO2 y O3

En esta etapa se construyen escenarios climáticos exploratorios para los tres contaminantes principales del estudio. La finalidad es evaluar cuánto cambian los impactos sanitarios y económicos cuando las condiciones climáticas favorecen o desfavorecen la formación, acumulación o dispersión de contaminantes en el área de estudio.

La lógica del procedimiento consiste en definir multiplicadores climáticos diferenciados por contaminante. Para O3 se adopta una sensibilidad más marcada, dado que su formación es especialmente dependiente de condiciones meteorológicas como temperatura y radiación. Para PM2.5 se considera una sensibilidad intermedia asociada a procesos de acumulación y ventilación. Para NO2 se adopta una sensibilidad más moderada, entendiendo que su comportamiento climático es relevante pero menos dominante que en el caso del ozono.

Se construyen tres escenarios: favorable, central y desfavorable. Estos escenarios no representan proyecciones climáticas formales, sino una sensibilidad exploratoria de exposición útil para medir la robustez de los resultados. El resultado esperado es una tabla por celda y una tabla resumen por corrida, con comparación frente al escenario climático central.

In [2]:
# ============================================================
# PASO 22: sensibilidad climática para PM2.5, NO2 y O3
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1) Parámetros económicos
# ------------------------------------------------------------
VSL_2015_USD = 8_705_114.0
FX_COP_PER_USD_2015 = 2773.43

# ------------------------------------------------------------
# 2) Escenarios climáticos exploratorios
#    O3 más sensible, PM2.5 intermedio, NO2 moderado
# ------------------------------------------------------------
climate_factors = pd.DataFrame([
    # PM2.5
    {"pollutant": "PM25", "climate_scenario": "Escenario climático favorable",   "climate_multiplier": 0.95, "climate_note": "Mayor dispersión / menor acumulación"},
    {"pollutant": "PM25", "climate_scenario": "Escenario climático central",     "climate_multiplier": 1.00, "climate_note": "Escenario de referencia"},
    {"pollutant": "PM25", "climate_scenario": "Escenario climático desfavorable","climate_multiplier": 1.10, "climate_note": "Mayor acumulación / menor ventilación"},

    # NO2
    {"pollutant": "NO2",  "climate_scenario": "Escenario climático favorable",   "climate_multiplier": 0.97, "climate_note": "Menor acumulación local"},
    {"pollutant": "NO2",  "climate_scenario": "Escenario climático central",     "climate_multiplier": 1.00, "climate_note": "Escenario de referencia"},
    {"pollutant": "NO2",  "climate_scenario": "Escenario climático desfavorable","climate_multiplier": 1.05, "climate_note": "Mayor estancamiento local"},

    # O3
    {"pollutant": "O3",   "climate_scenario": "Escenario climático favorable",   "climate_multiplier": 0.90, "climate_note": "Menor formación fotoquímica"},
    {"pollutant": "O3",   "climate_scenario": "Escenario climático central",     "climate_multiplier": 1.00, "climate_note": "Escenario de referencia"},
    {"pollutant": "O3",   "climate_scenario": "Escenario climático desfavorable","climate_multiplier": 1.15, "climate_note": "Mayor formación fotoquímica"},
])

# ------------------------------------------------------------
# 3) Rutas
# ------------------------------------------------------------
project_root = Path(r"D:\TRABAJO DE GRADO BEN-MAP")

base_root = project_root / "SALIDAS_SENSIBILIDAD" / "02_base_analitica"
sens_root = project_root / "SALIDAS_SENSIBILIDAD" / "04_sensibilidad" / "clima"
logs_root = project_root / "SALIDAS_SENSIBILIDAD" / "00_logs_validacion"

base_path = base_root / "base_analitica_final_por_corrida.csv"

sens_root.mkdir(parents=True, exist_ok=True)
logs_root.mkdir(parents=True, exist_ok=True)

if not base_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{base_path}")

# ------------------------------------------------------------
# 4) Cargar base analítica
# ------------------------------------------------------------
base = pd.read_csv(base_path, encoding="utf-8-sig")
base.columns = [str(c).replace("\ufeff", "").strip() for c in base.columns]

# normalizar numéricos
num_cols = [
    "cell_id", "delta_concentration", "population_target", "incidence_target",
    "beta", "beta_se", "valuation_value_fixed",
    "concentration_baseline", "concentration_control"
]
for c in num_cols:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

required_cols = ["run_id", "pollutant", "economic_method", "cell_id", "population_target", "incidence_target", "beta"]
missing_required = [c for c in required_cols if c not in base.columns]
if missing_required:
    raise ValueError(f"Faltan columnas requeridas en base analítica: {missing_required}")

# ------------------------------------------------------------
# 5) Cruzar con factores climáticos
# ------------------------------------------------------------
sens_clima = base.merge(
    climate_factors,
    on="pollutant",
    how="inner"
)

# ------------------------------------------------------------
# 6) Recalcular concentraciones / delta bajo clima
# ------------------------------------------------------------
has_full_conc = {"concentration_baseline", "concentration_control"}.issubset(sens_clima.columns)

if has_full_conc:
    sens_clima["concentration_baseline_clim"] = sens_clima["concentration_baseline"] * sens_clima["climate_multiplier"]
    sens_clima["concentration_control_clim"] = sens_clima["concentration_control"] * sens_clima["climate_multiplier"]
    sens_clima["delta_concentration_clim"] = (
        sens_clima["concentration_baseline_clim"] - sens_clima["concentration_control_clim"]
    )
else:
    sens_clima["concentration_baseline_clim"] = np.nan
    sens_clima["concentration_control_clim"] = np.nan
    sens_clima["delta_concentration_clim"] = sens_clima["delta_concentration"] * sens_clima["climate_multiplier"]

# ------------------------------------------------------------
# 7) Cálculo sanitario
# ------------------------------------------------------------
sens_clima["rr"] = np.exp(sens_clima["beta"] * sens_clima["delta_concentration_clim"])
sens_clima["af_avoided"] = 1 - np.exp(-sens_clima["beta"] * sens_clima["delta_concentration_clim"])
sens_clima["baseline_cases"] = sens_clima["incidence_target"] * sens_clima["population_target"]
sens_clima["avoided_cases"] = sens_clima["baseline_cases"] * sens_clima["af_avoided"]

# ------------------------------------------------------------
# 8) Valoración económica
# ------------------------------------------------------------
sens_clima["valuation_value_used_usd_2015"] = np.nan

mask_vsl = sens_clima["economic_method"].astype(str).str.strip() == "VSL"
sens_clima.loc[mask_vsl, "valuation_value_used_usd_2015"] = VSL_2015_USD

mask_unit = ~mask_vsl
if "valuation_value_fixed" in sens_clima.columns:
    sens_clima.loc[mask_unit, "valuation_value_used_usd_2015"] = pd.to_numeric(
        sens_clima.loc[mask_unit, "valuation_value_fixed"], errors="coerce"
    )

sens_clima["economic_value_usd_2015"] = sens_clima["avoided_cases"] * sens_clima["valuation_value_used_usd_2015"]
sens_clima["economic_value_cop_2015"] = sens_clima["economic_value_usd_2015"] * FX_COP_PER_USD_2015

# ------------------------------------------------------------
# 9) Resumen por corrida y escenario climático
# ------------------------------------------------------------
summary = (
    sens_clima.groupby(
        ["run_id", "pollutant", "endpoint_group", "endpoint", "economic_method", "climate_scenario", "climate_note"],
        as_index=False
    )
    .agg(
        n_rows=("cell_id", "size"),
        n_unique_cells=("cell_id", "nunique"),
        climate_multiplier=("climate_multiplier", "first"),
        delta_mean_clim=("delta_concentration_clim", "mean"),
        baseline_cases_total=("baseline_cases", "sum"),
        avoided_cases_total=("avoided_cases", "sum"),
        beta=("beta", "first"),
        economic_value_total_usd_2015=("economic_value_usd_2015", "sum"),
        economic_value_total_cop_2015=("economic_value_cop_2015", "sum"),
    )
)

# ------------------------------------------------------------
# 10) Comparación contra escenario climático central
# ------------------------------------------------------------
central_ref = (
    summary[summary["climate_scenario"] == "Escenario climático central"][
        ["run_id", "avoided_cases_total", "economic_value_total_usd_2015", "economic_value_total_cop_2015"]
    ]
    .rename(columns={
        "avoided_cases_total": "cases_central",
        "economic_value_total_usd_2015": "usd_central",
        "economic_value_total_cop_2015": "cop_central"
    })
)

summary_compare = summary.merge(central_ref, on="run_id", how="left")

summary_compare["diferencia_casos_vs_central"] = (
    summary_compare["avoided_cases_total"] - summary_compare["cases_central"]
)

summary_compare["razon_casos_vs_central"] = np.where(
    summary_compare["cases_central"].abs() > 0,
    summary_compare["avoided_cases_total"] / summary_compare["cases_central"],
    np.nan
)

summary_compare["diferencia_usd_vs_central"] = (
    summary_compare["economic_value_total_usd_2015"] - summary_compare["usd_central"]
)

summary_compare["razon_usd_vs_central"] = np.where(
    summary_compare["usd_central"].abs() > 0,
    summary_compare["economic_value_total_usd_2015"] / summary_compare["usd_central"],
    np.nan
)

# ------------------------------------------------------------
# 11) Mostrar resultados
# ------------------------------------------------------------
print("=" * 95)
print("SENSIBILIDAD CLIMÁTICA PARA PM2.5, NO2 Y O3")
print("=" * 95)
display(summary_compare)

print("\nVista previa por celda:")
display(
    sens_clima[[
        "run_id", "pollutant", "climate_scenario", "climate_multiplier",
        "cell_id", "delta_concentration", "delta_concentration_clim",
        "population_target", "incidence_target", "beta",
        "avoided_cases", "economic_value_usd_2015", "economic_value_cop_2015"
    ]].head(30)
)

# ------------------------------------------------------------
# 12) Exportar
# ------------------------------------------------------------
cells_csv = sens_root / "sensibilidad_climatica_por_celda.csv"
summary_csv = sens_root / "sensibilidad_climatica_resumen.csv"
compare_csv = sens_root / "sensibilidad_climatica_resumen_comparado_central.csv"
scenarios_csv = sens_root / "escenarios_climaticos_parametros.csv"
log_json = logs_root / "22_sensibilidad_climatica.json"

sens_clima.to_csv(cells_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_compare.to_csv(compare_csv, index=False, encoding="utf-8-sig")
climate_factors.to_csv(scenarios_csv, index=False, encoding="utf-8-sig")

payload = {
    "timestamp": datetime.now().isoformat(),
    "VSL_2015_USD": VSL_2015_USD,
    "FX_COP_PER_USD_2015": FX_COP_PER_USD_2015,
    "sens_clima_shape": list(sens_clima.shape),
    "summary_shape": list(summary.shape),
    "summary_compare_shape": list(summary_compare.shape),
    "outputs": {
        "cells_csv": str(cells_csv),
        "summary_csv": str(summary_csv),
        "compare_csv": str(compare_csv),
        "scenarios_csv": str(scenarios_csv)
    }
}

with open(log_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print("-", cells_csv)
print("-", summary_csv)
print("-", compare_csv)
print("-", scenarios_csv)
print("-", log_json)

SENSIBILIDAD CLIMÁTICA PARA PM2.5, NO2 Y O3


,run_id,pollutant,endpoint_group,endpoint,economic_method,climate_scenario,climate_note,n_rows,n_unique_cells,climate_multiplier,...,beta,economic_value_total_usd_2015,economic_value_total_cop_2015,cases_central,usd_central,cop_central,diferencia_casos_vs_central,razon_casos_vs_central,diferencia_usd_vs_central,razon_usd_vs_central
0,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario climático central,Escenario de referencia,254,254,1.00,...,0.003324,1.427983e+05,3.960412e+08,12.713526,1.427983e+05,3.960412e+08,0.000000,1.000000,0.000000e+00,1.000000
1,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario climático desfavorable,Mayor estancamiento local,254,254,1.05,...,0.003324,1.499203e+05,4.157935e+08,12.713526,1.427983e+05,3.960412e+08,0.634081,1.049875,7.121995e+03,1.049875
2,NO2_ASTHMA,NO2,Hospital Admissions,Asthma,UNIT_VALUE_ASTHMA_HA,Escenario climático favorable,Menor acumulación local,254,254,0.97,...,0.003324,1.385243e+05,3.841875e+08,12.713526,1.427983e+05,3.960412e+08,-0.380521,0.970070,-4.274015e+03,0.970070
3,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario climático central,Escenario de referencia,254,254,1.00,...,0.001850,1.453574e+05,4.031385e+08,9.454138,1.453574e+05,4.031385e+08,0.000000,1.000000,0.000000e+00,1.000000
4,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario climático desfavorable,Mayor estancamiento local,254,254,1.05,...,0.001850,1.526151e+05,4.232672e+08,9.454138,1.453574e+05,4.031385e+08,0.472046,1.049930,7.257712e+03,1.049930
5,NO2_CLD,NO2,Hospital Admissions,Chronic Lung Disease,UNIT_VALUE_CLD_HA,Escenario climático favorable,Menor acumulación local,254,254,0.97,...,0.001850,1.410023e+05,3.910600e+08,9.454138,1.453574e+05,4.031385e+08,-0.283258,0.970039,-4.355091e+03,0.970039
6,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario climático central,Escenario de referencia,254,254,1.00,...,0.000867,1.919088e+07,5.322457e+10,2.204553,1.919088e+07,5.322457e+10,0.000000,1.000000,0.000000e+00,1.000000
7,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario climático desfavorable,Mayor formación fotoquímica,254,254,1.15,...,0.000867,2.206779e+07,6.120346e+10,2.204553,1.919088e+07,5.322457e+10,0.330484,1.149910,2.876902e+06,1.149910
8,O3_MAIN,O3,Mortality,Respiratory mortality,VSL,Escenario climático favorable,Menor formación fotoquímica,254,254,0.90,...,0.000867,1.727270e+07,4.790462e+10,2.204553,1.919088e+07,5.322457e+10,-0.220352,0.900047,-1.918185e+06,0.900047
9,PM25_MAIN,PM25,Mortality,All-cause mortality,VSL,Escenario climático central,Escenario de referencia,254,254,1.00,...,0.008066,1.131490e+07,3.138108e+10,1.299799,1.131490e+07,3.138108e+10,0.000000,1.000000,0.000000e+00,1.000000



Vista previa por celda:


,run_id,pollutant,climate_scenario,climate_multiplier,cell_id,delta_concentration,delta_concentration_clim,population_target,incidence_target,beta,avoided_cases,economic_value_usd_2015,economic_value_cop_2015
0,NO2_ASTHMA,NO2,Escenario climático favorable,0.97,0,1.423132,1.380438,66007,0.000167,0.003324,0.050385,565.919765,1.569539e+06
1,NO2_ASTHMA,NO2,Escenario climático central,1.00,0,1.423132,1.423132,66007,0.000167,0.003324,0.051939,583.381073,1.617967e+06
2,NO2_ASTHMA,NO2,Escenario climático desfavorable,1.05,0,1.423132,1.494288,66007,0.000167,0.003324,0.054530,612.477748,1.698664e+06
3,NO2_ASTHMA,NO2,Escenario climático favorable,0.97,41,1.443691,1.400380,53333,0.000167,0.003324,0.041297,463.847992,1.286450e+06
4,NO2_ASTHMA,NO2,Escenario climático central,1.00,41,1.443691,1.443691,53333,0.000167,0.003324,0.042571,478.159413,1.326142e+06
5,NO2_ASTHMA,NO2,Escenario climático desfavorable,1.05,41,1.443691,1.515875,53333,0.000167,0.003324,0.044694,502.007203,1.392282e+06
6,NO2_ASTHMA,NO2,Escenario climático favorable,0.97,82,1.404677,1.362536,6180,0.000167,0.003324,0.004656,52.299509,1.450490e+05
7,NO2_ASTHMA,NO2,Escenario climático central,1.00,82,1.404677,1.404677,6180,0.000167,0.003324,0.004800,53.913247,1.495246e+05
8,NO2_ASTHMA,NO2,Escenario climático desfavorable,1.05,82,1.404677,1.474911,6180,0.000167,0.003324,0.005039,56.602307,1.569825e+05
9,NO2_ASTHMA,NO2,Escenario climático favorable,0.97,1,1.454715,1.411074,35848,0.000167,0.003324,0.027969,314.152637,8.712803e+05



Archivos generados:
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\clima\sensibilidad_climatica_por_celda.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\clima\sensibilidad_climatica_resumen.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\clima\sensibilidad_climatica_resumen_comparado_central.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\04_sensibilidad\clima\escenarios_climaticos_parametros.csv
- D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_SENSIBILIDAD\00_logs_validacion\22_sensibilidad_climatica.json
